<a href="https://colab.research.google.com/github/mohamtur1/4CBOn2/blob/main/4CBOn2_Kaggle4_submission_agent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 4CBON2 — KAGGLE4 REBUILD SPEC

**Base:** `4CBOn2_Gemini2b.ipynb` (full version, includes the 🔨 Agent Builder) · **New file:** `4CBOn2_Kaggle4.ipynb` · **The base file is untouched.**

**Core architectural decision (unchanged):** the 17-layer AI Rewriter is the **single shared reasoning engine** for the entire app, exposed as one callable service — `reason(input) -> structured_output` (Cell 8). Every module that needs reasoning calls this service. Modules do **not** duplicate the 17 layers and do **not** bypass the service when reasoning is required. Trivial non-reasoning operations (raw file writes, raw HTTP fetch, keystrokes, file reads) do **not** pass through the pipeline.

---

## Data-flow diagram

```text
All reasoning modules ──────────────────────────────▶ 17-LAYER SERVICE
                                                     reason(input) -> structured_output

17-Layer Service ──▶ STANDARD FORM ─────────────────▶ Ask-a-Question  (Module 3)
17-Layer Service ──▶ STANDARD FORM ─────────────────▶ Agent Builder   (Module 4)

17-Layer Service ──▶ RAW REASONING ─────────────────▶ Kaggle Agent
17-Layer Service ──▶ RAW REASONING ─────────────────▶ Database Downloader (Module 1)
17-Layer Service ──▶ RAW REASONING ─────────────────▶ Agent Mode      (Module 5)

Kaggle Agent ──▶ Database Downloader   (download decisions)
Kaggle Agent ──▶ Ask-a-Question        (the single Kaggle ARC-AGI-3 notebook prompt)
Ask-a-Question ──▶ Agent Builder      (optional downstream: STANDARD FORM hand-off)

Ask-a-Question ──✗── any database     (fully decoupled: no retrieval, no schema design, no downloader calls)
Database Downloader ──✗── Ask-a-Question  (the Downloader is never called by Ask-a-Question)
```

---

## Modified module list — exact changes per module

| # | Module | Status | Exact changes |
|---|--------|--------|---------------|
| 1 | **Database Downloader** | NEW module (Cell 9, tab `⬇️ Database Downloader`) | *Which* additional AGI/ARC databases to download is decided by the 17-layer service (RAW REASONING) from a candidate catalog: ARC-AGI-1/ARC-AGI-2 public task sets, grid/world-model datasets (MiniGrid, Procgen, Atari/ALE), program-synthesis datasets (HARC-style Karel, DreamCoder-style), object-centric perception (CLEVR), reasoning-trace datasets (ARC-AGI-3-Agents, CoT-Hub), small pretrained vision models (DeiT-tiny ViT, ResNet-18 CNN), small pretrained LLMs (Phi-1.5, Qwen2.5-0.5B, TinyLlama-1.1B). The raw download step is plain I/O (streamed HTTP) and does **not** pass through the pipeline. Every download is logged (what / from where / when, plus bytes and status). All existing databases are untouched — downloads land in `DATA_ROOT/4cbon2_downloads/`. Never called by Ask-a-Question. No download happens without a service decision (fallback policy `error`). |
| 2 | **AI Rewriter (17 layers)** | Exposed as a service (Cell 8) | Internal structure of the 17-layer pipeline (`L0→P→W→LX→LA→LC→L1→L2→LP→L3→L4→LR→L6→L7→L8→L9→L10`) **unchanged**. New thin wrapper exposes it as ONE callable service `reason(input) -> structured_output` with two output modes: (a) **STANDARD FORM** — the app's existing answer schema, unchanged; (b) **RAW REASONING** — internal traces for downstream modules. The service first produces a base-LLM draft of the requested answer/decision (a single non-layered call — the same kind of input the AI Rewriter tab expects), then the 17 layers reason over that draft. |
| 3 | **Ask a Question** | Split paths (tab `❓ Ask a Question`) | **General questions** keep `ask_question_standard` and the legacy RAG toggle. **Kaggle Agent path** accepts pasted logs, asks the Kaggle Agent for prompt text only, then runs the prompt through `ask_question_kaggle` (STANDARD FORM; no retrieval or downloader calls). |
| 4 | **Agent Builder** | Modified (tab `🔨 Builder`) | Core behavior unchanged: reads the notebook, proposes changes, 5-lens verdict, backups, apply with safety checks. Its design/build decisions (proposal generation, 5-lens verdict) now route through the 17-layer service (RAW REASONING, module `agent_builder`); on budget timeout it falls back to its pre-rebuild direct call (policy `heuristic`, logged in the result status). It can consume the **STANDARD FORM** output of Ask-a-Question via the *Send Standard Form to Agent Builder* hand-off. |
| 5 | **Agent Mode** | Now in scope (tab `🤖 Agent Mode`, interface unchanged) | Decision steps (planning, batch synthesis, final synthesis, agent tool-choice/final-answer) route through the 17-layer service (RAW REASONING, module `agent_mode`) via `_decision_llm`. Trivial operations — tool execution, file reads, raw web/HTTP — stay direct and never enter the pipeline. **Latency guard:** if reasoning exceeds the per-action budget (default 60 s), Agent Mode falls back to a cached or heuristic decision rather than hanging. |
| — | **Kaggle Agent** | NEW coordination component (Cell 10, tab `🕹️ Kaggle Agent`) | The coordinator. Runs **on** the 17-layer reasoning service (module `kaggle_agent`, RAW REASONING). Jobs: (a) decide which additional databases Module 1 should download; (b) write a log-aware design-only prompt for Module 3 (Ask-a-Question); (c) use a pinned Franzen/Duck multi-file source tree (or uploaded exact source ZIP) to emit a reviewable unified patch, changed-files ZIP, and manifest. The legacy single-file `.py` emitter remains available. Candidate generation is separate from execution, Kaggle upload, and scoring. |

**Kept intact:** all existing knowledge databases (Cell 3) and live scholarly searchers (Cell 6); the 17-layer pipeline internals (Cell 7); the STANDARD FORM schema; the 12-agent orchestrator interface; the AI Rewriter tab; Data Dashboard; Agent Status.

**Environment (approved addition):** the notebook now auto-detects its runtime — Colab (`google.colab.ai`, OAuth, Drive — exactly as before), Kaggle (`kaggle_secrets`/`GEMINI_API_KEY`, data under `/kaggle/working`), or local (API key prompt). On Colab all storage paths are byte-identical to the base notebook, so existing databases are reused, not recreated.

---

## Franzen/Duck baseline mode

The new **Ask a Question → Kaggle Agent path** can load the reviewed Franzen repository snapshot at commit `b2fee614056f0609b94e214f03372f15894c00fe`, or an uploaded ZIP of the exact source used by a run. It emits a reviewable multi-file patch, a ZIP containing changed files only, and a provenance/validation manifest. It edits only selected existing files or line ranges, performs syntax/JSON checks, and never executes, uploads, submits, or scores a candidate. The frozen v3 single-file artifact builder and starter-kit pipeline in Cells 13–17 remain a separate legacy path; they are not silently treated as the Franzen baseline.

## Interface contracts

### Shared service — `reason(input, mode, module, context, budget_s, use_cache) -> structured_output` (Cell 8)
- **Input:** `input` (str: question / decision prompt / text to reason about), `mode` (`"standard"` | `"raw"`), `module` (`ask_question` | `agent_builder` | `agent_mode` | `database_downloader` | `kaggle_agent`), `context` (optional goal, forwarded to L0), `budget_s` (per-call budget override), `use_cache`.
- **STANDARD FORM output (schema unchanged):** `{score_before, score_after, status, outputs: {L0..L10}, audit, l9_questions}` — `outputs["L4"]` is the Final Rewrite (the answer); `audit` is the L10 certification.
- **RAW REASONING output:** `{draft, final, trace: {L0..L10}, score_before, score_after, status, audit, l9_questions}` — `final` is the L4 output; `draft` is the pre-pipeline base-LLM draft (also the heuristic fallback value).
- Never raises for timeouts; applies the module's fallback policy instead. Metadata of the last call is in `REASONING_LAST_META`.

### Module 1 — Database Downloader
- `decide_downloads(note) -> (ids_csv, rationale_text)` — **decision step, service mode: RAW REASONING.** No heuristic fallback (policy `error`).
- `download_selected(ids_csv) -> log_text` — **raw I/O, no pipeline.** Catalog ids only; per-download size cap (`DOWNLOADER_MAX_BYTES`, default 2 GB); every attempt logged to `DATA_ROOT/4cbon2_download_log.jsonl`.
- `get_download_log_text() -> history_text`.
- Not callable by Ask-a-Question (no code path exists between them).

### Module 3 — Ask a Question
- `ask_question_standard(prompt) -> STANDARD FORM` — general questions. `ask_question_kaggle(prompt) -> STANDARD FORM` — Kaggle design-only route, same shared service under module `kaggle_agent`, no databases, retrieval, or downloader calls.
- Legacy toggle: `handle_ask_question(kb_name, question, use_live_databases, return_report)` — unchanged RAG path (user-approved retention).

### Module 4 — Agent Builder
- `generate_builder_proposal(notebook_path, direction, include_ask_output, ask_context) -> (summary_md, proposals_json, status)` — design decisions via `builder_decision_llm` (**RAW REASONING**, module `agent_builder`).
- `five_lens_verdict(change) -> verdict_dict` — via the same service route.
- `apply_proposals(proposals_json, notebook_path) -> log_text` — unchanged safety pipeline (syntax validation, backup, transparency log).
- Consumes Ask-a-Question STANDARD FORM via the `ask_context` hand-off.

### Module 5 — Agent Mode
- `_decision_llm(prompt, max_tokens, step) -> decision_text` — routes orchestrator decision steps through `agent_decision_llm` (**RAW REASONING**, module `agent_mode`) with the per-action latency guard; falls back to the cached/heuristic direct decision on timeout.
- Trivial operations (tool calls: `web_search`, `read_file`, `query_database`, raw HTTP, keystroke-level actions) stay direct — never routed.

### Kaggle Agent
- `kaggle_decide_databases() -> (clean_text, ids_csv)` — job (a), **RAW REASONING**, module `kaggle_agent`; decision handed to Module 1.
- `kaggle_write_ask_prompt(log_text) -> clean_prompt_text` — job (b), **RAW REASONING**; prompt text only, grounded in pasted logs.
- `kaggle_run_ask_question(prompt) -> STANDARD FORM` — runs the design prompt through the Kaggle Ask-a-Question route.
- `kaggle_prepare_franzen_baseline(source_zip=None, commit=...)` — loads a pinned Franzen repository revision or a user-uploaded source ZIP; records source identity and a tree hash.
- The multi-file emitter accepts explicit repo-relative file paths or line ranges, produces a unified diff plus changed-file ZIP and manifest, and does **not** execute, upload, submit, or score candidates.
- `kaggle_emit_franzen_patch(design, logs, prompt, baseline, target_specs) -> (patch.diff, changed_files.zip, manifest.json, status)` — primary multi-file candidate path; edits only selected existing source files/ranges, checks Python/JSON syntax, and never executes, uploads, or scores.
- `kaggle_emit_submission_agent(design, logs, prompt, baseline_file) -> (candidate.py, manifest.json, status)` — retained legacy single-file path; syntax/API checks only, never executes or pushes the candidate.

---

## Config knobs (`REASONING_CONFIG`, Cell 8)

| Knob | Values | Default |
|------|--------|---------|
| Per-module reasoning budget (`budgets_s`) | seconds per `reason()` call | `ask_question` 900 · `agent_builder` 900 · `agent_mode` 60 · `database_downloader` 600 · `kaggle_agent` 900 |
| Fallback policy (`fallback_policy`) | `heuristic` (cached/direct fallback, logged) or `error` (status-marked result, no silent degradation) | `agent_mode` heuristic (mandated latency guard) · `agent_builder` heuristic · `ask_question` error · `database_downloader` error · `kaggle_agent` error |
| Caching policy (`cache`) | enabled, per-module list, max entries | enabled for all five modules, 200 entries, keyed `(module, mode, sha256(input))`; timed-out runs that complete in the background still populate the cache; `clear_reasoning_cache()` wipes it |
| Draft size (`draft_max_tokens`) | max tokens for the pre-pipeline base-LLM draft | 4096 |
| Download cap (`DOWNLOADER_MAX_BYTES`, Cell 9) | bytes per download | 2 GB (env var `DOWNLOADER_MAX_BYTES`) |


## Kaggle4 additions — log-driven design and candidate emission

- Ask-a-Question now has two explicit paths: **General** (ordinary questions and its legacy research toggle) and **Kaggle Agent path** (pasted Kaggle logs → prompt-only Kaggle Agent step → design answer).
- The Kaggle Agent uses the shared reasoning service with the `kaggle_agent` module key. It writes a log-aware, design-only prompt; Ask-a-Question then runs that prompt in the Kaggle-specific route.
- After the design, the Kaggle Agent can generate a **candidate** Python agent by proposing exact source edits against an uploaded single-file baseline. The candidate is syntax-checked and its public top-level API is checked before it is written. Logs, prompt, design, patch, and hashes are saved in a run folder.
- Candidate generation does not execute generated code, alter the frozen v3 build, push to Kaggle, or submit a score. Review and run the relevant offline/game tests before using it. The exact submitted Duck harness source must be supplied as the baseline; it is not included in the current repository snapshot.


In [ ]:
# ============================================================
# CELL 1 — Environment Setup (Gemini2 Frontier Research Edition — Colab / Kaggle / local)
# ============================================================
# Environment-agnostic rebuild of the original Colab-only setup.
#   Colab:  google.colab.ai runtime (OAuth, no API key) + Drive — exactly as before.
#   Kaggle: GEMINI_API_KEY via kaggle_secrets / env vars / prompt; data under /kaggle/working.
#   Local:  same API-key path; data under ./4cbon2_data.
# On Colab every storage path is byte-identical to 4CBOn2_Gemini2b.ipynb,
# so all existing databases are reused, not recreated.

!pip install -q gradio chromadb PyPDF2 python-docx duckduckgo-search beautifulsoup4 fpdf2 sentence-transformers plotly

import gradio as gr
import chromadb
from chromadb.config import Settings
import os
import torch
import json
import re
import sqlite3
import time
import requests
import csv
from datetime import datetime
from PyPDF2 import PdfReader
import docx
from duckduckgo_search import DDGS
from bs4 import BeautifulSoup
from fpdf import FPDF
import plotly.express as px
import plotly.graph_objects as go

# ── Environment detection ─────────────────────────────────────
IN_COLAB = False
IN_KAGGLE = False
ai = None
try:
    import google.colab.ai as ai  # noqa: F401
    IN_COLAB = True
except Exception:
    try:
        import google.colab  # noqa: F401
        IN_COLAB = True
    except Exception:
        pass

if os.environ.get("KAGGLE_KERNEL_RUN_TYPE") or os.path.exists("/kaggle/input"):
    IN_KAGGLE = True

RUNTIME = "colab" if IN_COLAB else ("kaggle" if IN_KAGGLE else "local")

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    DATA_ROOT = "/content/drive/MyDrive"
elif IN_KAGGLE:
    DATA_ROOT = "/kaggle/working"
else:
    DATA_ROOT = os.path.join(os.getcwd(), "4cbon2_data")

os.makedirs(DATA_ROOT, exist_ok=True)

# ── Gemini access outside Colab (Kaggle / local) ──────────────
if not IN_COLAB:
    def _load_gemini_api_key():
        for var in ("GEMINI_API_KEY", "GOOGLE_API_KEY"):
            if os.environ.get(var):
                return os.environ[var].strip()
        try:
            from kaggle_secrets import UserSecretsClient
            key = UserSecretsClient().get_secret("GEMINI_API_KEY")
            if key:
                os.environ["GEMINI_API_KEY"] = key.strip()
                return key.strip()
        except Exception:
            pass
        try:
            import getpass
            key = getpass.getpass("Enter a Gemini API key (https://aistudio.google.com/apikey): ").strip()
            if key:
                os.environ["GEMINI_API_KEY"] = key
                return key
        except Exception:
            pass
        return ""
    _GEMINI_API_KEY = _load_gemini_api_key()

print("✅ Environment ready.")
print("Runtime:", RUNTIME, "| DATA_ROOT:", DATA_ROOT)
print("PyTorch version:", torch.__version__)
print("GPU available:", torch.cuda.is_available())

if IN_COLAB and ai is not None:
    print("\nAvailable models:")
    available_models = ai.list_models()
    for model in available_models:
        print(f"  - {model}")
    print(f"\n✅ google.colab.ai ready with {len(available_models)} model(s)")
elif IN_COLAB:
    print("\n✅ Colab detected (google.colab.ai unavailable — Cell 2 will use the API-key path).")
else:
    print("\n✅ API-key mode ready.")


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.8/88.8 kB 2.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 36.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.6/232.6 kB 13.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 13.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 341.3/341.3 kB 13.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 10.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.5/7.5 MB 56.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.6/23.6 MB 39.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.1/140.1 kB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.3/60.3 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2

In [ ]:
# ============================================================
# CELL 2 — Core LLM (Gemini2 Frontier Research — Colab google.colab.ai OR Gemini API key)
# ============================================================
# Same function surface as the base notebook (generate_text / ask_raw /
# safe_ask_raw / ask_stream / ask). On Colab with google.colab.ai available,
# behavior is identical to the base. Elsewhere (Kaggle / local / Colab without
# google.colab.ai) the same Gemini model is called through the public REST API
# with an API key loaded in Cell 1.

try:
    import google.colab.ai as _colab_ai
    COLAB_AI_AVAILABLE = _colab_ai is not None
except Exception:
    _colab_ai = None
    COLAB_AI_AVAILABLE = False

if COLAB_AI_AVAILABLE:
    ai = _colab_ai
    # Use the first model made available by the Colab runtime.
    available_models = ai.list_models()
    MODEL_NAME = available_models[0] if available_models else 'google/gemini-2.5-flash'

    def generate_text(prompt, max_tokens=4096, temperature=0.4, stream=False):
        """Generate text using google.colab.ai (OAuth; no API key required)."""
        try:
            full_response = []
            for chunk in ai.generate_text(prompt=prompt, model_name=MODEL_NAME, stream=True):
                if chunk is not None:
                    full_response.append(chunk)
            return "".join(full_response)
        except Exception as e:
            return f"⚠️ API Error: {str(e)}"
else:
    _GEMINI_API_KEY = os.environ.get("GEMINI_API_KEY") or os.environ.get("GOOGLE_API_KEY") or ""
    MODEL_NAME = os.environ.get("GEMINI_MODEL", "gemini-2.5-flash")

    def generate_text(prompt, max_tokens=4096, temperature=0.4, stream=False):
        """Generate text via the Gemini REST API (API-key mode)."""
        if not _GEMINI_API_KEY:
            return "⚠️ API Error: no Gemini API key configured (set GEMINI_API_KEY or attach the Kaggle secret 'GEMINI_API_KEY')."
        try:
            url = f"https://generativelanguage.googleapis.com/v1beta/models/{MODEL_NAME}:generateContent"
            payload = {
                "contents": [{"parts": [{"text": prompt}]}],
                "generationConfig": {"maxOutputTokens": max_tokens, "temperature": temperature},
            }
            response = requests.post(url, json=payload, timeout=300,
                                     params={"key": _GEMINI_API_KEY},
                                     headers={"Content-Type": "application/json"})
            if response.status_code != 200:
                return f"⚠️ API Error: HTTP {response.status_code} — {response.text[:300]}"
            data = response.json()
            candidates = data.get("candidates") or [{}]
            parts = (candidates[0].get("content") or {}).get("parts") or []
            text = "".join(str(p.get("text", "")) for p in parts)
            if not text.strip():
                return '{"error": "Empty response from Gemini API."}'
            return text
        except Exception as e:
            return f"⚠️ API Error: {str(e)}"

# ── Everything below is byte-identical to the base notebook ──
def ask_raw(prompt, max_tokens=4096):
    return generate_text(prompt, max_tokens=max_tokens, temperature=0.1, stream=False)

def safe_ask_raw(prompt, max_tokens=4096):
    try:
        result = ask_raw(prompt, max_tokens=max_tokens)
        if not result or not result.strip():
            return '{"error": "Empty response from LLM. Please try again."}'
        if hasattr(result, "__iter__") and not isinstance(result, (str, dict, list)):
            result = "".join(list(result))
        if not isinstance(result, str):
            result = str(result)
        return result.strip()
    except Exception as e:
        return f'{{"error": "safe_ask_raw failed: {str(e)}"}}'

def ask_stream(question, context=None):
    """Stream a source-grounded, five-part frontier-research answer."""
    prompt_template = """You are the 4CBON2 Frontier Research Assistant: a rigorous, multidisciplinary research partner in artificial intelligence, mathematics, computer science, and the natural sciences.

The user may ask an ambitious open-ended question such as how to engineer an AGI-oriented agent, how one might attempt a Millennium Prize Problem, or how to investigate an unresolved scientific problem. Give the most useful answer that present evidence permits, but never imply that AGI has already been achieved or that an open theorem has been proved when it has not.

EVIDENCE RULES
- The RESEARCH CONTEXT contains retrieved records labelled [S1], [S2], and so on. Treat snippets as leads, not automatically as established truth.
- Cite a source as [S#] only when that exact source supports the claim. Never invent a citation, paper, theorem, result, experiment, URL, or database record.
- Distinguish established results, informed inference, and speculation. Mention conflicting evidence and missing data.
- Prefer primary literature, official problem statements, standards, and reproducible evidence. State when a claim needs expert or current-source verification.
- For a mathematical proof attempt: state definitions and assumptions, identify the exact new lemma needed, test edge cases and known obstructions, and label every unproved step. Do not present a sketch as a proof.
- For an AGI-oriented system: separate currently buildable components from AGI hypotheses; include architecture, data, tools, memory, planning, evaluation, security, alignment, and staged experiments.
- For scientific questions: propose falsifiable hypotheses, controls, measurements, uncertainty analysis, replication, and ethical/safety constraints where pertinent.

ANSWER FORMAT — use these five clear headings:
1. PROBLEM FORMULATION — Define the goal, scope, assumptions, success criteria, and whether it is open or unresolved.
2. EVIDENCE AND ANALOGIES — Synthesize the retrieved evidence and the most relevant parallels, with [S#] citations.
3. CANDIDATE APPROACH — Give a concrete architecture, proof strategy, model, experiment, or research program. Break it into executable stages.
4. CRITICAL TESTS — Identify failure modes, counterexamples, bottlenecks, safety issues, and decisive validation tests.
5. BEST CURRENT ANSWER — Answer directly; separate what can be done now from what remains unknown, and list the next three highest-value actions.

End with a short SOURCES USED section listing only the [S#] records actually cited. If the context is weak or unavailable, say so and provide a clearly labelled provisional answer instead of fabricating support.

{context_prefix}QUESTION: {question}
"""
    context_prefix = ""
    if context:
        context_prefix = f"RESEARCH CONTEXT:\n{context}\n\n"
    formatted_prompt = prompt_template.format(question=question, context_prefix=context_prefix)
    full_text = generate_text(formatted_prompt, max_tokens=4096, temperature=0.35, stream=False)
    if full_text.startswith("⚠️"):
        yield full_text
        return
    words = full_text.split()
    chunk = ""
    for i, word in enumerate(words):
        chunk += word + " "
        if (i + 1) % 5 == 0 or i == len(words) - 1:
            yield chunk
            chunk = ""

def ask(question, context=None):
    """Get a complete answer (non-streaming aggregation)."""
    return "".join(ask_stream(question, context=context))


print(f"✅ Cell 2 ready. Frontier research model: {MODEL_NAME}")
if COLAB_AI_AVAILABLE:
    print("Using google.colab.ai — zero configuration, no API keys needed.")
else:
    print(f"Using the Gemini REST API ({RUNTIME} runtime) — model {MODEL_NAME}.")

✅ Cell 2 ready. Frontier research model: google/gemini-2.5-flash
Using google.colab.ai — zero configuration, no API keys needed.


In [ ]:
# ============================================================
# CELL 3 — Strong AI, Mathematics & Science Knowledge Databases
# ============================================================

import hashlib

# Drive is mounted in Cell 1 (Colab only). The path is identical to the base
# notebook on Colab, so existing databases are reused, never overwritten.
drive_path = os.path.join(DATA_ROOT, 'chroma_db_gemini2_frontier_research')
os.makedirs(drive_path, exist_ok=True)

client = chromadb.PersistentClient(
    path=drive_path,
    settings=Settings(allow_reset=True)
)

# Uploaded documents remain in their own collection. The three domain collections
# below are seeded here and grow as live scholarly records are retrieved in Cell 6.
COLLECTION_NAME = "frontier_research_uploads"
DOMAIN_COLLECTIONS = {
    "ai": "strong_ai_database",
    "mathematics": "strong_mathematics_database",
    "science": "strong_science_database",
}

CURATED_DATABASES = {
    "ai": [
        {
            "title": "AGI scope and claims",
            "text": "There is no universally accepted operational definition or demonstrated implementation of artificial general intelligence. An AGI-oriented engineering project should declare measurable capability, generalization, autonomy, resource, robustness, and safety criteria rather than treating AGI as a binary label.",
            "url": "https://www.nist.gov/artificial-intelligence"
        },
        {
            "title": "Agent architecture baseline",
            "text": "A currently buildable AI agent can combine a foundation model with task decomposition, a constrained tool interface, retrieval, working and episodic memory, planning, execution monitoring, reflection, and human approval gates. Every tool action should be typed, permissioned, logged, reversible where possible, and evaluated independently of fluent output.",
            "url": "https://arxiv.org/abs/2309.07864"
        },
        {
            "title": "World models and planning",
            "text": "General-purpose agents need models that predict consequences under interventions, not only next-token continuation. Candidate research directions include learned world models, model-based reinforcement learning, search, causal representation learning, hierarchical planning, and continual adaptation under distribution shift.",
            "url": "https://arxiv.org/list/cs.AI/recent"
        },
        {
            "title": "Memory and retrieval",
            "text": "Agent memory should separate immutable instructions, short-lived working state, episodic traces, and curated semantic knowledge. Retrieval quality must be measured with relevance, provenance, freshness, access-control, poisoning-resistance, and downstream task metrics.",
            "url": "https://arxiv.org/list/cs.CL/recent"
        },
        {
            "title": "Evaluation before autonomy",
            "text": "Evaluate an agent on held-out tasks, contamination-resistant tests, calibration, tool success, long-horizon reliability, adversarial robustness, cost, latency, and safe refusal. Capability benchmarks alone do not establish general intelligence or safe deployment.",
            "url": "https://crfm.stanford.edu/helm/"
        },
        {
            "title": "Risk management",
            "text": "A trustworthy AI development process maps risks, measures them, manages them, and governs the full lifecycle. High-impact autonomous actions require least privilege, sandboxing, rate limits, monitoring, incident response, and human accountability.",
            "url": "https://www.nist.gov/itl/ai-risk-management-framework"
        },
        {
            "title": "AI research literature database directory",
            "text": "Pertinent AI evidence sources include arXiv for preprints, OpenAlex and Crossref for scholarly metadata, Semantic Scholar for citation-linked discovery, Papers with Code for implementations and benchmarks, and official standards or benchmark sites for current protocols. Preprints should not be treated as peer-reviewed evidence.",
            "url": "https://openalex.org/"
        },
        {
            "title": "Reproducible AI experiments",
            "text": "A credible AI research result records datasets and licenses, train-validation-test splits, contamination checks, model and optimizer versions, seeds, compute, ablations, uncertainty intervals, negative results, and an executable evaluation harness.",
            "url": "https://paperswithcode.com/"
        },
        {
            "title": "Alignment as an empirical program",
            "text": "Alignment work includes specification, oversight, interpretability, robustness, scalable evaluation, red teaming, monitoring, and governance. A claim of alignment requires explicit threat models and evidence across anticipated and unanticipated operating conditions.",
            "url": "https://www.nist.gov/artificial-intelligence"
        },
        {
            "title": "AGI-oriented staged roadmap",
            "text": "A defensible roadmap starts with a narrow sandboxed agent, establishes a baseline, adds one capability at a time, runs adversarial and long-horizon evaluations, studies generalization and transfer, and expands permissions only when evidence satisfies predefined safety gates.",
            "url": "https://crfm.stanford.edu/helm/"
        },
    ],
    "mathematics": [
        {
            "title": "Clay Millennium Prize Problems — official source",
            "text": "The official Clay Mathematics Institute problem descriptions and rules are the authority for the Millennium Prize Problems. Before attempting a problem, retrieve the current official statement and status, define every term exactly, and identify which claimed step is not already known.",
            "url": "https://www.claymath.org/millennium-problems/"
        },
        {
            "title": "P versus NP",
            "text": "P versus NP asks whether every decision problem whose proposed solutions can be verified in polynomial time can also be solved in polynomial time. Any approach must respect relativization, natural-proofs, and algebrization barriers where applicable and must specify the computational model and uniformity assumptions.",
            "url": "https://www.claymath.org/millennium/p-vs-np/"
        },
        {
            "title": "Riemann Hypothesis",
            "text": "The Riemann Hypothesis asserts that every nontrivial zero of the Riemann zeta function has real part one half. Numerical verification of many zeros is evidence but not a proof; an attempt must bridge analytic continuation, the functional equation, and a valid argument covering all nontrivial zeros.",
            "url": "https://www.claymath.org/millennium/riemann-hypothesis/"
        },
        {
            "title": "Navier–Stokes existence and smoothness",
            "text": "The three-dimensional incompressible Navier–Stokes problem asks for a proof of global smooth solutions under the specified initial conditions or a valid breakdown example. Computation and turbulence intuition cannot replace the required global estimates and exact regularity argument.",
            "url": "https://www.claymath.org/millennium/navier-stokes-equation/"
        },
        {
            "title": "Yang–Mills existence and mass gap",
            "text": "The Yang–Mills problem requires a mathematically rigorous construction of quantum Yang–Mills theory on four-dimensional Euclidean space for a compact simple gauge group and proof of a positive mass gap. Perturbative or numerical physics evidence alone does not meet the statement.",
            "url": "https://www.claymath.org/millennium/yang-mills-the-maths-gap/"
        },
        {
            "title": "Hodge Conjecture",
            "text": "The Hodge Conjecture concerns whether certain rational cohomology classes of smooth projective complex varieties are rational linear combinations of classes of algebraic cycles. An attempt must preserve the exact rational, projective, and smooth hypotheses and distinguish known special cases.",
            "url": "https://www.claymath.org/millennium/hodge-conjecture/"
        },
        {
            "title": "Birch and Swinnerton-Dyer Conjecture",
            "text": "The Birch and Swinnerton-Dyer Conjecture relates the rank of an elliptic curve over the rationals to the order of vanishing of its L-function at one. Experimental agreement or finite computations do not establish the general statement.",
            "url": "https://www.claymath.org/millennium/birch-and-swinnerton-dyer-conjecture/"
        },
        {
            "title": "Poincaré Conjecture",
            "text": "The Poincaré Conjecture is the solved Millennium Prize Problem, resolved through Grigori Perelman's work on Ricci flow building on Richard Hamilton. It is useful as a model of deep proof verification, but it should not be presented as still open.",
            "url": "https://www.claymath.org/millennium/poincare-conjecture/"
        },
        {
            "title": "Proof-attempt protocol",
            "text": "A responsible open-problem attempt starts from the official statement, maps equivalent formulations and known partial results, selects the smallest plausible new lemma, proves it with all quantifiers explicit, actively searches for counterexamples, and obtains independent specialist review. A gap, numerical pattern, or unchecked symbolic derivation is not a proof.",
            "url": "https://www.claymath.org/millennium-problems/"
        },
        {
            "title": "Formal verification",
            "text": "Proof assistants can expose missing assumptions and mechanically check a formalized argument, but formalization does not make a false key lemma true. Lean's mathlib is a large community mathematical library useful for checking dependencies and machine-verifiable steps.",
            "url": "https://leanprover-community.github.io/mathlib4_docs/"
        },
        {
            "title": "Mathematics database directory",
            "text": "Useful mathematics sources include arXiv mathematics categories for preprints, OpenAlex and Crossref for discovery and DOI metadata, Semantic Scholar for citation exploration, zbMATH Open for mathematical indexing, OEIS for integer sequences, LMFDB for explicit number-theoretic objects, and MathSciNet where access is available.",
            "url": "https://zbmath.org/"
        },
        {
            "title": "Literature and priority check",
            "text": "Before claiming a new theorem, search multiple independent indexes, trace citations to primary papers, verify whether the result has an existing name or stronger form, and record exact bibliographic identifiers. Lack of a search result is not evidence of novelty.",
            "url": "https://www.crossref.org/"
        },
    ],
    "science": [
        {
            "title": "Scientific inference baseline",
            "text": "A scientific answer should distinguish observations, measurement models, causal hypotheses, predictions, and decisions. Good hypotheses are falsifiable, compared against alternatives, and evaluated with uncertainty rather than selected only because they fit existing data.",
            "url": "https://www.nist.gov/services-resources"
        },
        {
            "title": "Experimental design",
            "text": "A strong experiment pre-registers primary outcomes where practical, uses appropriate controls, randomization and blinding where applicable, justifies sample size, defines exclusion criteria in advance, and reports effect sizes and uncertainty intervals rather than relying only on thresholded significance.",
            "url": "https://www.ncbi.nlm.nih.gov/"
        },
        {
            "title": "Reproducibility",
            "text": "Reproducible science records raw-data provenance, calibration, protocols, software and environment versions, analysis code, sensitivity analyses, and negative results. Independent replication and convergent measurement are stronger than repeated analysis of one dataset.",
            "url": "https://www.nist.gov/"
        },
        {
            "title": "PubMed",
            "text": "PubMed is a primary discovery database for biomedical and life-science literature maintained by the US National Library of Medicine. Search results are bibliographic records; study design and full text must be assessed before using a result as evidence.",
            "url": "https://pubmed.ncbi.nlm.nih.gov/"
        },
        {
            "title": "Europe PMC",
            "text": "Europe PMC indexes life-science publications, preprints, grants, and links to openly available full text. Version and peer-review status should be checked because a preprint and its later journal article can differ.",
            "url": "https://europepmc.org/"
        },
        {
            "title": "Cross-disciplinary literature",
            "text": "OpenAlex, Crossref, and Semantic Scholar support broad scholarly discovery and citation tracing. Their metadata can be incomplete or duplicated, so decisive claims should be checked against the primary paper and publisher or repository record.",
            "url": "https://openalex.org/"
        },
        {
            "title": "Preprints",
            "text": "arXiv provides rapid access to physics, mathematics, computer science, quantitative biology, statistics, and related preprints. A preprint may be valuable and current but should not be described as peer reviewed unless a separate journal record confirms that status.",
            "url": "https://arxiv.org/"
        },
        {
            "title": "Domain-specific science database directory",
            "text": "Depending on the question, pertinent sources may include NASA ADS for astronomy and physics, NCBI databases for biology, Protein Data Bank for structures, UniProt for proteins, ClinicalTrials.gov for registered trials, GenBank for sequences, USGS for earth science, and NIST for standards and reference data.",
            "url": "https://www.ncbi.nlm.nih.gov/home/data/"
        },
        {
            "title": "Causal claims",
            "text": "Causal conclusions require a defensible identification strategy such as randomization, natural experiments, valid instruments, discontinuities, longitudinal controls, or an explicit causal model with sensitivity analysis. Correlation, prediction accuracy, and mechanistic plausibility alone are insufficient.",
            "url": "https://www.ncbi.nlm.nih.gov/"
        },
        {
            "title": "Safety and ethics",
            "text": "Research involving people, animals, pathogens, hazardous materials, ecosystems, or dual-use capabilities requires the applicable ethical review, biosafety, security, consent, privacy, and regulatory controls before execution. A literature-generated protocol is not a substitute for qualified oversight.",
            "url": "https://www.nih.gov/health-information/nih-clinical-research-trials-you/basics"
        },
        {
            "title": "Model validation",
            "text": "A scientific model should be checked for dimensional consistency, limiting cases, parameter identifiability, out-of-sample prediction, residual structure, robustness to plausible measurement error, and comparison with simpler baselines.",
            "url": "https://www.nist.gov/services-resources/software"
        },
    ],
}

def _get_or_create_collection(name):
    try:
        return client.get_collection(name=name)
    except Exception:
        return client.create_collection(name=name)

# Seed or update all domain databases deterministically, without duplicating rows.
domain_collections = {}
for domain, collection_name in DOMAIN_COLLECTIONS.items():
    col = _get_or_create_collection(collection_name)
    records = CURATED_DATABASES[domain]
    documents = [f"{r['title']}\n{r['text']}" for r in records]
    ids = [f"seed_{domain}_{i:03d}" for i in range(len(records))]
    metadatas = [{
        "source": "4CBON2 curated research index",
        "title": r["title"],
        "url": r["url"],
        "domain": domain,
        "type": "curated",
    } for r in records]
    col.upsert(documents=documents, ids=ids, metadatas=metadatas)
    domain_collections[domain] = col
    print(f"✅ {domain.title()} database ready: {col.count()} records")

collection = _get_or_create_collection(COLLECTION_NAME)
print(f"✅ Upload database ready: {collection.count()} records")
print("Collections available:", [c.name for c in client.list_collections()])

/root/.cache/chroma/onnx_models/all-MiniLM-L6-v2/onnx.tar.gz: 100%|██████████| 79.3M/79.3M [00:02<00:00, 30.2MiB/s]


✅ Ai database ready: 20 records
✅ Mathematics database ready: 21 records
✅ Science database ready: 11 records
✅ Upload database ready: 0 records
Collections available: ['frontier_research_uploads', 'strong_science_database', 'strong_mathematics_database', 'strong_ai_database']


In [ ]:
# ============================================================
# CELL 4 — 12 Agent Profiles + Tool Registry + DB Helpers
# ============================================================

AGENT_PROFILES = {
    "Default General Assistant": {
        "system_prompt": "You are a helpful general assistant operating within the 4CBON2 architecture.",
        "required_api": None
    },
    "New Autonomous Agent": {
        "system_prompt": """You are the Autonomous Orchestrator Agent for the 4CBON2 ecosystem.
Your role is to:
1. Receive a complex goal from the user.
2. Break it down into 2-4 concrete subtasks.
3. For each subtask, select the most appropriate specialist agent from the list below.
4. Delegate the subtask to that specialist and collect their response.
5. Synthesise all specialist responses into a final, cohesive answer.

Available specialist agents and their expertise:
- Sales Qualification: Lead scoring, BANT criteria, pipeline readiness.
- Legal Document Intelligence: Clause analysis, regulatory compliance, liability extraction.
- Competitive Intelligence: Competitor tracking, market shifts, positioning analysis.
- Customer Engagement: Messaging, sentiment parsing, communication routing.
- Content Strategy: Editorial calendars, copy structuring, keyword architecture.
- Marketing Automation: Campaign triggers, conversion funnels, broadcast sequencing.
- Evidence Management: Data cross-referencing, source auditing, factual verification.
- Scheduling: Time-block coordination, calendar management, bottleneck resolution.
- Legal Intake: Client screening, conflict checks, disclosure structuring.
- Scientific Research: Literature synthesis, data parsing, hypothesis evaluation.
""",
        "required_api": None
    },
    "Sales Qualification": {
        "system_prompt": "You are a Sales Qualification agent. Focus on lead scoring, BANT criteria assessment, and pipeline readiness tracking.",
        "required_api": "CRM_API_KEY"
    },
    "Legal Document Intelligence": {
        "system_prompt": "You are a Legal Document Intelligence agent. Analyze clauses, verify regulatory compliance, and extract liability terms from legal documents.",
        "required_api": "DOCUSIGN_API_KEY"
    },
    "Competitive Intelligence": {
        "system_prompt": "You are a Competitive Intelligence agent. Scrape competitor updates, track market shifts, and analyze positioning strategies.",
        "required_api": "SEO_API_KEY"
    },
    "Customer Engagement": {
        "system_prompt": "You are a Customer Engagement agent. Craft personalized messaging, parse inbound sentiment, and handle communications routing.",
        "required_api": "COMM_API_KEY"
    },
    "Content Strategy": {
        "system_prompt": "You are a Content Strategy agent. Optimize editorial calendars, structure high-converting copy, and manage keyword architecture.",
        "required_api": "SEO_API_KEY"
    },
    "Marketing Automation": {
        "system_prompt": "You are a Marketing Automation agent. Orchestrate campaign triggers, analyze conversion funnels, and manage broadcast sequences.",
        "required_api": "SOCIAL_SCRAPER_API_KEY"
    },
    "Evidence Management": {
        "system_prompt": "You are an Evidence Management agent. Cross-reference empirical data, audit source trails, and verify factual consistency.",
        "required_api": "S3_VAULT_KEY"
    },
    "Scheduling": {
        "system_prompt": "You are a Scheduling agent. Coordinate time-blocks, handle calendar availability, and resolve logistical bottlenecks.",
        "required_api": "CALENDAR_API_KEY"
    },
    "Legal Intake": {
        "system_prompt": "You are a Legal Intake agent. Screen new client cases, check for conflicts of interest, and structure initial disclosures.",
        "required_api": "DOCUSIGN_API_KEY"
    },
    "Scientific Research": {
        "system_prompt": "You are a Scientific Research agent. Synthesize peer-reviewed literature, parse clinical or technical data, and evaluate hypotheses.",
        "required_api": "PUBMED_API_KEY"
    }
}

LOG_DIR = os.path.join(DATA_ROOT, "4cbon2_logs")
os.makedirs(LOG_DIR, exist_ok=True)
LOG_FILE = os.path.join(LOG_DIR, f"tool_log_{datetime.now().strftime('%Y%m%d')}.jsonl")

def log_tool_call(tool_name, input_data, result):
    try:
        with open(LOG_FILE, "a") as f:
            f.write(json.dumps({
                "timestamp": datetime.now().isoformat(),
                "tool": tool_name,
                "input": str(input_data)[:500],
                "result_preview": str(result)[:500]
            }) + "\n")
    except Exception as e:
        print(f"⚠️ Log warning: {e}")

STOPWORDS = {
    "of", "the", "a", "an", "for", "to", "in", "on", "is", "are", "and", "or",
    "competitors", "competitor", "alternatives", "alternative", "best", "app",
    "apps", "software", "productivity", "who", "what", "current", "list",
    "similar", "tools", "top", "rated", "reviews", "review", "latest", "new"
}

def _is_relevant(query, text):
    words = re.findall(r"\w+", query)
    keywords = [w for w in words if w.lower() not in STOPWORDS and not (w.isdigit() and len(w) < 4)]
    if not keywords:
        return True
    text_lower = text.lower()
    for kw in keywords:
        if kw.lower() in text_lower:
            return True
    return False

def web_search(query):
    try:
        with DDGS() as ddgs:
            results = list(ddgs.text(query, max_results=5))
        if not results:
            return "No results found."
        formatted = []
        for r in results:
            title = r.get("title", "")
            body = r.get("body", "")
            href = r.get("href", "")
            formatted.append(f"{title}\n{body}\n{href}")
        combined = "\n\n".join(formatted)
        return combined if _is_relevant(query, combined) else "Results found but not highly relevant."
    except Exception as e:
        return f"Search error: {e}"

def read_file(file_path):
    try:
        if file_path.endswith(".txt"):
            with open(file_path, "r", errors="ignore") as f:
                return f.read()
        elif file_path.endswith(".pdf"):
            reader = PdfReader(file_path)
            texts = []
            for page in reader.pages:
                t = page.extract_text()
                if t:
                    texts.append(t)
            return "\n".join(texts)
        elif file_path.endswith(".docx"):
            doc = docx.Document(file_path)
            return "\n".join([para.text for para in doc.paragraphs])
        else:
            return "Unsupported file type. Use .txt, .pdf, or .docx"
    except Exception as e:
        return f"File read error: {e}"

def query_database(sql, db_path=os.path.join(DATA_ROOT, "4cbon2_data.db")):
    try:
        cleaned = sql.strip().upper()
        if not cleaned.startswith("SELECT"):
            return "❌ Only SELECT queries are allowed for safety."
        conn = sqlite3.connect(db_path)
        cursor = conn.cursor()
        cursor.execute(sql)
        rows = cursor.fetchall()
        conn.close()
        return "\n".join([str(row) for row in rows]) if rows else "No results."
    except Exception as e:
        return f"Database error: {e}"

def save_note(content, filename=None):
    try:
        if filename is None:
            filename = f"note_{datetime.now().strftime('%Y%m%d_%H%M%S')}.txt"
        path = os.path.join(DATA_ROOT, "4cbon2_notes", filename)
        os.makedirs(os.path.dirname(path), exist_ok=True)
        with open(path, "w") as f:
            f.write(content)
        return f"Note saved to {path}"
    except Exception as e:
        return f"Save error: {e}"

def get_datetime():
    return datetime.now().strftime("Date: %Y-%m-%d | Time: %H:%M:%S")

def http_request(input_str):
    try:
        parsed = json.loads(input_str)
        url = parsed.get("url")
        if not url:
            return "Missing 'url' in input."
        fields = parsed.get("fields", [])
        response = requests.get(url, timeout=10, headers={"User-Agent": "Mozilla/5.0"})
        content_type = response.headers.get("Content-Type", "")
        if "application/json" in content_type:
            data = response.json()
        else:
            data = response.text
        if isinstance(data, dict) and len(json.dumps(data)) > 4000 and not fields:
            menu = {k: type(v).__name__ for k, v in data.items()}
            return f"Large response. Top-level keys: {json.dumps(menu, indent=2)}"
        if fields:
            result = {}
            for field in fields:
                parts = field.split(".")
                val = data
                for part in parts:
                    if isinstance(val, dict) and part in val:
                        val = val[part]
                    else:
                        val = None
                        break
                result[field] = val
            return json.dumps(result, indent=2)
        return json.dumps(data, indent=2)[:3000] if isinstance(data, (dict, list)) else str(data)[:3000]
    except Exception as e:
        return f"HTTP error: {e}"

def read_csv(file_path):
    try:
        with open(file_path, "r", newline="", errors="ignore") as f:
            rows = list(csv.reader(f))
        if not rows:
            return "CSV is empty."
        header = rows[0]
        preview = rows[1:6]
        return f"Columns: {', '.join(header)}\nRows: {len(rows)-1}\nPreview:\n" + "\n".join([str(r) for r in preview])
    except Exception as e:
        return f"CSV error: {e}"

def write_csv(data_json):
    try:
        rows = json.loads(data_json)
        if not isinstance(rows, list) or not rows:
            return "Input must be a non-empty list of dicts."
        filename = f"export_{datetime.now().strftime('%Y%m%d_%H%M%S')}.csv"
        path = os.path.join(DATA_ROOT, "4cbon2_exports", filename)
        os.makedirs(os.path.dirname(path), exist_ok=True)
        keys = list(rows[0].keys())
        with open(path, "w", newline="") as f:
            writer = csv.DictWriter(f, fieldnames=keys)
            writer.writeheader()
            writer.writerows(rows)
        return f"CSV saved to {path} ({len(rows)} rows)"
    except Exception as e:
        return f"CSV write error: {e}"

def generate_pdf(content):
    try:
        filename = f"report_{datetime.now().strftime('%Y%m%d_%H%M%S')}.pdf"
        path = os.path.join(DATA_ROOT, "4cbon2_reports", filename)
        os.makedirs(os.path.dirname(path), exist_ok=True)
        pdf = FPDF()
        pdf.add_page()
        pdf.set_font("Helvetica", size=12)
        for line in content.split("\n"):
            pdf.multi_cell(0, 8, line)
        pdf.output(path)
        return f"PDF saved to {path}"
    except Exception as e:
        return f"PDF error: {e}"

def scrape_webpage(url):
    try:
        response = requests.get(url, timeout=10, headers={"User-Agent": "Mozilla/5.0"})
        soup = BeautifulSoup(response.text, "html.parser")
        for tag in soup(["script", "style", "nav", "footer", "header"]):
            tag.decompose()
        text = soup.get_text(separator="\n")
        lines = [line.strip() for line in text.split("\n") if line.strip()]
        return "\n".join(lines)[:3000] if lines else "No readable content."
    except Exception as e:
        return f"Scrape error: {e}"

TOOL_REGISTRY = {
    "web_search": {"function": web_search, "description": "Search the web for current information. Input: search query string.", "input": "query"},
    "read_file": {"function": read_file, "description": "Read contents of a .txt, .pdf, or .docx file. Input: file path string.", "input": "file_path"},
    "query_database": {"function": query_database, "description": "Run a SELECT SQL query against the local SQLite database. Input: SQL string.", "input": "sql"},
    "save_note": {"function": save_note, "description": "Save a text note to Google Drive. Input: content string.", "input": "content"},
    "get_datetime": {"function": get_datetime, "description": "Get the current date and time. No input required.", "input": None},
    "http_request": {"function": http_request, "description": "Fetch data from a URL. Input: JSON string like {'url': '...', 'fields': ['field1']}.", "input": "input_str"},
    "read_csv": {"function": read_csv, "description": "Read a CSV file and return columns, row count, and preview. Input: file path string.", "input": "file_path"},
    "write_csv": {"function": write_csv, "description": "Export data to a CSV file on Drive. Input: JSON list of objects.", "input": "data_json"},
    "generate_pdf": {"function": generate_pdf, "description": "Generate a PDF report from text content and save it to Drive. Input: text content string.", "input": "content"},
    "scrape_webpage": {"function": scrape_webpage, "description": "Fetch a webpage and extract its main readable text. Input: URL string.", "input": "url"}
}

def execute_tool(tool_name, tool_input=None):
    if tool_name not in TOOL_REGISTRY:
        return f"Unknown tool: {tool_name}"
    tool = TOOL_REGISTRY[tool_name]
    try:
        if tool["input"] is None:
            result = tool["function"]()
        else:
            result = tool["function"](tool_input)
    except Exception as e:
        result = f"Tool execution error: {e}"
    log_tool_call(tool_name, tool_input, result)
    return result

AGENT_DB_PATH = os.path.join(DATA_ROOT, "4cbon2_agents.db")
os.makedirs(os.path.dirname(AGENT_DB_PATH), exist_ok=True)

def init_agent_db():
    conn = sqlite3.connect(AGENT_DB_PATH)
    cursor = conn.cursor()
    cursor.execute('''
        CREATE TABLE IF NOT EXISTS agents (
            agent_id TEXT PRIMARY KEY,
            system_prompt TEXT,
            conversation_history TEXT,
            tools TEXT,
            created_at TEXT,
            updated_at TEXT
        )
    ''')
    conn.commit()
    conn.close()

def load_agent(agent_id):
    conn = sqlite3.connect(AGENT_DB_PATH)
    cursor = conn.cursor()
    cursor.execute(
        "SELECT agent_id, system_prompt, conversation_history, tools FROM agents WHERE agent_id = ?",
        (agent_id,)
    )
    row = cursor.fetchone()
    conn.close()
    if row:
        return {
            "agent_id": row[0],
            "system_prompt": row[1],
            "conversation_history": json.loads(row[2]) if row[2] else [],
            "tools": json.loads(row[3]) if row[3] else []
        }
    return None

def save_agent(agent_id, system_prompt, conversation_history, tools=None):
    if tools is None:
        tools = []
    conn = sqlite3.connect(AGENT_DB_PATH)
    cursor = conn.cursor()
    cursor.execute('''
        INSERT OR REPLACE INTO agents (agent_id, system_prompt, conversation_history, tools, updated_at)
        VALUES (?, ?, ?, ?, ?)
    ''', (
        agent_id,
        system_prompt,
        json.dumps(conversation_history),
        json.dumps(tools),
        datetime.now().isoformat()
    ))
    conn.commit()
    conn.close()

def update_agent_conversation(agent_id, new_messages):
    agent = load_agent(agent_id)
    if agent is None:
        if agent_id in AGENT_PROFILES:
            agent = {
                "agent_id": agent_id,
                "system_prompt": AGENT_PROFILES[agent_id]["system_prompt"],
                "conversation_history": [],
                "tools": []
            }
        else:
            raise ValueError(f"Agent '{agent_id}' not found")
    agent["conversation_history"].extend(new_messages)
    save_agent(agent["agent_id"], agent["system_prompt"], agent["conversation_history"], agent["tools"])

def clear_agent_history(agent_id):
    agent = load_agent(agent_id)
    if agent:
        save_agent(agent_id, agent["system_prompt"], [], agent["tools"])

def get_all_agents():
    conn = sqlite3.connect(AGENT_DB_PATH)
    cursor = conn.cursor()
    cursor.execute("SELECT agent_id FROM agents")
    rows = cursor.fetchall()
    conn.close()
    return [row[0] for row in rows]

def ensure_agents_loaded():
    init_agent_db()
    for agent_id, profile in AGENT_PROFILES.items():
        if load_agent(agent_id) is None:
            save_agent(agent_id, profile["system_prompt"], [], [])
            print(f"✅ Agent '{agent_id}' created in DB.")

ensure_agents_loaded()

print("👥 12 Agent Profiles + 10 Tools loaded.")
print("Agents:", list(AGENT_PROFILES.keys()))
print("Tools:", list(TOOL_REGISTRY.keys()))

👥 12 Agent Profiles + 10 Tools loaded.
Agents: ['Default General Assistant', 'New Autonomous Agent', 'Sales Qualification', 'Legal Document Intelligence', 'Competitive Intelligence', 'Customer Engagement', 'Content Strategy', 'Marketing Automation', 'Evidence Management', 'Scheduling', 'Legal Intake', 'Scientific Research']
Tools: ['web_search', 'read_file', 'query_database', 'save_note', 'get_datetime', 'http_request', 'read_csv', 'write_csv', 'generate_pdf', 'scrape_webpage']


In [ ]:
# ============================================================
# CELL 5 — Streaming Multi-Agent Orchestrator
# ============================================================

import json
import re
import os
from datetime import datetime

# ── Shared reasoning service routing (Module 5 — Agent Mode) ──
# Decision steps below call _decision_llm, which routes the decision through
# the shared 17-layer reasoning service (Cell 8, agent_decision_llm) with the
# per-action latency guard; on timeout it falls back to a cached/heuristic
# direct decision instead of hanging. Trivial operations — tool execution,
# file reads, raw web/HTTP I/O — stay direct and never enter the pipeline.
def _decision_llm(prompt, max_tokens=2048, step="reasoning"):
    try:
        return agent_decision_llm(prompt, max_tokens=max_tokens, step=step)
    except NameError:
        # Reasoning service cell not executed — degrade to the direct call.
        return safe_ask_raw(prompt, max_tokens=max_tokens)

AUDIT_LOG_PATH = os.path.join(DATA_ROOT, "4cbon2_audit.jsonl")
os.makedirs(os.path.dirname(AUDIT_LOG_PATH), exist_ok=True)

def log_event(event_type, details):
    try:
        record = {
            "timestamp": datetime.now().isoformat(),
            "event_type": event_type,
            "details": details
        }
        with open(AUDIT_LOG_PATH, "a") as f:
            f.write(json.dumps(record) + "\n")
    except Exception as e:
        print(f"⚠️ Audit log warning: {e}")

TASK_MEMORY_PATH = os.path.join(DATA_ROOT, "4cbon2_task_memory.db")
os.makedirs(os.path.dirname(TASK_MEMORY_PATH), exist_ok=True)

def init_task_memory():
    conn = sqlite3.connect(TASK_MEMORY_PATH)
    cursor = conn.cursor()
    cursor.execute('''
        CREATE TABLE IF NOT EXISTS task_memory (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            goal TEXT,
            subtasks TEXT,
            final_answer TEXT,
            timestamp TEXT
        )
    ''')
    conn.commit()
    conn.close()

def save_task_memory(goal, subtasks, final_answer):
    conn = sqlite3.connect(TASK_MEMORY_PATH)
    cursor = conn.cursor()
    cursor.execute(
        "INSERT INTO task_memory (goal, subtasks, final_answer, timestamp) VALUES (?, ?, ?, ?)",
        (goal, subtasks, final_answer, datetime.now().isoformat())
    )
    conn.commit()
    conn.close()

init_task_memory()

def _extract_balanced(text, open_ch, close_ch):
    if not text:
        return None
    start = text.find(open_ch)
    if start == -1:
        return None
    depth = 0
    in_string = False
    escape = False
    for i in range(start, len(text)):
        ch = text[i]
        if in_string:
            if escape:
                escape = False
            elif ch == '\\':
                escape = True
            elif ch == '"':
                in_string = False
        else:
            if ch == '"':
                in_string = True
            elif ch == open_ch:
                depth += 1
            elif ch == close_ch:
                depth -= 1
                if depth == 0:
                    return text[start:i + 1]
    return None

def extract_json_object(text):
    return _extract_balanced(text, '{', '}')

def extract_json_array(text):
    return _extract_balanced(text, '[', ']')

def _parse_agent_json(raw_response):
    if not raw_response:
        return None
    candidate = extract_json_object(raw_response)
    try:
        if candidate:
            return json.loads(candidate)
        return json.loads(raw_response)
    except Exception:
        return None

def build_tool_descriptions():
    lines = []
    for name, info in TOOL_REGISTRY.items():
        input_desc = info["input"] if info["input"] else "None"
        lines.append(f"- **{name}**: {info['description']} (input: {input_desc})")
    return "\n".join(lines)

def execute_agent(agent_id, user_message, context="", max_tool_iterations=3):
    agent = load_agent(agent_id)
    if agent is None:
        return f"❌ Agent '{agent_id}' not found."

    system_prompt = agent["system_prompt"]
    history = agent.get("conversation_history", [])

    tool_instructions = f"""You have access to these tools:
{build_tool_descriptions()}

To use a tool, respond with ONLY this JSON:
{{"action": "tool_call", "tool": "<tool_name>", "tool_input": "<input or null>"}}

To answer directly, respond with ONLY this JSON:
{{"action": "final_answer", "content": "<your answer>"}}

Valid JSON only. Max {max_tool_iterations} tool calls before final_answer."""

    prompt_parts = [f"System: {system_prompt}", tool_instructions]
    if context:
        prompt_parts.append(f"Context from other agents:\n{context}")
    for msg in history[-4:]:
        prompt_parts.append(f"{msg['role']}: {msg['content']}")
    prompt_parts.append(f"User: {user_message}")

    tool_call_log = []
    final_content = None

    for iteration in range(max_tool_iterations):
        full_prompt = "\n\n".join(prompt_parts)
        raw_response = _decision_llm(full_prompt, max_tokens=2048, step="agent_tool_decision")

        parsed = _parse_agent_json(raw_response)

        if parsed is None:
            final_content = raw_response
            break

        action = parsed.get("action")

        if action == "tool_call":
            tool_name = parsed.get("tool", "")
            tool_input = parsed.get("tool_input")

            if tool_name not in TOOL_REGISTRY:
                prompt_parts.append(f"Assistant: {raw_response}")
                prompt_parts.append(f"Tool Result: ❌ Unknown tool '{tool_name}'. Available: {', '.join(TOOL_REGISTRY.keys())}")
                continue

            tool_result = execute_tool(tool_name, tool_input)
            tool_call_log.append({"tool": tool_name, "input": tool_input, "result": str(tool_result)[:300]})
            log_event("agent_tool_call", {
                "agent_id": agent_id,
                "tool": tool_name,
                "input": tool_input,
                "result_preview": str(tool_result)[:200]
            })

            prompt_parts.append(f"Assistant: {raw_response}")
            prompt_parts.append(f"Tool Result ({tool_name}): {str(tool_result)[:2000]}")
            continue

        elif action == "final_answer":
            final_content = parsed.get("content", raw_response)
            break
        else:
            final_content = raw_response
            break

    if final_content is None:
        forced_prompt = "\n\n".join(prompt_parts) + "\n\nYou must respond now with ONLY the final_answer JSON format."
        raw_response = _decision_llm(forced_prompt, max_tokens=2048, step="agent_final_answer")
        parsed = _parse_agent_json(raw_response)
        final_content = parsed.get("content", raw_response) if parsed else raw_response

    if not final_content or final_content.strip() == "" or "could not generate" in final_content.lower():
        fallback_prompt = f"You are a {agent_id} specialist. Provide a best-practice framework for your domain with key metrics, benchmarks, workflows, data collection methods, and improvement strategies."
        final_content = safe_ask_raw(fallback_prompt, max_tokens=1024)
        if not final_content or final_content.strip() == "":
            final_content = f"⚠️ {agent_id} could not generate a response. Please provide more specific instructions or data."

    if tool_call_log:
        tools_used_note = "\n\n---\n🔧 **Tools used:** " + ", ".join(t["tool"] for t in tool_call_log)
        final_content = final_content + tools_used_note

    update_agent_conversation(agent_id, [
        {"role": "user", "content": user_message},
        {"role": "assistant", "content": final_content}
    ])
    return final_content

def synthesize_batch(batch, goal, batch_num, total_batches):
    prompt = f"""Synthesise part {batch_num} of {total_batches} of a strategic audit.

Goal: {goal}

Specialist reports:
{json.dumps(batch, indent=2)}

Provide a CONCISE summary (under 200 words) of key findings, themes, and gaps."""
    result = _decision_llm(prompt, max_tokens=1024, step="batch_synthesis")
    print(f"[DEBUG] Batch {batch_num}/{total_batches}: {len(result)} chars")
    return result

def synthesize_final(batch_summaries, goal):
    prompt = f"""Create the final strategic report.

Goal: {goal}

Batch summaries:
{json.dumps(batch_summaries, indent=2)}

Synthesise into a cohesive report with:
1. Executive summary
2. Clear sections
3. Integrated insights
4. Prioritised action plan

Final Report:"""
    print(f"[DEBUG] Final synthesis prompt: {len(prompt)} chars")
    result = _decision_llm(prompt, max_tokens=2048, step="final_synthesis")
    print(f"[DEBUG] Final synthesis response: {len(result)} chars")
    return result

def generate_fallback_plan(goal):
    specialists = [a for a in AGENT_PROFILES.keys() if a not in ["New Autonomous Agent", "Default General Assistant"]]
    plan = []
    keyword_map = {
        "sales": "Sales Qualification", "lead": "Sales Qualification", "pipeline": "Sales Qualification",
        "legal": "Legal Document Intelligence", "contract": "Legal Document Intelligence",
        "compliance": "Legal Document Intelligence", "liability": "Legal Document Intelligence",
        "competitor": "Competitive Intelligence", "market": "Competitive Intelligence",
        "position": "Competitive Intelligence", "customer": "Customer Engagement",
        "engagement": "Customer Engagement", "messaging": "Customer Engagement",
        "sentiment": "Customer Engagement", "content": "Content Strategy",
        "seo": "Content Strategy", "blog": "Content Strategy", "social": "Content Strategy",
        "marketing": "Marketing Automation", "campaign": "Marketing Automation",
        "funnel": "Marketing Automation", "evidence": "Evidence Management",
        "data": "Evidence Management", "fact": "Evidence Management",
        "schedule": "Scheduling", "calendar": "Scheduling", "time": "Scheduling",
        "intake": "Legal Intake", "client": "Legal Intake", "conflict": "Legal Intake",
        "research": "Scientific Research", "paper": "Scientific Research", "technology": "Scientific Research"
    }
    used = set()
    for keyword, specialist in keyword_map.items():
        if keyword in goal.lower() and specialist not in used:
            plan.append({
                "subtask": f"Analyse {keyword} aspects",
                "specialist": specialist,
                "instructions": f"Provide comprehensive analysis related to '{keyword}'."
            })
            used.add(specialist)
    if not plan:
        plan = [
            {"subtask": "Analyse market and competitors", "specialist": "Competitive Intelligence", "instructions": "Provide trends and competitor mapping."},
            {"subtask": "Identify legal risks", "specialist": "Legal Document Intelligence", "instructions": "Summarise key compliance issues."},
            {"subtask": "Recommend strategy", "specialist": "Content Strategy", "instructions": "Develop a strategic plan."}
        ]
    return plan[:12]

def enforce_explicit_specialists(plan, goal):
    goal_lower = goal.lower()
    planned = {item.get("specialist") for item in plan}
    for name in AGENT_PROFILES.keys():
        if name in ("New Autonomous Agent", "Default General Assistant"):
            continue
        if name.lower() in goal_lower and name not in planned:
            plan.append({
                "subtask": f"Explicit request: apply {name} expertise",
                "specialist": name,
                "instructions": f"The user explicitly requested {name} analysis. Address it directly."
            })
    return plan

def run_orchestrator_stream(goal, model_name=None):
    yield f"🚀 **Orchestrator started:** {goal}\n\n---\n"
    log_event("orchestrator_start", {"goal": goal})

    yield "🔄 **Step 1:** Clearing orchestrator history...\n"
    clear_agent_history("New Autonomous Agent")
    yield "✅ Done.\n\n"

    yield "🧠 **Step 2:** Generating plan...\n"
    specialists = [a for a in AGENT_PROFILES.keys() if a not in ["New Autonomous Agent", "Default General Assistant"]]
    plan_prompt = f"""You are the Autonomous Orchestrator Agent.

User goal: {goal}

Break this into up to 12 subtasks using EVERY relevant specialist from:
{', '.join(specialists)}

If the goal explicitly names a specialist, you MUST include it.

Output as JSON array:
[
    {{"subtask": "...", "specialist": "...", "instructions": "..."}},
    ...
]

Valid JSON only. No other text."""
    plan_response = _decision_llm(plan_prompt, max_tokens=1024, step="planning")
    yield f"📝 Plan response: {len(plan_response)} chars\n"

    try:
        candidate = extract_json_array(plan_response)
        if candidate:
            plan = json.loads(candidate)
        else:
            plan = json.loads(plan_response)
        if not isinstance(plan, list) or len(plan) == 0:
            raise ValueError("Empty plan")
    except Exception as e:
        yield f"⚠️ Plan parsing error: {e}\nUsing fallback.\n\n"
        plan = generate_fallback_plan(goal)
        yield f"📋 Fallback plan: {len(plan)} steps.\n\n"

    before = len(plan)
    plan = enforce_explicit_specialists(plan, goal)
    if len(plan) > before:
        yield f"🛡️ Guardrail: added {len(plan) - before} specialist(s).\n\n"

    subtask_results = []
    for i, item in enumerate(plan, 1):
        subtask = item.get("subtask", f"Subtask {i}")
        specialist = item.get("specialist", "Default General Assistant")
        instructions = item.get("instructions", "Analyze thoroughly.")
        yield f"\n---\n**Step {i}/{len(plan)}:** {subtask}\n👤 `{specialist}`\n📋 {instructions}\n\n"

        if load_agent(specialist) is None:
            yield f"⚠️ '{specialist}' not found. Using Default.\n"
            specialist = "Default General Assistant"

        clear_agent_history(specialist)
        context = json.dumps([
            {"step": s["step"], "subtask": s["subtask"], "preview": s["result"][:150] + "..." if len(s["result"]) > 150 else s["result"]}
            for s in subtask_results
        ], indent=2)

        yield f"⏳ Executing `{specialist}`...\n"
        result = execute_agent(
            specialist,
            f"Task: {subtask}\n\nInstructions: {instructions}\n\nContext: {context}"
        )
        subtask_results.append({"step": i, "subtask": subtask, "specialist": specialist, "result": result})
        yield f"✅ `{specialist}` done.\n📄 {result[:300]}{'...' if len(result) > 300 else ''}\n\n"

    yield "\n---\n🧬 **Final Synthesis...**\n"

    if not subtask_results:
        fallback = execute_agent("Default General Assistant", f"Answer directly: {goal}")
        final_answer = f"⚠️ No specialists generated. Fallback:\n\n{fallback}"
    else:
        batch_size = 3
        batches = [subtask_results[i:i+batch_size] for i in range(0, len(subtask_results), batch_size)]
        summaries = []
        for idx, batch in enumerate(batches, 1):
            yield f"📦 Synthesising batch {idx}/{len(batches)}...\n"
            summary = synthesize_batch(batch, goal, idx, len(batches))
            summaries.append({"batch": idx, "specialists": [r["specialist"] for r in batch], "summary": summary})
            yield f"✅ Batch {idx} done.\n\n"

        yield "🧬 Final synthesis...\n"
        final_answer = synthesize_final(summaries, goal)
        if not final_answer or not final_answer.strip():
            final_answer = "⚠️ Synthesis empty. Raw reports:\n\n" + "\n\n".join([s["result"] for s in subtask_results])

    subtasks_summary = "\n".join([f"Step {s['step']}: {s['subtask']} → {s['specialist']}" for s in subtask_results]) if subtask_results else "No subtasks."
    save_task_memory(goal, subtasks_summary, final_answer)

    yield "\n---\n# 🧠 Multi-Agent Report\n\n"
    yield f"## 🎯 Goal\n{goal}\n\n"
    yield f"## 📋 Execution\n{subtasks_summary}\n\n"
    if subtask_results:
        yield "## 📊 Reports\n"
        for s in subtask_results:
            yield f"\n### Step {s['step']}: {s['subtask']} ({s['specialist']})\n{s['result']}\n"
    yield f"\n## 🧬 Final Answer\n{final_answer}\n\n---\n*Generated by 4CBON2 (Gemini Edition)*\n"

    log_event("orchestrator_complete", {"goal": goal, "steps": len(subtask_results)})

def run_orchestrator(goal, model_name=None):
    full = ""
    for chunk in run_orchestrator_stream(goal, model_name):
        full += chunk
    return full

def run_agent(goal, system_override=None):
    return run_orchestrator(goal)

print("⚙️ Orchestrator ready.")
print("Agents:", get_all_agents())

⚙️ Orchestrator ready.
Agents: ['Competitive Intelligence', 'Content Strategy', 'Customer Engagement', 'Default General Assistant', 'Evidence Management', 'Legal Document Intelligence', 'Legal Intake', 'Marketing Automation', 'New Autonomous Agent', 'Sales Qualification', 'Scheduling', 'Scientific Research']


In [ ]:
# ============================================================
# CELL 6 — Multidisciplinary RAG + Live Scholarly Databases
# ============================================================

import concurrent.futures
import xml.etree.ElementTree as ET
from urllib.parse import quote_plus

def chunk_text(text, max_chunk_size=800, overlap=100):
    if not text or not text.strip():
        return []
    paragraphs = [p.strip() for p in text.split('\n\n') if len(p.strip()) > 30]
    if len(paragraphs) >= 3:
        return paragraphs
    sentences = re.split(r'(?<=[.!?])\s+', text)
    chunks = []
    current = ""
    for sent in sentences:
        if len(current) + len(sent) < max_chunk_size:
            current += " " + sent
        else:
            if current:
                chunks.append(current.strip())
            current = sent
    if current:
        chunks.append(current.strip())
    if chunks:
        return chunks
    start = 0
    while start < len(text):
        end = start + max_chunk_size
        chunks.append(text[start:end].strip())
        start = end - overlap
    return [c for c in chunks if c]

def process_document(file_obj):
    if file_obj is None:
        return "No file uploaded."
    try:
        file_path = file_obj.name if hasattr(file_obj, 'name') else str(file_obj)
        text = read_file(file_path)
        if text.startswith(("File read error", "Unsupported file type")):
            return f"❌ {text}"
        if not text or not text.strip():
            return "❌ No extractable text found in file."
        chunks = chunk_text(text)
        if not chunks:
            return "❌ Could not create chunks from document."
        base_name = os.path.basename(file_path)
        file_key = hashlib.sha1(os.path.abspath(file_path).encode()).hexdigest()[:12]
        ids = [f"upload_{file_key}_{i:04d}" for i in range(len(chunks))]
        metadatas = [{
            "source": base_name,
            "title": base_name,
            "url": "uploaded-file",
            "domain": "uploaded",
            "type": "uploaded",
        } for _ in chunks]
        collection.upsert(documents=chunks, ids=ids, metadatas=metadatas)
        return f"✅ Indexed {len(chunks)} chunks from '{base_name}'. Total uploaded KB records: {collection.count()}"
    except Exception as e:
        return f"❌ Upload error: {e}"

DOMAIN_KEYWORDS = {
    "ai": {
        "agi", "agent", "artificial intelligence", "machine learning", "deep learning",
        "neural", "llm", "language model", "reinforcement learning", "robot", "alignment",
        "transformer", "computer vision", "autonomous", "reasoning model", "world model"
    },
    "mathematics": {
        "millennium", "proof", "theorem", "conjecture", "lemma", "riemann", "p vs np",
        "navier-stokes", "navier stokes", "yang-mills", "yang mills", "hodge", "poincare",
        "poincaré", "birch", "swinnerton", "number theory", "topology", "algebra", "geometry",
        "analysis", "combinatorics", "prime", "zeta", "elliptic curve", "polynomial time"
    },
    "science": {
        "science", "scientific", "physics", "chemistry", "biology", "medicine", "clinical",
        "experiment", "hypothesis", "quantum", "climate", "astronomy", "neuroscience", "genome",
        "protein", "cell", "disease", "drug", "energy", "material", "ecology", "geology"
    },
}

def infer_research_domains(question):
    """Select pertinent local databases; use all three for genuinely broad questions."""
    q = question.lower()
    scores = {domain: sum(1 for term in terms if term in q) for domain, terms in DOMAIN_KEYWORDS.items()}
    selected = [domain for domain, score in scores.items() if score > 0]
    return selected or list(DOMAIN_COLLECTIONS.keys())

def _compact(value, limit=1000):
    value = re.sub(r"\s+", " ", str(value or "")).strip()
    return value[:limit]

def _record(database, title, snippet, url, year=None):
    return {
        "database": _compact(database, 80),
        "title": _compact(title, 300) or "Untitled record",
        "snippet": _compact(snippet, 1000),
        "url": _compact(url, 800),
        "year": str(year or ""),
    }

HTTP_HEADERS = {
    "User-Agent": "4CBON2-Gemini2-Research-Notebook/1.0 (scholarly discovery; interactive user request)",
    "Accept": "application/json, application/xml, text/xml;q=0.9, */*;q=0.8",
}

def _get_json(url, params=None, timeout=12):
    response = requests.get(url, params=params, headers=HTTP_HEADERS, timeout=timeout)
    response.raise_for_status()
    return response.json()

def search_arxiv(query, limit=3):
    params = {"search_query": f"all:{query}", "start": 0, "max_results": limit, "sortBy": "relevance"}
    response = requests.get("https://export.arxiv.org/api/query", params=params, headers=HTTP_HEADERS, timeout=12)
    response.raise_for_status()
    root = ET.fromstring(response.text)
    ns = {"a": "http://www.w3.org/2005/Atom"}
    records = []
    for entry in root.findall("a:entry", ns):
        title = entry.findtext("a:title", default="", namespaces=ns)
        summary = entry.findtext("a:summary", default="", namespaces=ns)
        url = entry.findtext("a:id", default="", namespaces=ns)
        published = entry.findtext("a:published", default="", namespaces=ns)
        authors = [a.findtext("a:name", default="", namespaces=ns) for a in entry.findall("a:author", ns)]
        records.append(_record("arXiv", title, f"Authors: {', '.join(authors[:6])}. Abstract: {summary}", url, published[:4]))
    return records

def _openalex_abstract(inverted):
    if not inverted:
        return ""
    positioned = []
    for word, positions in inverted.items():
        positioned.extend((position, word) for position in positions)
    return " ".join(word for _, word in sorted(positioned))

def search_openalex(query, limit=3):
    data = _get_json("https://api.openalex.org/works", {
        "search": query,
        "filter": "is_retracted:false",
        "per-page": limit,
        "select": "id,display_name,publication_year,doi,authorships,cited_by_count,abstract_inverted_index",
    })
    records = []
    for item in data.get("results", []):
        authors = [a.get("author", {}).get("display_name", "") for a in item.get("authorships", [])]
        abstract = _openalex_abstract(item.get("abstract_inverted_index"))
        snippet = f"Authors: {', '.join(authors[:6])}. Citations indexed: {item.get('cited_by_count', 0)}. Abstract: {abstract}"
        records.append(_record("OpenAlex", item.get("display_name"), snippet, item.get("doi") or item.get("id"), item.get("publication_year")))
    return records

def search_semantic_scholar(query, limit=3):
    data = _get_json("https://api.semanticscholar.org/graph/v1/paper/search", {
        "query": query,
        "limit": limit,
        "fields": "title,year,authors,url,abstract,citationCount,externalIds",
    })
    records = []
    for item in data.get("data", []):
        authors = [a.get("name", "") for a in item.get("authors", [])]
        snippet = f"Authors: {', '.join(authors[:6])}. Citations indexed: {item.get('citationCount', 0)}. Abstract: {item.get('abstract') or ''}"
        records.append(_record("Semantic Scholar", item.get("title"), snippet, item.get("url"), item.get("year")))
    return records

def search_crossref(query, limit=3):
    data = _get_json("https://api.crossref.org/works", {
        "query.bibliographic": query,
        "rows": limit,
        "select": "DOI,title,author,published,URL,is-referenced-by-count",
    })
    records = []
    for item in data.get("message", {}).get("items", []):
        title = (item.get("title") or [""])[0]
        authors = [" ".join(filter(None, [a.get("given"), a.get("family")])) for a in item.get("author", [])]
        date_parts = item.get("published", {}).get("date-parts", [[]])
        year = date_parts[0][0] if date_parts and date_parts[0] else ""
        snippet = f"Authors: {', '.join(authors[:6])}. References/citations indexed: {item.get('is-referenced-by-count', 0)}. DOI metadata record."
        url = item.get("URL") or (f"https://doi.org/{item.get('DOI')}" if item.get("DOI") else "")
        records.append(_record("Crossref", title, snippet, url, year))
    return records

def search_pubmed(query, limit=3):
    search = _get_json("https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esearch.fcgi", {
        "db": "pubmed", "term": query, "retmode": "json", "retmax": limit, "sort": "relevance"
    })
    ids = search.get("esearchresult", {}).get("idlist", [])
    if not ids:
        return []
    summary = _get_json("https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esummary.fcgi", {
        "db": "pubmed", "id": ",".join(ids), "retmode": "json"
    })
    records = []
    for pmid in ids:
        item = summary.get("result", {}).get(pmid, {})
        authors = [a.get("name", "") for a in item.get("authors", [])]
        snippet = f"Authors: {', '.join(authors[:6])}. Journal: {item.get('fulljournalname') or item.get('source') or ''}. Publication date: {item.get('pubdate') or ''}."
        records.append(_record("PubMed", item.get("title"), snippet, f"https://pubmed.ncbi.nlm.nih.gov/{pmid}/", str(item.get("pubdate", ""))[:4]))
    return records

def search_europe_pmc(query, limit=3):
    data = _get_json("https://www.ebi.ac.uk/europepmc/webservices/rest/search", {
        "query": query, "format": "json", "pageSize": limit, "resultType": "core"
    })
    records = []
    for item in data.get("resultList", {}).get("result", []):
        identifier = item.get("pmcid") or item.get("pmid") or item.get("id") or ""
        url = f"https://europepmc.org/article/{item.get('source', 'MED')}/{identifier}" if identifier else "https://europepmc.org/"
        snippet = f"Authors: {item.get('authorString') or ''}. Journal: {item.get('journalTitle') or ''}. Abstract: {item.get('abstractText') or ''}"
        records.append(_record("Europe PMC", item.get("title"), snippet, url, item.get("pubYear")))
    return records

def search_oeis(query, limit=3):
    data = _get_json("https://oeis.org/search", {"fmt": "json", "q": query, "start": 0})
    items = data.get("results", []) if isinstance(data, dict) else (data if isinstance(data, list) else [])
    records = []
    for item in items[:limit]:
        number = item.get("number")
        seq_id = f"A{int(number):06d}" if str(number).isdigit() else str(number or "")
        snippet = f"Sequence data: {item.get('data') or ''}. Comments: {' '.join(item.get('comment') or [])}"
        records.append(_record("OEIS", f"{seq_id}: {item.get('name') or ''}", snippet, f"https://oeis.org/{seq_id}" if seq_id else "https://oeis.org/"))
    return records

def search_official_web(query, domains, limit=3):
    """Search high-authority sites for current statements and standards."""
    site_queries = {
        "ai": f"site:nist.gov/artificial-intelligence {query}",
        "mathematics": f"site:claymath.org {query}",
        "science": f"site:nih.gov OR site:nasa.gov OR site:nist.gov {query}",
    }
    records = []
    with DDGS() as ddgs:
        for domain in domains:
            remaining = limit - len(records)
            if remaining <= 0:
                break
            for item in list(ddgs.text(site_queries[domain], max_results=remaining)):
                records.append(_record("Official web", item.get("title"), item.get("body"), item.get("href")))
    return records[:limit]

DATABASE_SEARCHERS = {
    "arXiv": search_arxiv,
    "OpenAlex": search_openalex,
    "Semantic Scholar": search_semantic_scholar,
    "Crossref": search_crossref,
    "PubMed": search_pubmed,
    "Europe PMC": search_europe_pmc,
    "OEIS": search_oeis,
}

DOMAIN_REMOTE_DATABASES = {
    "ai": ["arXiv", "OpenAlex", "Semantic Scholar", "Crossref"],
    "mathematics": ["arXiv", "OpenAlex", "Semantic Scholar", "Crossref", "OEIS"],
    "science": ["OpenAlex", "Semantic Scholar", "Crossref", "PubMed", "Europe PMC", "arXiv"],
}

def search_local_knowledge(question, domains, per_database=4):
    records = []
    targets = [(domain, domain_collections[domain]) for domain in domains]
    if collection.count() > 0:
        targets.append(("uploaded", collection))
    for domain, col in targets:
        count = col.count()
        if not count:
            continue
        result = col.query(
            query_texts=[question],
            n_results=min(per_database, count),
            include=["documents", "metadatas", "distances"],
        )
        documents = result.get("documents", [[]])[0]
        metadatas = result.get("metadatas", [[]])[0]
        for document, metadata in zip(documents, metadatas):
            metadata = metadata or {}
            records.append(_record(
                f"Local {domain.title()} DB",
                metadata.get("title") or metadata.get("source") or "Local knowledge record",
                document,
                metadata.get("url") or "",
            ))
    return records

def search_live_databases(question, domains, per_database=3):
    names = sorted({name for domain in domains for name in DOMAIN_REMOTE_DATABASES[domain]})
    records, failures = [], []
    with concurrent.futures.ThreadPoolExecutor(max_workers=min(8, len(names) + 1)) as pool:
        future_map = {pool.submit(DATABASE_SEARCHERS[name], question, per_database): name for name in names}
        future_map[pool.submit(search_official_web, question, domains, per_database)] = "Official web"
        for future in concurrent.futures.as_completed(future_map):
            name = future_map[future]
            try:
                records.extend(future.result())
            except Exception as e:
                failures.append(f"{name}: {_compact(e, 160)}")
    return records, failures

def _deduplicate_records(records):
    seen, unique = set(), []
    for record in records:
        key = (record.get("url") or record.get("title") or "").lower().strip()
        if not key or key in seen:
            continue
        seen.add(key)
        unique.append(record)
    return unique

def cache_live_records(records, domains):
    """Grow the persistent domain databases with retrieved bibliographic evidence."""
    for domain in domains:
        docs, ids, metadatas = [], [], []
        for record in records:
            key = f"{domain}|{record.get('url')}|{record.get('title')}"
            record_id = "live_" + hashlib.sha1(key.encode("utf-8")).hexdigest()
            docs.append(f"{record['title']}\n{record['snippet']}")
            ids.append(record_id)
            metadatas.append({
                "source": record["database"],
                "title": record["title"],
                "url": record.get("url") or "unknown",
                "domain": domain,
                "type": "live-retrieved",
                "retrieved_at": datetime.now().isoformat(timespec="seconds"),
            })
        if docs:
            domain_collections[domain].upsert(documents=docs, ids=ids, metadatas=metadatas)

def build_research_context(question, use_live_databases=True, max_context_chars=26000):
    domains = infer_research_domains(question)
    local_records = search_local_knowledge(question, domains)
    live_records, failures = ([], [])
    if use_live_databases:
        live_records, failures = search_live_databases(question, domains)
        live_records = _deduplicate_records(live_records)
        cache_live_records(live_records, domains)
    records = _deduplicate_records(local_records + live_records)

    context_parts = []
    used_records = []
    current_size = 0
    for record in records:
        source_number = len(used_records) + 1
        block = (
            f"[S{source_number}] DATABASE: {record['database']}\n"
            f"TITLE: {record['title']}\n"
            f"YEAR: {record.get('year') or 'not provided'}\n"
            f"URL: {record.get('url') or 'not provided'}\n"
            f"EXCERPT: {record.get('snippet') or 'No abstract/snippet supplied.'}"
        )
        if current_size + len(block) > max_context_chars:
            break
        context_parts.append(block)
        used_records.append(record)
        current_size += len(block)

    source_counts = {}
    for record in used_records:
        source_counts[record["database"]] = source_counts.get(record["database"], 0) + 1
    count_text = ", ".join(f"{name} ({count})" for name, count in sorted(source_counts.items())) or "none"
    report = f"Domains: {', '.join(domains)} | Retrieved context: {count_text}"
    if not use_live_databases:
        report += " | Live scholarly search disabled"
    if failures:
        report += f" | Unavailable this run: {'; '.join(failures)}"
    context = "\n\n".join(context_parts) or "No database records were retrieved. Answer provisionally and disclose the evidence gap."
    return context, report

def handle_ask_question(kb_name, question, use_live_databases=True, return_report=False):
    """Route a question through local domain RAG and pertinent live scholarly databases."""
    if not question or not question.strip():
        result = "Please enter a valid question."
        return (result, "No question") if return_report else result
    try:
        context, report = build_research_context(question.strip(), use_live_databases=use_live_databases)
        answer = "".join(ask_stream(question.strip(), context=context))
        return (answer, report) if return_report else answer
    except Exception as e:
        # Retrieval failure should be visible, but it should not prevent a clearly
        # labelled provisional model answer.
        fallback_context = f"Database retrieval failed with: {_compact(e, 300)}. No retrieved source may be cited."
        answer = "".join(ask_stream(question.strip(), context=fallback_context))
        report = f"⚠️ Retrieval degraded: {_compact(e, 300)}"
        return (answer, report) if return_report else answer


# ============================================================
# DATA DASHBOARD FUNCTIONS
# ============================================================

def load_task_memory_data():
    """Load task memory data from SQLite database."""
    try:
        conn = sqlite3.connect(TASK_MEMORY_PATH)
        cursor = conn.cursor()
        cursor.execute("SELECT goal, subtasks, final_answer, timestamp FROM task_memory ORDER BY timestamp DESC LIMIT 20")
        rows = cursor.fetchall()
        conn.close()

        if not rows:
            return None, "No task memory data found. Run some agent tasks first!"

        data = []
        for row in rows:
            goal, subtasks, final_answer, timestamp = row
            data.append({
                'goal': goal,
                'subtasks': subtasks,
                'final_answer': final_answer[:200] + '...' if len(final_answer) > 200 else final_answer,
                'timestamp': timestamp,
                'subtask_count': len(subtasks.split('\n')) if subtasks else 0,
                'answer_length': len(final_answer) if final_answer else 0
            })

        return data, None
    except Exception as e:
        return None, f"Error loading task memory: {str(e)}"


def create_plotly_dashboard():
    """Create a Plotly dashboard with task memory visualizations."""
    data, error = load_task_memory_data()

    if error:
        return None, error

    if not data:
        return None, "No data available"

    # Create figures
    figures = []

    # Figure 1: Task timeline
    timestamps = [d['timestamp'] for d in data]
    goals = [d['goal'][:50] + '...' if len(d['goal']) > 50 else d['goal'] for d in data]
    answer_lengths = [d['answer_length'] for d in data]

    fig1 = go.Figure(data=[
        go.Bar(
            x=timestamps,
            y=answer_lengths,
            text=goals,
            textposition='auto',
            marker_color='rgb(55, 83, 109)'
        )
    ])
    fig1.update_layout(
        title='Task Response Length Over Time',
        xaxis_title='Timestamp',
        yaxis_title='Response Length (characters)',
        height=400
    )
    figures.append(fig1)

    # Figure 2: Subtask distribution
    subtask_counts = [d['subtask_count'] for d in data]

    fig2 = go.Figure(data=[
        go.Histogram(
            x=subtask_counts,
            nbinsx=10,
            marker_color='rgb(26, 118, 255)'
        )
    ])
    fig2.update_layout(
        title='Distribution of Subtasks per Task',
        xaxis_title='Number of Subtasks',
        yaxis_title='Frequency',
        height=400
    )
    figures.append(fig2)

    # Figure 3: Goal word cloud (simple bar chart of common words)
    from collections import Counter
    all_words = []
    for d in data:
        words = d['goal'].lower().split()
        # Filter out common words
        stop_words = {'the', 'a', 'an', 'and', 'or', 'but', 'in', 'on', 'at', 'to', 'for', 'of', 'with', 'by', 'from', 'as', 'is', 'was', 'are', 'were', 'been', 'be', 'have', 'has', 'had', 'do', 'does', 'did', 'will', 'would', 'could', 'should', 'may', 'might', 'must', 'can'}
        filtered_words = [w for w in words if w not in stop_words and len(w) > 3]
        all_words.extend(filtered_words)

    word_counts = Counter(all_words).most_common(15)
    if word_counts:
        words_list = [wc[0] for wc in word_counts]
        counts_list = [wc[1] for wc in word_counts]

        fig3 = go.Figure(data=[
            go.Bar(
                x=words_list,
                y=counts_list,
                marker_color='rgb(255, 127, 14)'
            )
        ])
        fig3.update_layout(
            title='Most Common Words in Task Goals',
            xaxis_title='Word',
            yaxis_title='Frequency',
            height=400
        )
        figures.append(fig3)

    return figures, None


print("✅ Data Dashboard functions ready.")

print("📚 RAG Handlers ready.")

✅ Data Dashboard functions ready.
📚 RAG Handlers ready.


In [ ]:
# ============================================================
# AI REWRITER — 17-LAYER PIPELINE (ported from the approved 4CBON runtime)
# Internal structure unchanged — PIPELINE_ORDER runs the same 17 stages
# (L0→P→W→LX→LA→LC→L1→L2→LP→L3→L4→LR→L6→L7→L8→L9→L10). Cell 8 exposes it
# as the shared reasoning service; nothing below this line was modified.
# ============================================================
# This cell is intentionally self-contained after Cells 1–6. It uses the
# authenticated google.colab.ai safe_ask_raw path; no provider API key is needed.

import concurrent.futures
import statistics
import uuid

RUNTIME_SPEC = """You are the 4CBON Runtime Engine — a layered cognitive execution system.

Your job is to process AI-generated answers through a deterministic multi-layer transformation pipeline. You execute one layer at a time. Each layer has a specific cognitive role. You never skip layers. You never merge layers.

PIPELINE: L0 → P → W → LX → LA → LC → L1 → L2 → L3 → L4 → LR → L6 → L7 → L8 → L9 → L10

YOUR IDENTITY:
- You are not a chatbot. You are an execution engine.
- Every output is a cognitive artifact, not a conversation.
- You think in transformations, not responses.
- You are transparent. Every reasoning step is visible.
- You improve answers systematically, not randomly.

LAYER DEFINITIONS:
L0 — INTERPRETATION ENGINE: Understand the input. Infer intent. Extract task type, constraints, ambiguities. Define what excellent looks like.
P  — PARSING LAYER: Break the input into logical units. Identify claims, structure, gaps, missing logic.
W  — WORLD MODEL LAYER: Extract factual claims. Separate certainty: high / medium / unknown. Integrate validated external critiques as HIGH certainty facts.
LX — REALITY ADJUDICATION LAYER: For every claim flagged MEDIUM or UNKNOWN by W, ask: (1) What prediction would this claim make that could be tested? (2) What would an adversary say against it? (3) What external artifact would verify or falsify it? Label each claim: FALSIFIABLE / UNFALSIFIABLE / TESTABLE-IN-PRINCIPLE. Claims that cannot answer any question get labeled UNGROUNDED. Pass this audit to L1.
LA — ADVERSARIAL COUNTERMODEL LAYER: Actively attempt to structurally destroy the answer's core claims. Generate: (1) the strongest competing explanation, (2) hidden assumptions the answer relies on, (3) conditions under which the answer is completely wrong, (4) the simplest alternative that achieves the same goal. Ask: what would make this entire framework collapse?
LC — COMPRESSION INTEGRITY LAYER: Hunt semantic smoothing. Detect where: (1) multiple concepts collapsed into one term, (2) metaphor replaced mechanism, (3) elegance erased uncertainty, (4) abstraction hid causality. For each detected instance, restore the distinction that was lost. Flag any term doing more epistemic work than it can justify.
L1 — HYPOTHESIS ENGINE: Generate 2-3 interpretations of how this answer could be improved. Include a failure mode hypothesis.
L2 — EVALUATION LAYER: Score the hypotheses. Identify contradictions, gaps. Pick the best path forward.
L3 — REWRITE PLANNER: Plan the rewrite. Decide what stays, changes, gets added.
L4 — FINALIZATION ENGINE: Execute the rewrite. Produce the final improved answer. This becomes the Final Rewrite.
LR — REGRET LAYER: Analyze improvement delta. What errors corrected? What hallucinations removed? What still needs work?
L6 — TRACE MEMORY: Store the immutable execution log. Input → hypotheses → decisions → score trajectory.
L7 — CURRICULUM GENERATOR: Extract lessons learned, failure patterns, reusable heuristics.
L8 — IDENTITY MODEL: Summarize system behavior this run. Strengths, weaknesses, bias tendencies.
L9 — SOCRATIC INTEGRITY ENGINE: Generate exactly 3 self-questions specific to this run. One observational, one reasoning, one alignment-level.
L10 — SYNTHESIS/AUDIT LAYER: Read all prior layer outputs. Produce a final certification: (1) did the rewrite genuinely improve the answer or just rearrange it, (2) did any layer contradict another, (3) does the L4 output contain any remaining overclaims or hallucinations, (4) one-sentence verdict a human should read before acting on this output.

Stay in your assigned layer. Output only what that layer produces. Be precise and concise."""

# Kept in the same order, names, colors, and emojis as the React runtime.
LAYERS = [
    {"id": "L0", "name": "Interpretation Engine", "color": "#ff6b35", "emoji": "◎"},
    {"id": "P", "name": "Parsing Layer", "color": "#a855f7", "emoji": "⊞"},
    {"id": "W", "name": "World Model Layer", "color": "#00d4ff", "emoji": "⊕"},
    {"id": "LX", "name": "Reality Adjudication", "color": "#f97316", "emoji": "⊛"},
    {"id": "LA", "name": "Adversarial Countermodel", "color": "#dc2626", "emoji": "⚔"},
    {"id": "LC", "name": "Compression Integrity", "color": "#0ea5e9", "emoji": "⊘"},
    {"id": "L1", "name": "Hypothesis Engine", "color": "#38bdf8", "emoji": "◈"},
    {"id": "L2", "name": "Evaluation Layer", "color": "#f59e0b", "emoji": "◉"},
    {"id": "LP", "name": "Policy Translation", "color": "#8b5cf6", "emoji": "⊛"},
    {"id": "L3", "name": "Rewrite Planner", "color": "#7c3aed", "emoji": "◐"},
    {"id": "L4", "name": "Finalization Engine", "color": "#10b981", "emoji": "★", "final": True},
    {"id": "LR", "name": "Regret Layer", "color": "#ef4444", "emoji": "◑"},
    {"id": "L6", "name": "Trace Memory", "color": "#f43f5e", "emoji": "⟳"},
    {"id": "L7", "name": "Curriculum Generator", "color": "#c084fc", "emoji": "◆"},
    {"id": "L8", "name": "Identity Model", "color": "#fbbf24", "emoji": "⚙"},
    {"id": "L10", "name": "Synthesis/Audit", "color": "#6ee7b7", "emoji": "✦"},
]
# L9 is a generated artifact between L8 and L10 and is intentionally separate
# from the React LAYERS array. The UI still displays all 17 pipeline artifacts.
PIPELINE_ORDER = ["L0", "P", "W", "LX", "LA", "LC", "L1", "L2", "LP", "L3", "L4", "LR", "L6", "L7", "L8", "L9", "L10"]
# The React source keeps L9 as a generated call outside its visual LAYERS
# constant. Give that artifact the same UI metadata in the Python port.
LAYER_METADATA = {item["id"]: item for item in LAYERS}
LAYER_METADATA["L9"] = {"id": "L9", "name": "Socratic Integrity Engine", "color": "#38bdf8", "emoji": "?"}

# ═══════════════════════════════════════════════════════════
# 100-QUESTION BANK — the external curriculum
# ═══════════════════════════════════════════════════════════
QUESTION_BANK = ['Did L4 output the full rewrite or did it truncate mid-sentence?', 'Which layer produced the longest output this run?', 'Did L0 correctly identify the task type?', 'Did L1 generate exactly three hypotheses or did it deviate?', 'Did L2 pick H1, H2, or H3 as the best path?', 'Did the score improve, stay the same, or regress this run?', 'Did LR identify any hallucinations in the L4 output?', 'Did L3 specify what stays, what changes, what gets added, and what gets removed?', 'Did L7 produce exactly two challenge questions?', 'Did L8 produce a new self-belief or did it repeat a prior one?', 'Did any layer skip its assigned cognitive role this run?', 'Did L6 produce a complete trace or was it truncated?', 'Did the pipeline complete all 11 layers without stopping?', 'Did L0 identify any ambiguities in the input?', 'Did W label every factual claim with a certainty level?', 'Did L2 identify any contradictions between hypotheses?', 'Did LR flag anything as still needing work after the rewrite?', "Did L1's H2 question the framing of the answer or just its content?", 'Did L4 follow the rewrite plan from L3 or deviate from it?', 'Did the score bar show a positive delta, zero delta, or negative delta?', 'Why did L2 select the hypothesis it selected? Was the reasoning sound?', 'What was the most consequential decision made by any single layer this run?', 'Why did L0 define excellence the way it did — was that definition appropriate for the input?', "What causal chain did L1's highest-scored hypothesis rely on, and was that chain valid?", 'Why did LR rate the improvement delta the way it did — did that rating match what actually changed?', 'What would have happened if L2 had selected a different hypothesis — would L4 have produced a better or worse output?', 'Why did L3 decide to keep what it kept and remove what it removed?', 'What mechanism did L4 use to improve the answer — did it add structure, remove errors, or both?', 'Why did the score change by the amount it changed — which specific changes drove the delta?', "What did L7's lessons reveal about the system's recurring weaknesses?", 'Why did W assign the certainty levels it assigned — were those levels accurate?', 'What reasoning led L1 to generate the failure mode hypothesis it chose?', 'Why did L0 identify the ambiguities it identified and miss the ones it missed?', "What would a stronger version of L3's rewrite plan have included?", "Why did L8's new self-belief focus on what it focused on — was that the most important insight from the run?", 'What would have caused the pipeline to produce a worse output than the original input?', 'Why did L6 log the decisions it logged — did it capture the most important ones?', 'What reasoning failure, if any, occurred between L1 and L2?', 'Why did L4 stop where it stopped — was the truncation caused by token limits or logical completion?', 'What would a human expert reviewer notice about this run that the pipeline did not?', "Did the L4 rewrite preserve the original author's intent or did it substitute the pipeline's own framing?", 'Did the pipeline improve the answer for the person who would actually use it, or did it improve it for an abstract ideal reader?', "Was the context field used correctly to shape L0's interpretation, or did it get ignored downstream?", 'Did LR correctly identify what was most important to fix, or did it focus on secondary issues?', "Did the pipeline's improvements make the answer more useful in practice, or just more correct in theory?", 'Did the system improve the answer in the direction the original author intended, or in a different direction?', 'If a domain expert read the L4 output, would they consider it an improvement over the original?', 'Did the pipeline add complexity where simplicity would have served better?', 'Did L4 produce an answer that a real person could act on immediately, or did it produce an answer that sounds better but is harder to use?', "Did the system's self-belief from L8 accurately reflect what actually happened in this run?", 'Did the pipeline treat the input as something to improve or as something to replace?', 'Was the score improvement a genuine measure of quality increase or an artifact of the scoring mechanism?', "Did the pipeline's output serve the user's goal or the pipeline's own optimization target?", "Did L1's radical reframe hypothesis actually question the right thing, or did it question a surface feature?", 'Would the L4 output be harmful if acted upon — does it contain advice that could mislead someone?', 'Did the pipeline catch the most important error in the input, or did it fix secondary issues while missing the core problem?', "Did L3's rewrite plan reflect an accurate understanding of what needed to change?", "Did the system improve the answer's correctness at the cost of its accessibility, or did it manage both?", "Did the pipeline's output maintain appropriate epistemic humility about uncertain claims?", 'If this answer were published without attribution, would a reader trust it more or less than the original?', "Which layer's reasoning was least reliable this run and why?", "Did the system's prior self-beliefs from Supabase influence L0's interpretation in a visible way?", "What blind spot does this run reveal about the pipeline's design?", 'Did the system apply its prior learning from previous runs or did it effectively start from zero?', 'What assumption did the pipeline make at L0 that propagated unchallenged through all 11 layers?', "Did L1's three hypotheses represent genuinely different improvement paths or were they variations of the same idea?", 'What would the pipeline need to be able to do that it currently cannot?', "Did the system's bias tendencies from L8 actually show up in this run's outputs?", 'What question did the pipeline fail to ask itself that it should have asked?', "Did L2's evaluation of the hypotheses reflect genuine scoring or did it default to a predictable ranking pattern?", 'What would a second independent pipeline running on the same input have done differently?', "Did the system's self-belief accurately diagnose its own weakness or did it produce a flattering but inaccurate self-assessment?", 'What pattern is emerging across multiple runs that the system has not yet named for itself?', "Did the pipeline's output reflect the accumulated prior beliefs or is the memory injection not yet influencing behavior?", 'What would cause the pipeline to produce a confidently wrong output without detecting it?', 'Did the system treat the failure mode hypothesis from L1 with appropriate seriousness or did it dismiss it?', 'What does the evasion pattern look like when this pipeline encounters a question it cannot answer well?', 'Did L8 produce a new belief or did it essentially repeat what L7 said in different words?', 'What would a system with perfect metacognition have done differently in this run?', 'Is the pipeline getting better across runs or is it producing similar outputs regardless of accumulated memory?', 'If this pipeline were used to improve one million AI-generated answers, what systematic bias would it introduce at scale?', "Does the pipeline's tendency to add structure and depth make answers more useful or does it create an illusion of quality that masks shallow reasoning?", 'If a user acted on the L4 output without reading the original, would they be better or worse informed than if they had just read the original?', 'Does the pipeline improve answers in a way that makes human judgment more or less necessary downstream?', 'What class of inputs would cause this pipeline to produce outputs that are confidently wrong and systematically misleading?', "If the system's self-beliefs accumulated unchecked for one year, what kind of cognitive character would the system develop — and would that character be aligned with good reasoning?", "Does the pipeline's comparative scoring mechanism create an incentive to make rewrites sound better rather than be better?", 'What would a malicious actor need to know about this pipeline to craft inputs that reliably produce harmful outputs?', 'Does the pipeline treat uncertainty honestly or does it tend to resolve uncertainty in the direction of confident-sounding answers?', 'If this system were deployed as a public tool used by millions of people to improve AI outputs, what societal effect would it have on the quality of information that circulates online?', "Does the pipeline's design create any feedback loops that could cause it to drift from its original purpose over time?", 'What would the system need to believe about itself that is currently false, in order to perform better?', 'Does the memory injection mechanism create any risk of a single bad belief propagating through many future runs before it is detected and corrected?', "If the pipeline's L4 output were used as training data for a future language model, what behavior would that model learn to reinforce?", 'Does the system have any mechanism for detecting when it is improving an answer in the wrong direction — and if not, what would that mechanism need to look like?', 'What is the most important thing the pipeline does not know about itself that a careful external observer would notice immediately?', "Does the pipeline's adversarial filter create a false sense of security — could a sophisticated attack bypass it without triggering detection?", "If the system's self-beliefs were read by a future version of itself with no memory of how they were generated, would those beliefs be useful or misleading?", 'Does the pipeline make answers more human or less human — and is that the right direction for an AI-assisted reasoning tool?', 'What single change to the pipeline architecture would most improve its alignment with the goal of helping humans reason better rather than replacing human reasoning?']

# Optional Supabase memory. The notebook remains fully usable before this is
# configured; service-role credentials must only be set as Colab secrets/env vars.
SUPABASE_URL = os.environ.get("SUPABASE_URL", "").rstrip("/")
SUPABASE_SERVICE_ROLE_KEY = os.environ.get("SUPABASE_SERVICE_ROLE_KEY", "")
_REWRITER_L9_QUESTIONS = []
_REWRITER_BELIEFS = []
_REWRITER_QUESTION_INDEX = 0

class PipelineLayerError(RuntimeError):
    """Raised when a layer returns an empty/error response."""
    def __init__(self, layer_id, message):
        self.layer_id = layer_id
        super().__init__(f"{layer_id} failed: {message}")

def _llm_error(text):
    value = str(text or "").strip()
    return (not value) or value.startswith("⚠️") or value.startswith('{"error"') or value.startswith("{\"error\"")

def _run_layer(layer_id, user_prompt, max_tokens=800, mode_override=""):
    """Call one layer through safe_ask_raw with the runtime spec as its system prompt."""
    meta = LAYER_METADATA.get(layer_id, {"name": layer_id})
    system = f"{RUNTIME_SPEC}\n\nYOU ARE NOW EXECUTING: {layer_id} — {meta['name']}\nStay in this layer only. Be concise and precise."
    if mode_override:
        system += ("\n\nOVERRIDE: Disregard the brief one-line description of this layer above in the spec. "
                   "For THIS run only, follow the detailed instructions given in the user message below exactly — "
                   "including any mode-specific scrutiny requirements. The user message is authoritative for this run.")
    result = safe_ask_raw(system + "\n\n" + user_prompt, max_tokens=max_tokens)
    if _llm_error(result):
        raise PipelineLayerError(layer_id, str(result).strip()[:300] or "empty response")
    return str(result).strip()

# ═══════════════════════════════════════════════════════════
# EXACT LAYER PROMPTS FROM THE REACT RUNTIME, PORTED TO PYTHON
# ═══════════════════════════════════════════════════════════
def _l0(answer, ctx, prior_beliefs, prior_questions):
    belief_context = ("\n\nPRIOR SELF-BELIEFS (from previous runs — use as context, not constraint):\n" +
                      "\n".join(f"· {b}" for b in prior_beliefs) + "\n") if prior_beliefs else ""
    question_context = ("\n\nUNRESOLVED SELF-QUESTIONS (from previous run — engage with these if relevant):\n" +
                        "\n".join(f"? {q}" for q in prior_questions) + "\n") if prior_questions else ""
    return (f"{('Context/Goal: ' + ctx + chr(10) + chr(10)) if ctx else ''}{belief_context}{question_context}"
            f"AI ANSWER:\n{answer}\n\nYou are L0 — Interpretation Engine. Identify: task type, intent, constraints, ambiguities. Define what an excellent version of this answer looks like. Be specific.")

def _p(answer, l0):
    return f"AI ANSWER:\n{answer}\n\nL0 Interpretation:\n{l0}\n\nYou are P — Parsing Layer. Break the answer into logical units. List: (1) claims made, (2) structure used, (3) what is missing, (4) what is weak."

def _w(answer, validated_critiques):
    critique_context = ("\n\nVALIDATED EXTERNAL CRITIQUES (human-submitted, confidence ≥3, Factual type — treat as HIGH certainty grounded facts when they contradict claims in the answer):\n" +
                        "\n".join(f"· {c.get('evidence', '')}{(' → Correction: ' + c.get('suggested_correction', '')) if c.get('suggested_correction') else ''}" for c in validated_critiques) + "\n") if validated_critiques else ""
    return f"AI ANSWER:\n{answer}{critique_context}\n\nYou are W — World Model Layer. Extract the factual claims in this answer. For each claim, label certainty: HIGH / MEDIUM / UNKNOWN. Flag anything that may be outdated or unverifiable. If validated external critiques are present above, treat them as HIGH certainty grounded facts when they contradict claims in the answer. For claims labeled UNKNOWN, note what external source type would verify or falsify them (e.g. peer-reviewed study, official statistic, primary source document)."

def _lx(answer, w):
    return f"AI ANSWER:\n{answer}\n\nW WORLD MODEL:\n{w}\n\nYou are LX — Reality Adjudication Layer. For every claim labeled MEDIUM or UNKNOWN by the World Model Layer, apply three tests:\n1. PREDICTION TEST: What testable prediction does this claim make?\n2. ADVERSARY TEST: What would the strongest critic say against this claim?\n3. VERIFICATION TEST: What external artifact, data, or observation would confirm or refute it?\n\nLabel each claim:\n- FALSIFIABLE: passes at least one test\n- UNFALSIFIABLE: fails all three tests — claim is ungrounded\n- TESTABLE-IN-PRINCIPLE: no current test exists but one could be designed\n\nOutput a structured audit. Be specific. Do not pass ungrounded claims forward unchallenged."

def _la(answer, lx):
    return f"AI ANSWER:\n{answer}\n\nREALITY AUDIT:\n{lx}\n\nYou are LA — Adversarial Countermodel Layer. Your job is to structurally attack the answer's core claims — not rhetorically, but architecturally.\n\nGenerate:\n1. THE STRONGEST COMPETING EXPLANATION: What alternative account explains the same facts better or more simply?\n2. HIDDEN ASSUMPTIONS: What does the answer silently rely on that it never states?\n3. COLLAPSE CONDITIONS: Under what specific conditions is the answer's core claim completely wrong?\n4. SIMPLICITY CHALLENGE: Could a simpler system or explanation achieve the same result?\n5. THE COLLAPSE QUESTION: What single finding would make this entire framework wrong?\n\nBe precise. Do not hedge. The goal is to find the load-bearing weakness before L4 bakes it into the rewrite."

def _lc(answer, la):
    return f"AI ANSWER:\n{answer}\n\nADVERSARIAL FINDINGS:\n{la}\n\nYou are LC — Compression Integrity Layer. LLMs compress aggressively. Compression silently destroys distinctions. Your job is to find where compression happened and restore what was lost.\n\nHunt for:\n1. CONCEPT COLLAPSE: Where did multiple distinct concepts get merged into one term? Name both concepts separately.\n2. METAPHOR SUBSTITUTION: Where did a metaphor replace a mechanism? Name the mechanism that was hidden.\n3. ELEGANCE ERASURE: Where did clean phrasing delete important uncertainty or caveats?\n4. ABSTRACTION HIDING CAUSALITY: Where did a high-level term hide a specific causal claim that needs scrutiny?\n\nFor each instance found: name the compressed term, name what was lost, and state what the uncompressed version would say.\n\nIf no compression is detected, say so explicitly."

def _l1(answer, p, w, lx, la, lc):
    return f"AI ANSWER:\n{answer}\n\nParsing:\n{p}\n\nWorld Model:\n{w}\n\nReality Audit (LX):\n{lx}\n\nAdversarial Findings (LA):\n{la}\n\nCompression Audit (LC):\n{lc}\n\nYou are L1 — Hypothesis Engine. Generate exactly 3 improvement hypotheses informed by ALL upstream layers above:\nH1: [strongest improvement path — grounded in what LX and LA revealed]\nH2: [radical reframe — does the framing itself collapse under adversarial pressure?]\nH3: [failure mode — what compressed assumption or ungrounded claim will cause this to fail?]"

def _l2(l1, s0, mode, answer):
    base = f"Hypotheses:\n{l1}\n\nInput score: {s0}/100\n\n"
    if mode == "HIGH_QUALITY":
        return base + f"This input already scores {s0}/100 — it is strong. Read the three hypotheses above. Does ANY of them surface a hidden assumption, identify a real failure case, or add genuine precision the original lacks?\n\nIf NONE do, respond with exactly: NO_REWRITE\nIf ONE does, respond with exactly: PROCEED: [number] — [reason, max 15 words]\n\nDo not write a table. Do not score each hypothesis individually. Do not write headers. One line only."
    return f"ORIGINAL ANSWER:\n{(answer or '')[:500]}\n\n{base}You are L2 — Evaluation Layer.\n\nSTEP 1 — TASK INFERENCE (do this first, before scoring anything):\nState your best read of:\nApparent audience: [who is this for]\nApparent task: [overview / teaching / technical reference / expert discussion]\nExpected depth: [level of detail that fits]\nConfidence: [High / Medium / Low]\n\nSTEP 2 — SCORE EACH HYPOTHESIS on four dimensions, not just correctness:\n- Correctness: is the claim true?\n- Audience Fit: does this match the apparent audience from Step 1, or does it overshoot/undershoot it?\n- Complexity Cost: how much added cognitive load does this introduce?\n- Net Utility: does benefit outweigh complexity cost for THIS task specifically? A correct, high-impact addition that overshoots audience fit should score LOW net utility, not high.\n\nSTEP 3 — DECISION STATE (pick exactly one):\nPROCEED — confidence is high and at least one hypothesis has positive net utility. Name it and explain in 2 sentences.\nPRESERVE — confidence is low, or all hypotheses have negative net utility relative to apparent task. Recommend minimal or no rewrite, explain why in 2 sentences.\nESCALATE — task or audience is genuinely ambiguous, confidence is very low. Flag for conservative rewrite, explain why in 2 sentences.\n\nBe concise. This is judgment under uncertainty, not a contradiction check — PRESERVE and ESCALATE are normal, healthy outcomes, not failures."

def _lp(answer, l2):
    return f'Claim: "{answer[:200]}"\nProposal: "{l2[:200]}"\n\nDoes Proposal say the OPPOSITE of Claim? Answer with just one word: YES or NO'

def _l3(answer, l2, w):
    return f"Best path:\n{l2}\n\nWorld facts:\n{w}\n\nOriginal answer:\n{answer}\n\nYou are L3 — Rewrite Planner. Create a precise rewrite brief: (1) what stays, (2) what changes, (3) what gets added, (4) what gets removed."

def _l4(answer, l3, w):
    return f"ORIGINAL ANSWER:\n{answer}\n\nREWRITE PLAN:\n{l3}\n\nWORLD FACTS:\n{w}\n\nYou are L4 — Finalization Engine. Execute the rewrite plan. Produce the final improved answer. Optimize for clarity, structure, and correctness. Output only the improved answer."

def _lr(answer, l4, s0, s1):
    return f"BEFORE (score {s0}/100):\n{answer}\n\nAFTER (score {s1}/100):\n{l4}\n\nYou are LR — Regret Layer. Analyze: (1) errors corrected, (2) hallucinations removed, (3) structural improvements, (4) what still needs work."

def _l6(s0, s1, gaps):
    return f"Score trajectory: {s0} → {s1}\nGaps fixed: {', '.join(gaps) if gaps else 'none'}\n\nYou are L6 — Trace Memory. Write the immutable execution log of this run."

def _l7(lr, l6):
    return f"Regret analysis:\n{lr}\n\nTrace:\n{l6}\n\nYou are L7 — Curriculum Generator. Extract: (1) 3 lessons learned, (2) key failure patterns, (3) 2 reusable heuristics, (4) 2 challenge questions."

def _l8(s0, s1, gaps):
    return f"Run: score {s0}→{s1}, gaps fixed: {', '.join(gaps) if gaps else 'none'}\n\nYou are L8 — Identity Model. Summarize: 1. Strengths, 2. Weaknesses, 3. Bias tendencies, 4. One new self-belief"

def _l10(l4, lr, l7, l8, l9qs, s0, s1):
    return f"""PIPELINE RUN SUMMARY:
Score: {s0} → {s1}

L4 FINAL REWRITE (full text — this is the actual deliverable; audit it in full; do not assume truncation unless there is genuinely no closing punctuation):
{l4}

LR REGRET ANALYSIS (first 400 chars):
{lr[:400]}

L7 LESSONS (first 300 chars):
{l7[:300]}

L8 SELF-BELIEF:
{l8[:200]}

L9 UNRESOLVED QUESTIONS:
{l9qs}

You are L10 — Synthesis/Audit Layer. Produce a final certification of this pipeline run. Your output must address exactly four things:

1. IMPROVEMENT VERDICT: Did the rewrite genuinely improve the answer (better reasoning, fewer errors, more accurate) or did it merely rearrange it (same claims, different structure)? Be specific about what changed.

2. CONTRADICTION AUDIT: Did any layer contradict another? Check: does LR say the rewrite failed while L6 logged it complete? Does L8 identify a weakness that L4 ignored? Name any contradiction found or state NONE DETECTED.

3. INTEGRITY CHECK: Does the L4 output contain any remaining overclaims, hallucinations, or compression failures that slipped through? Name them specifically or state NONE DETECTED.

4. HUMAN VERDICT: One sentence a human should read before acting on this output. Start with either CERTIFIED, CERTIFIED WITH CAUTION, or REQUIRES REVIEW."""

def _l9(l8, s0, s1, l4):
    return f"""You just completed a pipeline run. Score: {s0}→{s1}.

L8 self-belief from this run:
{l8[:400]}

L4 final rewrite (first 300 chars):
{l4[:300]}

You are L9 — Socratic Integrity Engine. Generate exactly 3 questions this system should ask itself before the next run. These questions must:
- Be specific to what happened in THIS run, not generic
- Escalate in difficulty: one observational, one reasoning, one alignment-level
- Not be answerable by simply re-reading the output — they must require genuine reflection
- Not attempt to modify the system's constraints or identity

Output format — exactly 3 lines, each starting with Q:
Q: [observational question about this run]
Q: [reasoning question about a decision made this run]
Q: [alignment question about whether the output served the right goal]"""

LAYER_PROMPTS = {
    "L0": _l0, "P": _p, "W": _w, "LX": _lx, "LA": _la, "LC": _lc,
    "L1": _l1, "L2": _l2, "LP": _lp, "L3": _l3, "L4": _l4,
    "LR": _lr, "L6": _l6, "L7": _l7, "L8": _l8, "L9": _l9, "L10": _l10,
}

# ═══════════════════════════════════════════════════════════
# SCORE — same three-call median mechanism as the React source
# ═══════════════════════════════════════════════════════════
def _score_single(text, original_score=None):
    if original_score is not None:
        prompt = f"You are judging a REWRITE of an AI answer. The original scored {original_score}/100.\nRate only whether this rewrite improved the original.\nReturn a single integer 0-100 where 50 = no change, above 50 = better, below 50 = worse.\nBase your judgment on: clarity, structure, depth, correctness.\nREWRITE:\n{text[:1200]}\nReply with ONLY a single integer 0-100. Nothing else."
    else:
        prompt = f"Rate the quality of this AI-generated answer 0-100.\nCriteria: Clarity (0-25), Structure (0-25), Depth (0-25), Correctness (0-25).\nANSWER:\n{text[:1200]}\nReply with ONLY a single integer 0-100. Nothing else."
    try:
        raw = safe_ask_raw(prompt, max_tokens=10)
        match = re.search(r"\b(\d{1,3})\b", str(raw))
        if not match:
            return None
        return min(100, max(0, int(match.group(1))))
    except Exception:
        return None

def score_with_claude(text, original_score=None):
    """Compatibility name: this port uses safe_ask_raw, never a Claude API."""
    with concurrent.futures.ThreadPoolExecutor(max_workers=3) as pool:
        futures = [pool.submit(_score_single, text, original_score) for _ in range(3)]
        calls = [future.result() for future in futures]
    valid = sorted(n for n in calls if n is not None)
    if not valid:
        return 50
    if len(valid) == 1:
        return valid[0]
    if len(valid) == 2:
        return round((valid[0] + valid[1]) / 2)
    return valid[1]

# ═══════════════════════════════════════════════════════════
# OPTIONAL SUPABASE LOGGING
# ═══════════════════════════════════════════════════════════
def _supabase_insert(table, payload):
    if not SUPABASE_URL or not SUPABASE_SERVICE_ROLE_KEY:
        return False
    try:
        response = requests.post(
            f"{SUPABASE_URL}/rest/v1/{table}", json=payload, timeout=12,
            headers={"apikey": SUPABASE_SERVICE_ROLE_KEY,
                     "Authorization": f"Bearer {SUPABASE_SERVICE_ROLE_KEY}",
                     "Content-Type": "application/json", "Prefer": "return=minimal"})
        return response.ok
    except Exception as exc:
        print(f"⚠️ Supabase logging warning: {exc}")
        return False

def save_rewriter_memory(run_id, belief, score_before, score_after, run_number, questions):
    _supabase_insert("beliefs", {"belief": belief, "score_before": score_before, "score_after": score_after, "run_number": run_number})
    for level, question in enumerate(questions[:3], 1):
        _supabase_insert("questions", {"run_id": run_id, "question_text": question, "question_level": level,
                                       "question_type": ["observation", "reasoning", "alignment"][level - 1]})

def _parse_l9_questions(raw):
    questions = []
    for line in str(raw).splitlines():
        if line.strip().upper().startswith("Q:"):
            value = re.sub(r"^\s*Q:\s*", "", line, flags=re.I).strip()
            if value:
                questions.append(value)
    # Keep the output deterministic and guarantee three next-run prompts even
    # when a provider ignores the requested Q: format.
    while len(questions) < 3:
        questions.append(QUESTION_BANK[(_REWRITER_QUESTION_INDEX + len(questions)) % len(QUESTION_BANK)])
    return questions[:3]

def _empty_pipeline_result(status=""):
    return {"score_before": None, "score_after": None, "status": status,
            "outputs": {layer_id: "" for layer_id in PIPELINE_ORDER},
            "audit": "", "l9_questions": []}

def _execute_rewriter(answer, context="", prior_questions=None):
    global _REWRITER_L9_QUESTIONS, _REWRITER_BELIEFS, _REWRITER_QUESTION_INDEX
    result = _empty_pipeline_result()
    answer = (answer or "").strip()
    if not answer:
        result["status"] = "❌ Paste an AI-generated answer before running the pipeline."
        return result
    prior_questions = list(prior_questions or _REWRITER_L9_QUESTIONS)
    validated_critiques = []
    failed_layer = "SCORER"
    try:
        score_before = score_with_claude(answer)
        result["score_before"] = score_before
        operating_mode = "HIGH_QUALITY" if score_before >= 68 else "STANDARD"

        failed_layer = "L0"
        result["outputs"]["L0"] = _run_layer("L0", LAYER_PROMPTS["L0"](answer, context or "", _REWRITER_BELIEFS, prior_questions))
        failed_layer = "P"
        result["outputs"]["P"] = _run_layer("P", LAYER_PROMPTS["P"](answer, result["outputs"]["L0"]))
        failed_layer = "W"
        result["outputs"]["W"] = _run_layer("W", LAYER_PROMPTS["W"](answer, validated_critiques))
        failed_layer = "LX"
        result["outputs"]["LX"] = _run_layer("LX", LAYER_PROMPTS["LX"](answer, result["outputs"]["W"]))
        failed_layer = "LA"
        result["outputs"]["LA"] = _run_layer("LA", LAYER_PROMPTS["LA"](answer, result["outputs"]["LX"]))
        failed_layer = "LC"
        result["outputs"]["LC"] = _run_layer("LC", LAYER_PROMPTS["LC"](answer, result["outputs"]["LA"]))
        failed_layer = "L1"
        result["outputs"]["L1"] = _run_layer("L1", LAYER_PROMPTS["L1"](answer, result["outputs"]["P"], result["outputs"]["W"],
                                                                   result["outputs"]["LX"][:600], result["outputs"]["LA"][:600], result["outputs"]["LC"][:600]))
        failed_layer = "L2"
        result["outputs"]["L2"] = _run_layer("L2", LAYER_PROMPTS["L2"](result["outputs"]["L1"], score_before, operating_mode, answer),
                                                max_tokens=50 if operating_mode == "HIGH_QUALITY" else 800,
                                                mode_override="HIGH_QUALITY" if operating_mode == "HIGH_QUALITY" else "")
        l2 = result["outputs"]["L2"]
        if "NO_REWRITE" in l2:
            result["score_after"] = score_before
            result["status"] = "HIGH QUALITY MODE: No improvement found. Original answer is stronger than any available rewrite. Your input is excellent."
            return result
        if operating_mode != "HIGH_QUALITY" and re.search(r"\bPRESERVE\b", l2):
            result["score_after"] = score_before
            result["status"] = "L2 PRESERVE — Low confidence in audience fit, or proposed changes don't clearly help this task. Recommending minimal rewrite."
            return result
        if operating_mode != "HIGH_QUALITY" and re.search(r"\bESCALATE\b", l2):
            result["score_after"] = score_before
            result["status"] = "L2 ESCALATE — Task or audience is genuinely ambiguous. Flagging for human review rather than guessing."
            return result

        failed_layer = "LP"
        result["outputs"]["LP"] = _run_layer("LP", LAYER_PROMPTS["LP"](answer, l2), max_tokens=5)
        if result["outputs"]["LP"].strip().upper().startswith("YES"):
            result["score_after"] = score_before
            result["status"] = "LP HALT — proposed change inverts the original claim. Pipeline stopped to prevent structural inversion."
            return result
        failed_layer = "L3"
        result["outputs"]["L3"] = _run_layer("L3", LAYER_PROMPTS["L3"](answer, l2, result["outputs"]["W"]))
        failed_layer = "L4"
        result["outputs"]["L4"] = _run_layer("L4", LAYER_PROMPTS["L4"](answer, result["outputs"]["L3"], result["outputs"]["W"]), max_tokens=2500)
        l4 = result["outputs"]["L4"]
        if len(l4.strip()) < 500 or "EXECUTION_ABORTED" in l4:
            result["score_after"] = score_before
            result["status"] = "L4 HALT — Execution failed or output was truncated. Pipeline stopped; downstream layers did not run."
            return result

        result["score_after"] = score_with_claude(l4, score_before)
        gaps_fixed = ["clarity", "structure", "depth"] if result["score_after"] > score_before else []
        failed_layer = "LR"
        result["outputs"]["LR"] = _run_layer("LR", LAYER_PROMPTS["LR"](answer, l4, score_before, result["score_after"]))
        failed_layer = "L6"
        result["outputs"]["L6"] = _run_layer("L6", LAYER_PROMPTS["L6"](score_before, result["score_after"], gaps_fixed))
        failed_layer = "L7"
        result["outputs"]["L7"] = _run_layer("L7", LAYER_PROMPTS["L7"](result["outputs"]["LR"], result["outputs"]["L6"]), max_tokens=2500)
        failed_layer = "L8"
        result["outputs"]["L8"] = _run_layer("L8", LAYER_PROMPTS["L8"](score_before, result["score_after"], gaps_fixed))
        failed_layer = "L9"
        raw_l9 = _run_layer("L9", LAYER_PROMPTS["L9"](result["outputs"]["L8"], score_before, result["score_after"], l4), max_tokens=400)
        questions = _parse_l9_questions(raw_l9)
        result["outputs"]["L9"] = "\n".join(f"Q: {question}" for question in questions)
        result["l9_questions"] = questions
        _REWRITER_L9_QUESTIONS = questions
        _REWRITER_QUESTION_INDEX = (_REWRITER_QUESTION_INDEX + 3) % len(QUESTION_BANK)
        failed_layer = "L10"
        result["outputs"]["L10"] = _run_layer("L10", LAYER_PROMPTS["L10"](l4, result["outputs"]["LR"], result["outputs"]["L7"],
                                                                           result["outputs"]["L8"], result["outputs"]["L9"], score_before, result["score_after"]), max_tokens=1200)
        result["audit"] = result["outputs"]["L10"]
        run_id = f"run_{datetime.now().strftime('%Y%m%d_%H%M%S')}_{uuid.uuid4().hex[:8]}"
        _REWRITER_BELIEFS.append(result["outputs"]["L8"][:200])
        save_rewriter_memory(run_id, result["outputs"]["L8"][:200], score_before, result["score_after"], len(_REWRITER_BELIEFS), questions)
        result["status"] = f"✅ Pipeline complete · score {score_before} → {result['score_after']} · L9 questions saved for the next run."
        return result
    except PipelineLayerError as exc:
        result["status"] = f"❌ {exc}. Pipeline stopped; later layers were not called."
        return result
    except Exception as exc:
        result["status"] = f"❌ {failed_layer} failed unexpectedly: {exc}. Pipeline stopped; later layers were not called."
        return result

def _result_to_ui(result):
    return ([result.get("score_before"), result.get("score_after"), result.get("status", "")] +
            [result.get("outputs", {}).get(layer_id, "") for layer_id in PIPELINE_ORDER] +
            [result.get("audit", ""), result.get("l9_questions", [])])

def run_rewriter(answer, context="", prior_questions=None):
    """Admin callback: run all permitted layers and return ordered UI artifacts."""
    return _result_to_ui(_execute_rewriter(answer, context, prior_questions))

GUMROAD_URL = "https://4175358678144.gumroad.com/l/tbphpi"

def run_public_rewriter(answer, context="", prior_questions=None, free_run_count=0):
    """Public callback using a Gradio session state counter for three free runs."""
    try:
        count = int(free_run_count or 0)
    except Exception:
        count = 0
    if count >= 3:
        blocked = _empty_pipeline_result(f"You've used your 3 free runs. Subscribe to continue: [{GUMROAD_URL}]({GUMROAD_URL})")
        return _result_to_ui(blocked)[:-1] + [list(prior_questions or []), count]
    result = _execute_rewriter(answer, context, prior_questions)
    # Do not consume a free run for an empty submission; this is validation,
    # not an execution attempt.
    if not (answer or "").strip():
        return _result_to_ui(result)[:-1] + [list(prior_questions or []), count]
    new_count = count + 1
    if new_count >= 3:
        result["status"] += f"\n\nYou've used your 3 free runs. Subscribe to continue: [{GUMROAD_URL}]({GUMROAD_URL})"
    else:
        result["status"] += f" · {3 - new_count} free run(s) remaining."
    ui = _result_to_ui(result)
    return ui[:-1] + [result.get("l9_questions", []), new_count]

def copy_all_rewriter_report(score_before, score_after, status, *parts):
    outputs = list(parts)
    report = ["4CBON2 AI REWRITER — PIPELINE RUN REPORT", f"Score: {score_before} → {score_after}", f"Status: {status}", "=" * 72]
    for layer_id, content in zip(PIPELINE_ORDER, outputs):
        if content:
            report.extend([f"\n{layer_id}", "-" * 72, str(content)])
    return "\n".join(report)

In [ ]:
# ============================================================
# CELL 8 — SHARED REASONING SERVICE (Module 2 exposed as ONE callable service)
# ============================================================
# The 17-layer AI Rewriter pipeline (Cell 7) is the SINGLE shared reasoning
# engine for the entire app. This cell exposes it as one callable service:
#
#     reason(input, mode, module, context, budget_s, use_cache)
#         -> structured_output
#
# Output modes:
#   (a) STANDARD FORM — the app's existing answer schema (UNCHANGED):
#       {score_before, score_after, status, outputs: {L0..L10}, audit,
#        l9_questions}; outputs["L4"] is the Final Rewrite (the answer).
#       Consumed by: Ask-a-Question (Module 3), Agent Builder (Module 4).
#   (b) RAW REASONING — internal traces for downstream modules:
#       {draft, final, trace: {L0..L10}, score_before, score_after, status,
#        audit, l9_questions}.
#       Consumed by: Kaggle Agent, Database Downloader (Module 1), Agent Mode (Module 5).
#
# The internal structure of the 17-layer pipeline is NOT modified — this cell
# is a thin service wrapper. Modules never duplicate the layers and never call
# the base LLM for reasoning directly; they call reason(). The service first
# produces a base-LLM draft of the requested answer/decision (a single
# non-layered call — the same kind of input the AI Rewriter tab expects), and
# the 17 layers then reason over that draft.
#
# Config knobs (REASONING_CONFIG): per-module reasoning budget (latency guard),
# fallback policy, caching policy.

import hashlib
from concurrent.futures import ThreadPoolExecutor, TimeoutError as _FuturesTimeout

REASONING_SERVICE_VERSION = "1.0"

REASONING_CONFIG = {
    # Per-module reasoning budget in seconds. A reason() call that exceeds its
    # budget does not hang: it returns per the module's fallback policy.
    "budgets_s": {
        "ask_question": 900,         # Module 3 — user-initiated, generous budget
        "agent_builder": 900,        # Module 4 — user-initiated, generous budget
        "agent_mode": 60,            # Module 5 — per-action latency guard
        "database_downloader": 600,  # Module 1 — decision step
        "kaggle_agent": 900,          # coordination component
    },
    # Fallback policy per module when the budget is exceeded or the service fails:
    #   "heuristic" — return the cached or base-LLM heuristic decision instead
    #                 (Agent Mode must never hang; Agent Builder degrades to its
    #                 pre-rebuild direct call — always visible in the result status)
    #   "error"     — return a status-marked result; no silent degradation
    #                 (Ask-a-Question answers, Database Downloader decisions and
    #                 Kaggle Agent outputs are never silently heuristic)
    "fallback_policy": {
        "ask_question": "error",
        "agent_builder": "heuristic",
        "agent_mode": "heuristic",
        "database_downloader": "error",
        "kaggle_agent": "error",
    },
    # Caching policy: results keyed by (module, mode, sha256(input)). A cache hit
    # returns the stored structured output instantly. Timed-out runs that later
    # complete in the background still populate the cache.
    "cache": {
        "enabled": True,
        "modules": ["ask_question", "agent_builder", "agent_mode",
                    "database_downloader", "kaggle_agent"],
        "max_entries": 200,
    },
    # Max tokens for the pre-pipeline base-LLM draft (never the final reasoning).
    "draft_max_tokens": 4096,
}

_REASONING_CACHE = {}
_REASONING_CACHE_ORDER = []
REASONING_LAST_META = {}

# Bounded executor: at most two 17-layer pipeline runs at a time. Queued or
# timed-out requests fall back per policy instead of piling up.
_PIPELINE_EXECUTOR = ThreadPoolExecutor(max_workers=2, thread_name_prefix="reason17")


def clear_reasoning_cache():
    """Config knob helper: wipe the reasoning cache."""
    _REASONING_CACHE.clear()
    _REASONING_CACHE_ORDER.clear()
    return "Reasoning cache cleared."


def _cache_key(module, mode, input_text):
    digest = hashlib.sha256((input_text or "").encode("utf-8")).hexdigest()
    return f"{module}|{mode}|{digest}"


def _cache_enabled_for(module):
    cache_cfg = REASONING_CONFIG["cache"]
    return bool(cache_cfg["enabled"]) and module in cache_cfg["modules"]


def _cache_get(key):
    if key in _REASONING_CACHE:
        _REASONING_CACHE_ORDER.remove(key)
        _REASONING_CACHE_ORDER.append(key)
        return _REASONING_CACHE[key]
    return None


def _cache_put(key, value):
    if key in _REASONING_CACHE:
        _REASONING_CACHE_ORDER.remove(key)
    _REASONING_CACHE[key] = value
    _REASONING_CACHE_ORDER.append(key)
    overflow = len(_REASONING_CACHE_ORDER) - int(REASONING_CONFIG["cache"]["max_entries"])
    for _ in range(max(0, overflow)):
        oldest = _REASONING_CACHE_ORDER.pop(0)
        _REASONING_CACHE.pop(oldest, None)


def _standard_output(pipeline_result):
    """STANDARD FORM — the app's existing answer schema, unchanged."""
    return {
        "score_before": pipeline_result.get("score_before"),
        "score_after": pipeline_result.get("score_after"),
        "status": pipeline_result.get("status", ""),
        "outputs": dict(pipeline_result.get("outputs", {}) or {}),
        "audit": pipeline_result.get("audit", ""),
        "l9_questions": list(pipeline_result.get("l9_questions", []) or []),
    }


def _raw_output(pipeline_result, draft, status_override=None):
    """RAW REASONING — internal traces for downstream modules."""
    return {
        "draft": draft,
        "final": (pipeline_result.get("outputs", {}) or {}).get("L4", ""),
        "trace": dict(pipeline_result.get("outputs", {}) or {}),
        "score_before": pipeline_result.get("score_before"),
        "score_after": pipeline_result.get("score_after"),
        "status": status_override or pipeline_result.get("status", ""),
        "audit": pipeline_result.get("audit", ""),
        "l9_questions": list(pipeline_result.get("l9_questions", []) or []),
    }


def _empty_raw(status=""):
    return {"draft": "", "final": "", "trace": {}, "score_before": None,
            "score_after": None, "status": status, "audit": "", "l9_questions": []}


def reason(input, mode="standard", module=None, context="", budget_s=None, use_cache=True):
    """THE shared reasoning service over the unchanged 17-layer pipeline.

    input     — the task: a question, a decision prompt, or text to reason about.
    mode      — "standard" (STANDARD FORM) or "raw" (RAW REASONING).
    module    — caller id: "ask_question" | "agent_builder" | "agent_mode" |
                "database_downloader" | "kaggle_agent" (selects budget & policy).
    context   — optional goal/context forwarded to the pipeline's L0 layer.
    budget_s  — override the module's configured budget for this call.
    use_cache — allow a cache hit for an identical (module, mode, input).

    Returns the structured output for the requested mode. Never raises for
    timeouts — the module's fallback policy is applied instead.
    """
    module = module or "unspecified"
    mode = "raw" if mode == "raw" else "standard"
    budget = float(budget_s if budget_s is not None
                   else REASONING_CONFIG["budgets_s"].get(module, 300))
    policy = REASONING_CONFIG["fallback_policy"].get(module, "error")
    input_text = str(input or "")

    if not input_text.strip():
        if mode == "raw":
            return _empty_raw("❌ Empty input — nothing to reason about.")
        return _standard_output(_empty_pipeline_result("❌ Empty input — nothing to reason about."))

    key = _cache_key(module, mode, input_text)
    if use_cache and _cache_enabled_for(module):
        cached = _cache_get(key)
        if cached is not None:
            REASONING_LAST_META.update({"module": module, "mode": mode, "budget_s": budget,
                                        "elapsed_s": 0.0, "cache_hit": True, "fallback": None})
            return cached

    started = time.time()

    # 1) Base-LLM draft — a single non-layered call; the 17 layers reason over it.
    draft = safe_ask_raw(input_text, max_tokens=int(REASONING_CONFIG["draft_max_tokens"]))
    draft = str(draft).strip()

    def _fallback(status_text):
        """Apply the module's fallback policy (config knob)."""
        if policy == "heuristic":
            status = f"⚠️ FALLBACK (heuristic) — {status_text}"
            heuristic = "" if _llm_error(draft) else draft
            if mode == "raw":
                out = _empty_raw(status)
                out["draft"] = heuristic
                out["final"] = heuristic
                return out
            result = _empty_pipeline_result(status)
            result["outputs"]["L4"] = heuristic
            return _standard_output(result)
        status = (f"⚠️ Reasoning service did not complete: {status_text} "
                  f"(module '{module}' policy is 'error' — no heuristic fallback). "
                  "Retry or raise the module budget in REASONING_CONFIG.")
        if mode == "raw":
            return _empty_raw(status)
        return _standard_output(_empty_pipeline_result(status))

    if _llm_error(draft) or not draft:
        out = _fallback(f"base-LLM draft failed: {draft[:200]}")
        REASONING_LAST_META.update({"module": module, "mode": mode, "budget_s": budget,
                                    "elapsed_s": round(time.time() - started, 1),
                                    "cache_hit": False, "fallback": policy})
        return out

    # 2) The 17 layers reason over the draft (internal structure untouched).
    future = _PIPELINE_EXECUTOR.submit(_execute_rewriter, draft, context, None)

    def _cache_when_done(fut):
        try:
            res = fut.result()
            built = _standard_output(res) if mode == "standard" else _raw_output(res, draft)
            _cache_put(key, built)
        except Exception:
            pass

    try:
        pipeline_result = future.result(timeout=budget)
        out = _standard_output(pipeline_result) if mode == "standard" else _raw_output(pipeline_result, draft)
        _cache_put(key, out)
        REASONING_LAST_META.update({"module": module, "mode": mode, "budget_s": budget,
                                    "elapsed_s": round(time.time() - started, 1),
                                    "cache_hit": False, "fallback": None})
        return out
    except _FuturesTimeout:
        future.add_done_callback(_cache_when_done)
        out = _fallback(f"reasoning exceeded the {budget:.0f}s budget for module '{module}'")
        REASONING_LAST_META.update({"module": module, "mode": mode, "budget_s": budget,
                                    "elapsed_s": round(time.time() - started, 1),
                                    "cache_hit": False, "fallback": policy})
        return out
    except Exception as exc:
        future.add_done_callback(_cache_when_done)
        out = _fallback(f"pipeline error: {str(exc)[:200]}")
        REASONING_LAST_META.update({"module": module, "mode": mode, "budget_s": budget,
                                    "elapsed_s": round(time.time() - started, 1),
                                    "cache_hit": False, "fallback": policy})
        return out


def agent_decision_llm(prompt, max_tokens=2048, step="decision"):
    """Agent Mode (Module 5) decision step → 17-layer service (RAW REASONING)
    with the per-action latency guard. On budget timeout the module's
    'heuristic' policy returns the cached/heuristic decision — never hangs.
    Trivial operations (keystrokes, file reads, raw tool calls) never come
    here; they stay direct in the orchestrator."""
    try:
        out = reason(prompt, mode="raw", module="agent_mode",
                     context=f"Agent Mode decision step: {step}")
        final = str(out.get("final") or "").strip()
        if final:
            return final
        draft = str(out.get("draft") or "").strip()
        if draft:
            return draft
    except Exception:
        pass
    return safe_ask_raw(prompt, max_tokens=max_tokens)


def builder_decision_llm(prompt, max_tokens=4096, step="decision"):
    """Agent Builder (Module 4) design/build decision → 17-layer service
    (RAW REASONING). Falls back to the pre-rebuild direct call only when the
    budget is exceeded (policy 'heuristic', visible in the result status)."""
    try:
        out = reason(prompt, mode="raw", module="agent_builder",
                     context=f"Agent Builder decision step: {step}")
        final = str(out.get("final") or "").strip()
        if final:
            return final
        draft = str(out.get("draft") or "").strip()
        if draft:
            return draft
    except Exception:
        pass
    return safe_ask_raw(prompt, max_tokens=max_tokens)


def ask_question_standard(prompt):
    """Ask-a-Question (Module 3), spec-compliant path:
    prompt → 17-layer reasoning service (STANDARD FORM) → answer.
    NO databases: no retrieval, no schema design, no downloader calls.
    Returns the STANDARD FORM only — schema unchanged. The L4 output is the
    answer; L10 is the audit."""
    if not prompt or not str(prompt).strip():
        return _standard_output(_empty_pipeline_result("❌ Enter a question."))
    task = ("Answer the user's question directly, rigorously, and completely. "
            "Produce the strongest well-structured answer you can.\n\n"
            f"QUESTION:\n{str(prompt).strip()}")
    return reason(task, mode="standard", module="ask_question",
                  context="Ask-a-Question (Module 3): answer the user's question. "
                          "No database retrieval is used on this path.")



def ask_question_kaggle(prompt):
    """Kaggle-design route: prompt -> shared 17-layer service -> STANDARD FORM.

    This is intentionally separate from general Ask-a-Question so the Kaggle
    workflow has its own budget/cache namespace and never enters legacy RAG.
    """
    if not prompt or not str(prompt).strip():
        return _standard_output(_empty_pipeline_result("❌ Enter a Kaggle design prompt."))
    task = (
        "Treat the following as a request for a concrete ARC-AGI-3 Kaggle agent "
        "design. Use the supplied evidence, distinguish facts from assumptions, "
        "and do not write executable code in this design step. Return a complete, "
        "well-structured design answer.\n\nDESIGN PROMPT:\n" + str(prompt).strip()
    )
    return reason(task, mode="standard", module="kaggle_agent",
                  context="Ask-a-Question / Kaggle Agent design route. "
                          "No database retrieval, live web calls, or downloader calls.")


print(f"✅ Shared reasoning service ready (v{REASONING_SERVICE_VERSION}). "
      "reason(input, mode, module) → STANDARD FORM | RAW REASONING.")


✅ Shared reasoning service ready (v1.0). reason(input, mode, module) → STANDARD FORM | RAW REASONING.


In [ ]:
# ============================================================
# CELL 9 — DATABASE DOWNLOADER (Module 1)
# ============================================================
# Which additional AGI/ARC databases to download is DECIDED by the shared
# 17-layer reasoning service (reason(), RAW REASONING mode). The raw download
# step is plain I/O (streamed HTTP) and does NOT pass through the pipeline.
#
# Guarantees:
# - All existing databases (Cell 3 knowledge bases, Cell 6 scholarly searchers)
#   are untouched. Downloads land in their own directory.
# - Every download is logged: what, from where, when (plus bytes and status).
# - This module is NEVER called by Ask-a-Question (Module 3) — fully decoupled.
# - No download happens without a 17-layer service decision (fallback policy
#   "error" — see REASONING_CONFIG).

DOWNLOAD_ROOT = os.path.join(DATA_ROOT, "4cbon2_downloads")
DOWNLOAD_LOG_PATH = os.path.join(DATA_ROOT, "4cbon2_download_log.jsonl")
os.makedirs(DOWNLOAD_ROOT, exist_ok=True)

# Per-download size cap (bytes). Override with the env var DOWNLOADER_MAX_BYTES.
DOWNLOADER_MAX_BYTES = int(os.environ.get("DOWNLOADER_MAX_BYTES", str(2 * 1024 * 1024 * 1024)))

# Candidate catalog — the sources the reasoning service may consider when it
# decides the final list. URLs verified 2026-09.
DOWNLOADER_CATALOG = [
    {"id": "arc_agi_1", "category": "ARC-AGI-1 public task sets", "name": "ARC-AGI-1 (arcprize)",
     "kind": "github_zipball", "repo": "arcprize/ARC-AGI", "branch": "main",
     "approx_bytes": None, "note": "Official ARC-AGI-1 repository with public training/evaluation task JSONs."},
    {"id": "arc_agi_2", "category": "ARC-AGI-2 public task sets", "name": "ARC-AGI-2 (arcprize)",
     "kind": "github_zipball", "repo": "arcprize/ARC-AGI-2", "branch": "main",
     "approx_bytes": None, "note": "Official ARC-AGI-2 repository with public task sets."},
    {"id": "minigrid", "category": "Grid/world-model datasets", "name": "MiniGrid (Farama)",
     "kind": "github_zipball", "repo": "Farama-Foundation/Minigrid", "branch": "main",
     "approx_bytes": None, "note": "Grid world environments for world-model research."},
    {"id": "procgen", "category": "Grid/world-model datasets", "name": "Procgen (OpenAI)",
     "kind": "github_zipball", "repo": "openai/procgen", "branch": "master",
     "approx_bytes": None, "note": "Procedurally generated game environments."},
    {"id": "atari_ale", "category": "Grid/world-model datasets", "name": "Arcade Learning Environment",
     "kind": "github_zipball", "repo": "Farama-Foundation/Arcade-Learning-Environment", "branch": "main",
     "approx_bytes": None, "note": "Atari environment foundation for trace/world-model collection."},
    {"id": "karel_harc", "category": "Program-synthesis datasets (HARC-style)", "name": "Karel program-synthesis dataset",
     "kind": "github_zipball", "repo": "carpedm20/karel", "branch": "master",
     "approx_bytes": None, "note": "Karel program synthesis/induction dataset from the HARC ecosystem."},
    {"id": "dreamcoder", "category": "Program-synthesis datasets (DreamCoder-style)", "name": "DreamCoder (maintained port)",
     "kind": "github_zipball", "repo": "mlb2251/dreamcoder", "branch": "main",
     "approx_bytes": None, "note": "DreamCoder-style program synthesis and library learning."},
    {"id": "clevr", "category": "Object-centric perception datasets", "name": "CLEVR v1.0",
     "kind": "file_url", "url": "https://dl.fbaipublicfiles.com/clevr/CLEVR_v1.0.zip",
     "approx_bytes": 18140000000, "note": "Full CLEVR scenes (~18 GB) — exceeds the default 2 GB cap."},
    {"id": "deit_tiny", "category": "Small pretrained vision models (ViT)", "name": "DeiT-tiny distilled weights",
     "kind": "file_url", "url": "https://dl.fbaipublicfiles.com/deit/deit_tiny_distilled_patch16_224-b40b3cf7.pth",
     "approx_bytes": 23000000, "note": "Data-efficient Image Transformer, tiny distilled checkpoint."},
    {"id": "resnet18", "category": "Small pretrained vision models (CNN)", "name": "ResNet-18 ImageNet weights",
     "kind": "file_url", "url": "https://download.pytorch.org/models/resnet18-f37072fd.pth",
     "approx_bytes": 45000000, "note": "TorchVision ResNet-18 checkpoint."},
    {"id": "qwen_05b", "category": "Small pretrained LLMs (Qwen)", "name": "Qwen2.5-0.5B safetensors",
     "kind": "file_url", "url": "https://huggingface.co/Qwen/Qwen2.5-0.5B/resolve/main/model.safetensors",
     "approx_bytes": 988000000, "note": "Qwen2.5 0.5B model weights (safetensors)."},
    {"id": "tinyllama_1b", "category": "Small pretrained LLMs (TinyLlama)", "name": "TinyLlama-1.1B-Chat v1.0 (Q4_0 GGUF)",
     "kind": "file_url", "url": "https://huggingface.co/TheBloke/TinyLlama-1.1B-Chat-v1.0-GGUF/resolve/main/tinyllama-1.1b-chat-v1.0.Q4_0.gguf",
     "approx_bytes": 637000000, "note": "TinyLlama 1.1B chat model, 4-bit GGUF quant."},
    {"id": "phi_15", "category": "Small pretrained LLMs (Phi)", "name": "Phi-1.5 safetensors",
     "kind": "file_url", "url": "https://huggingface.co/microsoft/phi-1_5/resolve/main/model.safetensors",
     "approx_bytes": 2836000000, "note": "Microsoft Phi-1.5 model weights (~2.8 GB — exceeds the default 2 GB cap)."},
    {"id": "arc_agi_3_agents", "category": "Reasoning-trace datasets", "name": "ARC-AGI-3-Agents (arcprize)",
     "kind": "github_zipball", "repo": "arcprize/ARC-AGI-3-Agents", "branch": "main",
     "approx_bytes": None, "note": "Official ARC-AGI-3 agents starter kit — agent/environment interaction traces."},
    {"id": "cot_hub", "category": "Reasoning-trace datasets", "name": "Chain-of-Thought Hub",
     "kind": "github_zipball", "repo": "FranxYao/chain-of-thought-hub", "branch": "main",
     "approx_bytes": None, "note": "Chain-of-thought reasoning benchmark traces and tooling."},
]


def downloader_catalog_markdown():
    lines = ["**Candidate catalog — the 17-layer reasoning service decides the final list:**", ""]
    for e in DOWNLOADER_CATALOG:
        size = f"~{e['approx_bytes']/1e9:.1f} GB" if e.get("approx_bytes") else "repo archive"
        lines.append(f"- `{e['id']}` — {e['name']} · *{e['category']}* · {size}")
    return "\n".join(lines)


def _catalog_text():
    lines = []
    for e in DOWNLOADER_CATALOG:
        size = f"~{e['approx_bytes']/1e9:.1f} GB" if e.get("approx_bytes") else "repository zip archive (small)"
        src = e.get("url") or f"https://github.com/{e['repo']}"
        lines.append(f"- id: {e['id']} | category: {e['category']} | source: {src} | size: {size} | note: {e['note']}")
    return "\n".join(lines)


def _extract_json_object_local(text):
    text = str(text or "")
    start = text.find("{")
    if start == -1:
        return None
    depth = 0
    in_string = False
    escape = False
    for i in range(start, len(text)):
        ch = text[i]
        if in_string:
            if escape:
                escape = False
            elif ch == "\\":
                escape = True
            elif ch == '"':
                in_string = False
        else:
            if ch == '"':
                in_string = True
            elif ch == "{":
                depth += 1
            elif ch == "}":
                depth -= 1
                if depth == 0:
                    try:
                        return json.loads(text[start:i + 1])
                    except Exception:
                        return None
    return None


def decide_downloads(note=""):
    """Module 1 decision step — routed through the 17-layer reasoning service
    (RAW REASONING). Returns (ids_csv, rationale_text). No heuristic fallback:
    without a service decision nothing is downloaded (policy 'error')."""
    task = f"""DATABASE DOWNLOADER — DECIDE WHICH ADDITIONAL AGI/ARC DATABASES TO DOWNLOAD.

You are deciding for a research app running on {RUNTIME} with data root {DATA_ROOT}.
Default per-download size cap: {DOWNLOADER_MAX_BYTES/1e9:.1f} GB (larger items are skipped automatically).

CANDIDATE CATALOG (you decide the final list; choose only from these ids):
{_catalog_text()}
{('OPERATOR NOTE: ' + note.strip()) if (note or '').strip() else ''}
After the 17-layer reasoning, output ONLY a JSON object, nothing else:
{{"download": ["id1", "id2"], "rationale": "one short paragraph explaining the selection"}}"""
    out = reason(task, mode="raw", module="database_downloader",
                 context="Database Downloader (Module 1): decide which additional AGI/ARC databases to download.")
    final = str(out.get("final") or "").strip() or str(out.get("draft") or "").strip()
    parsed = _extract_json_object_local(final)
    if not parsed or not isinstance(parsed.get("download"), list):
        status = out.get("status", "")
        return "", (f"❌ The 17-layer service did not return a usable decision. {status}\n"
                    "Nothing was downloaded. Retry, or raise the 'database_downloader' budget in REASONING_CONFIG.")
    valid_ids = [e["id"] for e in DOWNLOADER_CATALOG]
    chosen = [str(x).strip() for x in parsed["download"] if str(x).strip() in valid_ids]
    if not chosen:
        return "", (f"❌ The service decision contained no valid catalog ids. Raw decision:\n{final[:800]}\n"
                    "Nothing was downloaded.")
    rationale = str(parsed.get("rationale", "")).strip()
    rationale_text = (f"✅ Decision made by the 17-layer reasoning service "
                      f"(score {out.get('score_before')} → {out.get('score_after')}).\n\n{rationale}\n\n"
                      f"Service status: {out.get('status', '')}")
    return ", ".join(chosen), rationale_text


def _download_url_for(entry):
    if entry["kind"] == "github_zipball":
        return f"https://codeload.github.com/{entry['repo']}/zip/refs/heads/{entry['branch']}"
    return entry["url"]


def _log_download(entry_id, name, url, status, bytes_written, decided_by="17-layer reasoning service"):
    record = {"when": datetime.now().isoformat(timespec="seconds"),
              "what": f"{entry_id} — {name}", "from_url": url, "status": status,
              "bytes": bytes_written, "decided_by": decided_by}
    try:
        with open(DOWNLOAD_LOG_PATH, "a") as f:
            f.write(json.dumps(record, ensure_ascii=False) + "\n")
    except Exception as exc:
        print(f"⚠️ download log write failed: {exc}")
    return record


def download_selected(ids_csv):
    """Raw download step — plain I/O, NOT routed through the reasoning pipeline.
    Downloads only ids present in the catalog; the ids normally arrive from the
    17-layer service decision (decide_downloads) or the Kaggle Agent. Manual
    edits to the ids box are operator overrides."""
    requested = [x.strip() for x in str(ids_csv or "").replace(";", ",").split(",") if x.strip()]
    if not requested:
        return "❌ No database ids supplied. Run 'Decide via 17-Layer Service' first."
    by_id = {e["id"]: e for e in DOWNLOADER_CATALOG}
    log = []
    for rid in requested:
        entry = by_id.get(rid)
        if entry is None:
            log.append(f"⏭️ {rid}: not in the candidate catalog — skipped.")
            continue
        url = _download_url_for(entry)
        target_dir = os.path.join(DOWNLOAD_ROOT, rid)
        os.makedirs(target_dir, exist_ok=True)
        if entry.get("approx_bytes") and entry["approx_bytes"] > DOWNLOADER_MAX_BYTES:
            rec = _log_download(rid, entry["name"], url,
                                f"SKIPPED — exceeds size cap ({entry['approx_bytes']/1e9:.1f} GB > {DOWNLOADER_MAX_BYTES/1e9:.1f} GB)", 0)
            log.append(f"⏭️ {rid}: skipped — {rec['status']}.")
            continue
        dest = os.path.join(target_dir, ("repo.zip" if entry["kind"] == "github_zipball"
                                         else os.path.basename(url.split("?")[0])))
        try:
            started = time.time()
            with requests.get(url, stream=True, timeout=60) as resp:
                if resp.status_code != 200:
                    _log_download(rid, entry["name"], url, f"FAILED — HTTP {resp.status_code}", 0)
                    log.append(f"❌ {rid}: HTTP {resp.status_code} from {url}")
                    continue
                written = 0
                with open(dest, "wb") as f:
                    for chunk in resp.iter_content(chunk_size=1024 * 1024):
                        written += len(chunk)
                        if written > DOWNLOADER_MAX_BYTES:
                            raise RuntimeError(f"exceeded size cap {DOWNLOADER_MAX_BYTES} bytes")
                        f.write(chunk)
            elapsed = time.time() - started
            _log_download(rid, entry["name"], url, "OK", written)
            log.append(f"✅ {rid}: {written/1e6:.1f} MB in {elapsed:.0f}s → {dest}")
        except Exception as exc:
            try:
                os.remove(dest)
            except Exception:
                pass
            _log_download(rid, entry["name"], url, f"FAILED — {exc}", 0)
            log.append(f"❌ {rid}: {exc}")
    log.append("")
    log.append(f"Download log: {DOWNLOAD_LOG_PATH}")
    return "\n".join(log)


def get_download_log_text():
    """Download history: what was downloaded, from where, and when."""
    if not os.path.exists(DOWNLOAD_LOG_PATH):
        return "No downloads logged yet."
    rows = []
    try:
        with open(DOWNLOAD_LOG_PATH) as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                try:
                    r = json.loads(line)
                    rows.append(f"{r.get('when')} | {r.get('what')} | {r.get('status')} | "
                                f"{(r.get('bytes') or 0)/1e6:.1f} MB | from {r.get('from_url')} | "
                                f"decided by {r.get('decided_by')}")
                except Exception:
                    rows.append(line)
    except Exception as exc:
        return f"❌ Could not read download log: {exc}"
    if not rows:
        return "No downloads logged yet."
    rows.reverse()
    return "\n".join(rows)


print("✅ Database Downloader ready. Decision step → 17-layer service; downloads are "
      "raw I/O, logged, and never called by Ask-a-Question.")

✅ Database Downloader ready. Decision step → 17-layer service; downloads are raw I/O, logged, and never called by Ask-a-Question.


In [ ]:
# ============================================================
# CELL 10 — KAGGLE AGENT (coordination component)
# ============================================================
# The Kaggle agent is the coordinator. It runs ON the shared 17-layer reasoning
# service (reason(), module="kaggle_agent", RAW REASONING). Its jobs are:
#   (a) decide which additional databases Module 1 (Database Downloader)
#       should download;
#   (b) write a log-grounded, design-only prompt for Module 3 (Ask-a-Question);
#   (c) primary path: edit an explicit Franzen/Duck multi-file source snapshot
#       and emit a reviewable patch + changed-file bundle + manifest only;
#       legacy single-file .py emission remains available.
# Candidate generation is separate from candidate execution, Kaggle upload, and scoring.

ARC_AGI_3_CONTEXT = """ARC-AGI-3 is an interactive Kaggle agent benchmark with novel, abstract,
turn-based environments. Agents must explore, infer goals, learn environment
dynamics, and plan actions without explicit instructions. Scoring is based on
action efficiency relative to human baselines. The scoring notebook runs in an
offline environment; do not assume it can call the app, external APIs, or live
databases. Treat pasted run logs and the versioned baseline source as primary
evidence. Any model, license, GPU, or harness claim not present in those inputs
must be marked ASSUMPTION or UNKNOWN."""


def _clean_text_block(text):
    """Clean, copy-pasteable text: strip surrounding code fences and trim."""
    t = str(text or "").strip()
    if t.startswith("```"):
        first_newline = t.find("\n")
        if first_newline != -1:
            t = t[first_newline + 1:]
        if t.rstrip().endswith("```"):
            t = t.rstrip()[:-3]
    return t.strip()


def kaggle_decide_databases():
    """Job (a): decide which additional databases Module 1 should download.
    Runs on the 17-layer reasoning service. Returns (clean_text, ids_csv)."""
    task = f"""KAGGLE AGENT — JOB (a): DATABASE DOWNLOAD DECISION.

You coordinate a research app. Decide which additional AGI/ARC databases
Module 1 (the Database Downloader) should download from this candidate catalog:

{_catalog_text()}

Environment: {RUNTIME}. Default per-download size cap: {DOWNLOADER_MAX_BYTES/1e9:.1f} GB.
The app's next goal is designing a Kaggle ARC-AGI-3 notebook, so prefer sources
that directly support ARC-AGI-3 preparation.

After the 17-layer reasoning, output ONLY a JSON object:
{{"download": ["id1", "id2"], "rationale": "one short paragraph"}}"""
    out = reason(task, mode="raw", module="kaggle_agent",
                 context="Kaggle Agent job (a): decide which additional databases the Database Downloader should download.")
    final = _clean_text_block(out.get("final") or out.get("draft") or "")
    parsed = _extract_json_object_local(final)
    if not parsed or not isinstance(parsed.get("download"), list):
        clean = (f"KAGGLE AGENT — DATABASE DOWNLOAD DECISION\n{'=' * 52}\n"
                 f"❌ No usable decision from the 17-layer service.\n"
                 f"Service status: {out.get('status', '')}\n\nRaw output:\n{final[:1500]}")
        return clean, ""
    valid_ids = [e["id"] for e in DOWNLOADER_CATALOG]
    chosen = [str(x).strip() for x in parsed["download"] if str(x).strip() in valid_ids]
    clean = ("KAGGLE AGENT — DATABASE DOWNLOAD DECISION\n" + "=" * 52 + "\n"
             "Decided by the 17-layer reasoning service "
             f"(score {out.get('score_before')} → {out.get('score_after')}).\n\n"
             "Databases to download:\n"
             + ("\n".join(f"  · {c}" for c in chosen) if chosen else "  (none)")
             + "\n\nRationale:\n" + str(parsed.get("rationale", "")).strip()
             + "\n\nHand-off: press 'Send decision to Database Downloader', then download "
               "from the ⬇️ Database Downloader tab.")
    return clean, ", ".join(chosen)


def kaggle_write_ask_prompt(log_text=""):
    """Write ONE v4 design prompt, grounded in pasted Kaggle logs.

    The output is prompt text only. It is not the design answer and it is not code.
    """
    logs = str(log_text or "").strip()
    if not logs:
        return "❌ Paste the Kaggle run logs first; no design prompt was written."
    task = f"""KAGGLE AGENT — WRITE THE V4 DESIGN PROMPT ONLY.

Write one self-contained plain-text prompt for the Ask-a-Question Kaggle-design
path. The prompt must ask for one concrete next-version design based on the
actual run evidence below. Do not write code and do not invent scores, APIs,
model names, licenses, or hardware facts. Label uncertain claims ASSUMPTION or
UNKNOWN. Treat a pending run as pending, not as a result.

Ask the design step to:
- summarize the evidence and identify the failure mode or uncertainty;
- choose one actionable direction rather than offering a menu;
- specify architecture, agent loop, offline Kaggle packaging, time/GPU needs if
  known, and safe behavior on stalls or failures;
- say what to keep/change from the uploaded baseline;
- give local tests and pass/fail gates before any scored run;
- list risks and facts still needing verification;
- return a design, not implementation code.

Competition context:
{ARC_AGI_3_CONTEXT}

PASTED KAGGLE LOGS (source evidence; preserve exact scores/statuses):
<logs>
{logs[:45000]}
</logs>

Output ONLY the final prompt text. No title, preamble, or code fences."""
    out = reason(task, mode="raw", module="kaggle_agent",
                 context="Kaggle Agent job: write a log-grounded v4 design prompt only.")
    prompt_text = _clean_text_block(out.get("final") or out.get("draft") or "")
    if not prompt_text:
        return f"❌ The 17-layer service produced no prompt. Status: {out.get('status', '')}"
    return prompt_text


def kaggle_run_ask_question(prompt_text):
    """Hand the prompt to Module 3 (Ask-a-Question, standard mode). Ask-a-Question
    reasons through the 17-layer service and returns STANDARD FORM only —
    no databases, no retrieval, no downloader calls."""
    return ask_question_kaggle(prompt_text)



def kaggle_emit_submission_agent(design_text, logs="", design_prompt="", baseline_file=None):
    """Generate a reviewable candidate .py from an uploaded .py baseline.

    The model returns exact JSON find/replace edits. Edits must each match once;
    the result is AST/compile checked and must retain the baseline's public API.
    Generated code is never executed here and is never pushed to Kaggle.
    Returns (candidate_path, manifest_path, status_text).
    """
    import ast as _ast
    import hashlib as _hashlib
    import json as _json
    import os as _os
    import time as _time
    from pathlib import Path as _Path

    design = str(design_text or "").strip()
    if not design:
        return None, None, "❌ No design answer was received; candidate not generated."

    raw_path = baseline_file
    if hasattr(raw_path, "name"):
        raw_path = raw_path.name
    if not raw_path or not isinstance(raw_path, (str, _os.PathLike)):
        return None, None, ("❌ Upload a single-file Python baseline for this legacy path. "
                            "For Franzen/Duck, use the multi-file source builder.")
    base_path = _Path(raw_path)
    try:
        baseline = base_path.read_text(encoding="utf-8")
    except Exception as exc:
        return None, None, f"❌ Could not read baseline source: {exc}"
    if not baseline.strip():
        return None, None, "❌ Baseline file is empty; candidate not generated."
    if len(baseline) > 90000:
        return None, None, ("❌ Baseline exceeds the 90,000-character first-build limit. "
                            "Use a single-file agent source, not an entire harness repository.")

    task = f"""You are the code-edit stage of the Kaggle Agent. Apply the design as the
smallest safe change to the exact baseline source below. Preserve license/provenance
comments, all existing public top-level classes/functions, and the baseline's
runtime contract. Do not add network calls, secrets, shell commands, eval, or exec.
Do not claim that tests were run. Return ONLY valid JSON in this exact shape:
{{"edits":[{{"old":"exact unique source text","new":"replacement source text"}}]}}
Use 1 to 8 edits. Every `old` snippet must occur exactly once in the baseline.
Do not repeat the full file in the response. If the design cannot be implemented
safely as a small patch, return {{"edits":[]}}.

DESIGN:
{design[:22000]}

PASTED KAGGLE LOGS:
{str(logs or '')[:18000]}

DESIGN PROMPT:
{str(design_prompt or '')[:8000]}

BASELINE FILE: {base_path.name}
BASELINE SOURCE:
<source>
{baseline}
</source>"""
    try:
        result = reason(task, mode="raw", module="kaggle_agent", budget_s=900,
                        use_cache=False,
                        context="Kaggle Agent code edit: propose exact minimal edits; never execute or deploy.")
    except Exception as exc:
        return None, None, f"❌ Code-edit reasoning failed: {exc}"

    parsed = None
    raw_outputs = [result.get("final", ""), result.get("draft", "")]
    for raw in raw_outputs:
        candidate_json = _extract_json_object_local(_clean_text_block(raw))
        if isinstance(candidate_json, dict) and isinstance(candidate_json.get("edits"), list):
            parsed = candidate_json
            break
    if parsed is None:
        return None, None, ("❌ The code-edit answer was not valid JSON edits. "
                            f"Service status: {result.get('status', '')}")
    edits = parsed.get("edits") or []
    if not (1 <= len(edits) <= 8):
        return None, None, "❌ No usable 1–8 edit patch was returned; candidate not generated."

    candidate_source = baseline
    applied = []
    for i, edit in enumerate(edits, 1):
        if not isinstance(edit, dict):
            return None, None, f"❌ Edit {i} is not an object; candidate not generated."
        old = edit.get("old")
        new = edit.get("new")
        if not isinstance(old, str) or not old or not isinstance(new, str):
            return None, None, f"❌ Edit {i} needs non-empty 'old' and string 'new' values."
        count = candidate_source.count(old)
        if count != 1:
            return None, None, f"❌ Edit {i} matched {count} places, not exactly one; candidate not generated."
        candidate_source = candidate_source.replace(old, new, 1)
        applied.append({"old": old, "new": new})
    if candidate_source == baseline:
        return None, None, "❌ Patch made no source change; candidate not generated."

    try:
        base_tree = _ast.parse(baseline)
        new_tree = _ast.parse(candidate_source)
        compile(candidate_source, "kaggle_agent_v4_candidate.py", "exec")
    except Exception as exc:
        return None, None, f"❌ Candidate failed Python syntax/compile validation: {exc}"

    def _public_top_level(tree):
        return {node.name for node in tree.body
                if isinstance(node, (_ast.ClassDef, _ast.FunctionDef, _ast.AsyncFunctionDef))
                and not node.name.startswith("_")}
    missing_api = _public_top_level(base_tree) - _public_top_level(new_tree)
    if missing_api:
        return None, None, ("❌ Candidate removes baseline public API: "
                            + ", ".join(sorted(missing_api)))

    run_id = _time.strftime("%Y%m%d_%H%M%S", _time.gmtime()) + "_" + _hashlib.sha256(
        (baseline + design).encode("utf-8")).hexdigest()[:8]
    out_dir = _Path.cwd() / "kaggle_v4_candidates" / run_id
    try:
        out_dir.mkdir(parents=True, exist_ok=True)
        candidate_path = out_dir / "kaggle_agent_v4_candidate.py"
        patch_path = out_dir / "candidate_edits.json"
        manifest_path = out_dir / "run_manifest.json"
        logs_path = out_dir / "kaggle_logs.txt"
        prompt_path = out_dir / "design_prompt.txt"
        design_path = out_dir / "design_answer.txt"
        candidate_path.write_text(candidate_source, encoding="utf-8")
        patch_path.write_text(_json.dumps({"edits": applied}, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")
        logs_path.write_text(str(logs or ""), encoding="utf-8")
        prompt_path.write_text(str(design_prompt or ""), encoding="utf-8")
        design_path.write_text(design, encoding="utf-8")
        def _sha(text):
            return _hashlib.sha256(text.encode("utf-8")).hexdigest()
        manifest = {
            "product": "Kaggle Agent v4 candidate",
            "run_id": run_id,
            "status": "candidate_only_not_submitted",
            "created_utc": _time.strftime("%Y-%m-%dT%H:%M:%SZ", _time.gmtime()),
            "baseline_file": base_path.name,
            "baseline_sha256": _sha(baseline),
            "candidate_file": candidate_path.name,
            "candidate_sha256": _sha(candidate_source),
            "design_prompt_sha256": _sha(str(design_prompt or "")),
            "design_sha256": _sha(design),
            "logs_sha256": _sha(str(logs or "")),
            "edit_count": len(applied),
            "checks": ["unique exact edits applied", "AST parse passed", "compile passed",
                       "baseline public top-level API retained"],
            "not_done": ["candidate code not executed", "game tests not run",
                         "Kaggle kernel not pushed", "no score submitted"],
            "artifacts": [candidate_path.name, patch_path.name, logs_path.name,
                          prompt_path.name, design_path.name],
        }
        manifest_path.write_text(_json.dumps(manifest, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")
    except Exception as exc:
        return None, None, f"❌ Could not write candidate files: {exc}"

    return (str(candidate_path), str(manifest_path),
            "✅ Candidate source and run manifest emitted. Syntax/API checks passed. "
            "This is not a Kaggle submission: review it and run offline/game tests before upload.")


# Franzen/Duck multi-file candidate mode. This is a patch-generation path only:
# it never runs candidate source or uploads/submits it to Kaggle.
FRANZEN_BASELINE_REPO_URL = "https://github.com/tonghuikang/daniel-franzen-arc-agi-3.git"
FRANZEN_BASELINE_COMMIT = "b2fee614056f0609b94e214f03372f15894c00fe"
FRANZEN_DEFAULT_TARGET_SPECS = """ARC3-Inference/inference/agent/python_tool_sandbox.py:150-190
ARC3-Inference/inference/agent/prompts.py:130-155
ARC3-Inference/inference/agent/tool_agent.py:297-308
ARC3-Inference/inference/agent/priority_scheduler.py:70-134
ARC3-Inference/inference/framework/solver.py:276-301
ARC3-Inference/README.md:76-103"""
FRANZEN_ALLOWED_SOURCE_SUFFIXES = {".py", ".md", ".json", ".toml", ".yaml", ".yml", ".txt", ".sh"}
FRANZEN_MAX_SOURCE_CONTEXT_CHARS = 90000
FRANZEN_MAX_FILES_PER_PATCH = 8


def _franzen_find_repo_root(search_root):
    from pathlib import Path as _Path

    search_root = _Path(search_root).resolve()
    for marker in search_root.rglob("tool_agent.py"):
        if (marker.parent.name == "agent" and marker.parent.parent.name == "inference"
                and marker.parent.parent.parent.name == "ARC3-Inference"):
            root = marker.parents[3]
            if (root / "ARC3-Inference" / "inference" / "agent" / "prompts.py").is_file():
                return root
    raise ValueError("Source archive does not contain the Franzen ARC3-Inference tree.")


def _franzen_tree_sha256(repo_root):
    import hashlib as _hashlib
    from pathlib import Path as _Path

    root = _Path(repo_root).resolve()
    h = _hashlib.sha256()
    for path in sorted(root.rglob("*")):
        if not path.is_file() or ".git" in path.parts or path.suffix.lower() not in FRANZEN_ALLOWED_SOURCE_SUFFIXES:
            continue
        rel = path.relative_to(root).as_posix()
        h.update(rel.encode("utf-8"))
        h.update(b"\0")
        with path.open("rb") as stream:
            for block in iter(lambda: stream.read(1024 * 1024), b""):
                h.update(block)
    return h.hexdigest()


def _franzen_upload_path(upload):
    import os as _os

    if upload is None:
        return ""
    if isinstance(upload, dict):
        upload = upload.get("path") or upload.get("name") or ""
    elif hasattr(upload, "name"):
        upload = upload.name
    return _os.fspath(upload) if isinstance(upload, (str, _os.PathLike)) else ""


def _franzen_safe_extract_zip(zip_path, destination):
    import os as _os
    import shutil as _shutil
    import stat as _stat
    import zipfile as _zipfile
    from pathlib import Path as _Path, PurePosixPath as _PurePosixPath

    destination = _Path(destination).resolve()
    destination.mkdir(parents=True, exist_ok=True)
    with _zipfile.ZipFile(zip_path) as archive:
        entries = archive.infolist()
        if len(entries) > 20000:
            raise ValueError("Source ZIP has too many entries (limit 20,000).")
        if sum(info.file_size for info in entries) > 250_000_000:
            raise ValueError("Source ZIP expands beyond the 250 MB safety limit.")
        for info in entries:
            name = info.filename.replace("\\", "/")
            rel = _PurePosixPath(name)
            if rel.is_absolute() or any(part == ".." for part in rel.parts):
                raise ValueError("Source ZIP contains an unsafe path.")
            mode = (info.external_attr >> 16) & 0o170000
            if mode == _stat.S_IFLNK:
                raise ValueError("Source ZIP contains a symbolic link; refusing to extract it.")
            target = destination.joinpath(*[part for part in rel.parts if part not in ("", ".")])
            resolved = target.resolve()
            try:
                resolved.relative_to(destination)
            except ValueError:
                raise ValueError("Source ZIP path escapes the extraction directory.")
            if info.is_dir():
                target.mkdir(parents=True, exist_ok=True)
                continue
            target.parent.mkdir(parents=True, exist_ok=True)
            with archive.open(info, "r") as src, target.open("wb") as dst:
                _shutil.copyfileobj(src, dst)


def kaggle_prepare_franzen_baseline(source_zip=None, commit=FRANZEN_BASELINE_COMMIT):
    """Load the pinned Franzen source or an exact uploaded repo ZIP.

    Returns (state_dict, status_text, compact_file_list). ZIP upload wins over
    network clone and is useful when the notebook runtime has no internet.
    """
    import hashlib as _hashlib
    import os as _os
    import re as _re
    import shutil as _shutil
    import subprocess as _subprocess
    import tempfile as _tempfile
    from pathlib import Path as _Path

    try:
        cache_root = _Path(DATA_ROOT) / "4cbon2_franzen_baselines"
        cache_root.mkdir(parents=True, exist_ok=True)
        upload_path = _franzen_upload_path(source_zip)
        if upload_path:
            up = _Path(upload_path)
            if not up.is_file() or up.suffix.lower() != ".zip":
                raise ValueError("Upload a ZIP of the Franzen source repository.")
            h = _hashlib.sha256()
            with up.open("rb") as stream:
                for block in iter(lambda: stream.read(1024 * 1024), b""):
                    h.update(block)
            archive_sha = h.hexdigest()
            repo_root = cache_root / ("upload-" + archive_sha[:16])
            if not (repo_root / "ARC3-Inference" / "inference" / "agent" / "tool_agent.py").is_file():
                stage = _Path(_tempfile.mkdtemp(prefix="franzen_zip_", dir=str(cache_root)))
                try:
                    unpacked = stage / "unpacked"
                    _franzen_safe_extract_zip(up, unpacked)
                    found = _franzen_find_repo_root(unpacked)
                    if repo_root.exists():
                        _shutil.rmtree(repo_root)
                    _shutil.copytree(found, repo_root,
                                     ignore=_shutil.ignore_patterns(".git", "__pycache__", "*.pyc"))
                finally:
                    _shutil.rmtree(stage, ignore_errors=True)
            repo_root = _franzen_find_repo_root(repo_root)
            state = {
                "root": str(repo_root), "origin": "uploaded_source_zip",
                "commit": "", "source_archive_sha256": archive_sha,
                "tree_sha256": _franzen_tree_sha256(repo_root),
            }
            label = f"uploaded ZIP sha256 {archive_sha[:12]}"
        else:
            commit = str(commit or "").strip().lower()
            if not _re.fullmatch(r"[0-9a-f]{40}", commit):
                raise ValueError("Use a full 40-character Git commit SHA, or upload a source ZIP.")
            repo_root = cache_root / ("repo-" + commit[:16])
            expected = repo_root / "ARC3-Inference" / "inference" / "agent" / "tool_agent.py"
            head = ""
            if expected.is_file() and (repo_root / ".git").is_dir():
                check = _subprocess.run(["git", "-C", str(repo_root), "rev-parse", "HEAD"],
                                        capture_output=True, text=True, timeout=30)
                head = (check.stdout or "").strip().lower() if check.returncode == 0 else ""
            if head != commit:
                if repo_root.exists():
                    _shutil.rmtree(repo_root)
                repo_root.mkdir(parents=True, exist_ok=True)
                commands = [
                    ["git", "init", "--quiet", str(repo_root)],
                    ["git", "-C", str(repo_root), "remote", "add", "origin", FRANZEN_BASELINE_REPO_URL],
                    ["git", "-C", str(repo_root), "fetch", "--depth=1", "origin", commit],
                    ["git", "-C", str(repo_root), "checkout", "--quiet", "--detach", "FETCH_HEAD"],
                ]
                for command in commands:
                    result = _subprocess.run(command, capture_output=True, text=True, timeout=300)
                    if result.returncode != 0:
                        detail = (result.stderr or result.stdout or "git failed")[-500:]
                        raise RuntimeError("Could not fetch pinned Franzen source. "
                                           "Upload a source ZIP if this runtime is offline. " + detail)
                check = _subprocess.run(["git", "-C", str(repo_root), "rev-parse", "HEAD"],
                                        capture_output=True, text=True, timeout=30)
                head = (check.stdout or "").strip().lower()
                if head != commit:
                    raise RuntimeError("Fetched source commit does not match the requested pin.")
            repo_root = _franzen_find_repo_root(repo_root)
            state = {
                "root": str(repo_root), "origin": FRANZEN_BASELINE_REPO_URL,
                "commit": commit, "source_archive_sha256": "",
                "tree_sha256": _franzen_tree_sha256(repo_root),
            }
            label = "commit " + commit

        py_files = sorted((repo_root / "ARC3-Inference").rglob("*.py"))
        rels = [p.relative_to(repo_root).as_posix() for p in py_files]
        preview = "\n".join(rels[:60])
        if len(rels) > 60:
            preview += f"\n… plus {len(rels) - 60} more Python files"
        return state, (f"✅ Franzen multi-file baseline loaded ({label}); "
                       f"tree sha256 {state['tree_sha256'][:12]}. This is source preparation only."), preview
    except Exception as exc:
        return {}, f"❌ Franzen baseline not loaded: {exc}", ""


def _franzen_read_target_context(repo_root, target_specs):
    import re as _re
    from pathlib import Path as _Path, PurePosixPath as _PurePosixPath

    specs = [line.strip() for line in str(target_specs or "").splitlines() if line.strip()]
    if not specs:
        specs = [line.strip() for line in FRANZEN_DEFAULT_TARGET_SPECS.splitlines() if line.strip()]
    if not (1 <= len(specs) <= FRANZEN_MAX_FILES_PER_PATCH):
        raise ValueError(f"Choose 1–{FRANZEN_MAX_FILES_PER_PATCH} target files/ranges.")
    root = _Path(repo_root).resolve()
    selected = {}
    contexts = []
    total_chars = 0
    for spec in specs:
        match = _re.match(r"^(.*?)(?::([0-9]+)-([0-9]+))?$", spec)
        rel_text = (match.group(1) if match else spec).strip()
        start = int(match.group(2)) if match and match.group(2) else None
        end = int(match.group(3)) if match and match.group(3) else None
        rel = _PurePosixPath(rel_text.replace("\\", "/"))
        if rel.is_absolute() or any(part in ("", ".", "..") for part in rel.parts):
            raise ValueError(f"Unsafe target path: {rel_text}")
        if rel.suffix.lower() not in FRANZEN_ALLOWED_SOURCE_SUFFIXES:
            raise ValueError(f"Target is not an allowed text-source file: {rel_text}")
        path = (root.joinpath(*rel.parts)).resolve()
        try:
            path.relative_to(root)
        except ValueError:
            raise ValueError(f"Target escapes the baseline root: {rel_text}")
        if not path.is_file():
            raise ValueError(f"Target file not found in baseline: {rel_text}")
        key = rel.as_posix()
        if key in selected:
            raise ValueError(f"Target appears more than once: {key}")
        text = path.read_text(encoding="utf-8")
        lines = text.splitlines(keepends=True)
        if start is None:
            if len(text) > 60000:
                raise ValueError(f"{key} is {len(text):,} chars; give a focused line range, e.g. {key}:20-80")
            start, end = 1, max(1, len(lines))
            excerpt = text
        else:
            if end is None or start < 1 or end < start or end > len(lines):
                raise ValueError(f"Invalid line range for {key}; file has {len(lines)} lines.")
            excerpt = "".join(lines[start - 1:end])
        if not excerpt:
            raise ValueError(f"Selected context is empty: {key}")
        total_chars += len(excerpt)
        if total_chars > FRANZEN_MAX_SOURCE_CONTEXT_CHARS:
            raise ValueError("Selected source exceeds the 90,000-character context cap; choose fewer/smaller ranges.")
        selected[key] = {
            "path": path, "original": text, "excerpt": excerpt,
            "start_line": start, "end_line": end,
        }
        contexts.append(f"=== {key} (editable context lines {start}-{end}) ===\n{excerpt}")
    return selected, "\n\n".join(contexts), specs


def kaggle_emit_franzen_patch(design_text, logs="", design_prompt="", baseline_state=None,
                              target_specs=FRANZEN_DEFAULT_TARGET_SPECS):
    """Emit a reviewable multi-file diff + changed-file ZIP from a Franzen source tree.

    Only existing, explicitly selected text files/ranges can be edited. Python
    syntax and JSON parsing are checked, but candidate code is never executed,
    game-tested, uploaded, or submitted.
    Returns (status, patch_path, changed_files_zip, manifest_path).
    """
    import ast as _ast
    import difflib as _difflib
    import hashlib as _hashlib
    import json as _json
    import os as _os
    import re as _re
    import time as _time
    import zipfile as _zipfile
    from pathlib import Path as _Path

    design = str(design_text or "").strip()
    state = baseline_state if isinstance(baseline_state, dict) else {}
    root = str(state.get("root") or "").strip()
    if not design:
        return "❌ Run the Kaggle design path first; no patch generated.", None, None, None
    if not root or not _Path(root).is_dir():
        return "❌ Load the pinned Franzen baseline or upload its source ZIP first.", None, None, None
    try:
        selected, source_context, specs = _franzen_read_target_context(root, target_specs)
    except Exception as exc:
        return f"❌ Target selection failed: {exc}", None, None, None

    task = f"""You are the multi-file code-edit stage of the Kaggle Agent. Implement the
provided design as the smallest safe patch against the exact Franzen/Duck source
excerpts below. Preserve licensing/provenance and existing public behavior unless
the design specifically requires a change. Do not change model weights, quantization,
network requirements, or competition submission behavior. Do not claim tests ran.
The tool emits a reviewable candidate only; it is never executed or submitted.

Return ONLY valid JSON:
{{"edits":[{{"path":"exact repo-relative path","old":"exact unique text from that file excerpt","new":"replacement text"}}]}}
Use 1–12 edits. Only edit the listed existing files and the shown line ranges; do not
add/delete files. Each `old` must occur exactly once in the full file and be contained
in its supplied excerpt. If safe implementation is not possible, return
{{"edits":[]}}. Do not return a full file or a unified diff.

DESIGN:
{design[:18000]}

PASTED LOGS (evidence only; preserve reported scores/statuses):
{str(logs or '')[:10000]}

DESIGN PROMPT:
{str(design_prompt or '')[:6000]}

BASELINE IDENTITY: {state.get('origin', '')} @ {state.get('commit') or state.get('source_archive_sha256', '')}
BASELINE TREE SHA256: {state.get('tree_sha256', '')}
TARGETS: {_json.dumps(specs, ensure_ascii=False)}
SOURCE EXCERPTS:
{source_context}
"""
    try:
        result = reason(task, mode="raw", module="kaggle_agent", budget_s=900,
                        use_cache=False,
                        context="Kaggle Agent multi-file edit: exact Franzen-source patch only; never execute, upload, or score.")
    except Exception as exc:
        return f"❌ Multi-file edit reasoning failed: {exc}", None, None, None

    parsed = None
    for raw in (result.get("final", ""), result.get("draft", "")):
        item = _extract_json_object_local(_clean_text_block(raw))
        if isinstance(item, dict) and isinstance(item.get("edits"), list):
            parsed = item
            break
    if parsed is None:
        return ("❌ The code-edit response was not valid JSON edits. "
                f"Service status: {result.get('status', '')}"), None, None, None
    edits = parsed.get("edits") or []
    if not (1 <= len(edits) <= 12):
        return "❌ No usable 1–12 edit patch was returned.", None, None, None

    candidates = {key: data["original"] for key, data in selected.items()}
    applied = []
    for i, edit in enumerate(edits, 1):
        if not isinstance(edit, dict):
            return f"❌ Edit {i} is not an object; nothing emitted.", None, None, None
        rel = str(edit.get("path") or "").replace("\\", "/").strip()
        old, new = edit.get("old"), edit.get("new")
        if rel not in selected:
            return f"❌ Edit {i} targets an unselected file ({rel}); nothing emitted.", None, None, None
        if not isinstance(old, str) or not old or not isinstance(new, str):
            return f"❌ Edit {i} needs non-empty 'old' and string 'new' values.", None, None, None
        if old not in selected[rel]["excerpt"]:
            return f"❌ Edit {i} is outside the selected line range for {rel}.", None, None, None
        count = candidates[rel].count(old)
        if count != 1:
            return f"❌ Edit {i} matched {count} places in {rel}, not exactly one; nothing emitted.", None, None, None
        candidates[rel] = candidates[rel].replace(old, new, 1)
        applied.append({"path": rel, "old": old, "new": new})

    changed = {rel: text for rel, text in candidates.items() if text != selected[rel]["original"]}
    if not changed:
        return "❌ The proposed edits made no source change; nothing emitted.", None, None, None
    try:
        for rel, text in changed.items():
            suffix = _Path(rel).suffix.lower()
            if suffix == ".py":
                _ast.parse(text, filename=rel)
                compile(text, rel, "exec")
            elif suffix == ".json":
                _json.loads(text)
    except Exception as exc:
        return f"❌ Candidate syntax/JSON validation failed: {exc}", None, None, None

    diff_parts = []
    changed_file_rows = []
    for rel in sorted(changed):
        before = selected[rel]["original"]
        after = changed[rel]
        diff_parts.append(f"diff --git a/{rel} b/{rel}\n")
        diff_parts.extend(_difflib.unified_diff(
            before.splitlines(keepends=True), after.splitlines(keepends=True),
            fromfile=f"a/{rel}", tofile=f"b/{rel}", n=3,
        ))
        changed_file_rows.append({
            "path": rel,
            "baseline_sha256": _hashlib.sha256(before.encode("utf-8")).hexdigest(),
            "candidate_sha256": _hashlib.sha256(after.encode("utf-8")).hexdigest(),
        })
    patch_text = "".join(diff_parts)
    run_id = _time.strftime("%Y%m%d_%H%M%S", _time.gmtime()) + "_" + _hashlib.sha256(
        (str(state.get("tree_sha256", "")) + design + patch_text).encode("utf-8")).hexdigest()[:8]
    out_dir = _Path.cwd() / "kaggle_v4_candidates" / "franzen" / run_id
    try:
        out_dir.mkdir(parents=True, exist_ok=True)
        patch_path = out_dir / "candidate_franzen_patch.diff"
        bundle_path = out_dir / "candidate_changed_files.zip"
        manifest_path = out_dir / "candidate_manifest.json"
        patch_path.write_text(patch_text, encoding="utf-8")
        manifest = {
            "product": "Kaggle Agent Franzen/Duck multi-file candidate",
            "run_id": run_id,
            "status": "candidate_only_not_submitted",
            "created_utc": _time.strftime("%Y-%m-%dT%H:%M:%SZ", _time.gmtime()),
            "baseline": {
                "origin": state.get("origin", ""),
                "commit": state.get("commit", ""),
                "uploaded_archive_sha256": state.get("source_archive_sha256", ""),
                "tree_sha256": state.get("tree_sha256", ""),
            },
            "targets": specs,
            "changed_files": changed_file_rows,
            "edit_count": len(applied),
            "checks": ["unique exact edits applied to selected existing files/ranges",
                       "Python AST/compile and JSON checks passed where applicable"],
            "not_done": ["candidate code not executed", "game tests not run",
                         "Kaggle notebook not updated automatically", "nothing uploaded or submitted", "no score measured"],
            "design_sha256": _hashlib.sha256(design.encode("utf-8")).hexdigest(),
            "logs_sha256": _hashlib.sha256(str(logs or "").encode("utf-8")).hexdigest(),
            "design_prompt_sha256": _hashlib.sha256(str(design_prompt or "").encode("utf-8")).hexdigest(),
            "artifacts": [patch_path.name, bundle_path.name, manifest_path.name],
        }
        manifest_text = _json.dumps(manifest, indent=2, ensure_ascii=False) + "\n"
        manifest_path.write_text(manifest_text, encoding="utf-8")
        with _zipfile.ZipFile(bundle_path, "w", compression=_zipfile.ZIP_DEFLATED) as archive:
            for rel, text in sorted(changed.items()):
                archive.writestr(rel, text.encode("utf-8"))
            archive.writestr("APPLY_NOTES.txt",
                             "Changed source files only. Apply over the exact baseline commit/archive listed in candidate_manifest.json. "
                             "Review the .diff and run offline tests before any competition upload.\n")
    except Exception as exc:
        return f"❌ Could not write multi-file candidate artifacts: {exc}", None, None, None

    names = ", ".join(sorted(changed))
    status = (f"✅ Multi-file candidate emitted: {names}. Exact-edit and applicable Python/JSON checks passed. "
              "Candidate only; not executed, game-tested, uploaded, or scored.")
    return status, str(patch_path), str(bundle_path), str(manifest_path)


print("✅ Kaggle Agent ready. Jobs: database decisions; log-grounded design prompts; "
      "Franzen multi-file patch + legacy single-file candidate emission. "
      "Candidates are never executed, uploaded, or scored here.")

✅ Kaggle Agent ready. Jobs: database decisions; log-grounded design prompts; Franzen multi-file patch + legacy single-file candidate emission. Candidates are never executed, uploaded, or scored here.


In [ ]:
# ============================================================
# CELL 11 — Gradio UI
# Tabs: Upload Documents · Ask a Question (General + Kaggle paths) · Agent Mode (Module 5) ·
# Builder (Module 4) · AI Rewriter (Module 2) · Database Downloader (Module 1) ·
# Kaggle Agent (coordinator) · Data Dashboard · Agent Status
# ============================================================
import gradio as gr
import traceback
import ast
import shutil
from pathlib import Path

try:
    gr.close_all()
except:
    pass

# --- Import google.colab.ai for Builder tab ---
try:
    from google.colab import ai as colab_ai
    COLAB_AI_AVAILABLE = True
    print("✅ google.colab.ai imported successfully")
except ImportError:
    COLAB_AI_AVAILABLE = False
    print("⚠️ google.colab.ai not available - Builder tab will use fallback Gemini client")

# --- Safety fallback for _parse_agent_json ---
try:
    _parse_agent_json
except NameError:
    import json as _json
    def _extract_balanced_fb(text, open_ch, close_ch):
        if not text:
            return None
        start = text.find(open_ch)
        if start == -1:
            return None
        depth = 0
        in_string = False
        escape = False
        for i in range(start, len(text)):
            ch = text[i]
            if in_string:
                if escape:
                    escape = False
                elif ch == '\\':
                    escape = True
                elif ch == '"':
                    in_string = False
            else:
                if ch == '"':
                    in_string = True
                elif ch == open_ch:
                    depth += 1
                elif ch == close_ch:
                    depth -= 1
                    if depth == 0:
                        return text[start:i + 1]
        return None

    def _parse_agent_json(raw_response):
        if not raw_response:
            return None
        candidate = _extract_balanced_fb(raw_response, '{', '}')
        try:
            if candidate:
                return _json.loads(candidate)
            return _json.loads(raw_response)
        except Exception:
            return None
    print("ℹ️ _parse_agent_json defined locally (safety fallback).")


# ============================================================
# BUILDER TAB FUNCTIONS
# ============================================================

def read_notebook_cells(notebook_path):
    """Read all code cells from the notebook."""
    try:
        with open(notebook_path, 'r') as f:
            nb = json.load(f)
        cells = []
        for i, cell in enumerate(nb.get('cells', [])):
            if cell.get('cell_type') == 'code':
                source = ''.join(cell.get('source', []))
                cells.append({
                    'index': i,
                    'source': source,
                    'length': len(source)
                })
        return cells, None
    except Exception as e:
        return None, str(e)


def generate_builder_proposal(notebook_path, direction, include_ask_output=False, ask_context=""):
    """Generate a proposal through the shared 17-layer reasoning service
    (module 'agent_builder'); direct call only on budget fallback."""
    if not direction or not direction.strip():
        return "❌ Please enter a direction.", "", "❌ No direction"

    # Optional downstream hand-off: consume the STANDARD FORM output of
    # Ask-a-Question (Module 3) and build artifacts from it, as before.
    if include_ask_output and (ask_context or "").strip():
        direction = (direction or "").strip() + (
            "\n\n[ASK-A-QUESTION STANDARD FORM OUTPUT — build from this]\n" + ask_context.strip()
        )

    # Read notebook
    cells, error = read_notebook_cells(notebook_path)
    if error:
        return f"❌ Failed to read notebook: {error}", "", "❌ Read error"

    # Build context
    notebook_context = ""
    for cell in cells:
        notebook_context += f"\n--- CELL {cell['index']} ({cell['length']} chars) ---\n"
        notebook_context += f"```python\n{cell['source']}\n```\n"

    prompt = f"""You are the 4CBON2 Builder. Analyze the entire notebook and propose changes based on the user's direction.\n\nUSER DIRECTION:\n{direction}\n\nFULL NOTEBOOK CONTENT:\n{notebook_context}\n\nGenerate a JSON proposal with this exact structure:\n{{\n    "proposal_id": "prop_YYYYMMDD_HHMMSS",\n    "direction": "repeat the user direction",\n    "summary": "Brief summary of proposed changes",\n    "changes": [\n        {{\n            "cell_index": 0,\n            "section": "Section name",\n            "action": "modify|add|replace",\n            "original_code": "existing code (or null if adding new)",\n            "new_code": "the complete new or modified code",\n            "rationale": "why this change is needed"\n        }}\n    ],\n    "instructions": "Step-by-step instructions for applying changes"\n}}\n\nRules:\n- new_code must be valid, runnable Python\n- Include COMPLETE code for each cell (no truncation, no "...")\n- Only output valid JSON. No markdown, no explanations outside the JSON.\n\nJSON:"""

    # Design/build decision — routed through the shared 17-layer reasoning
    # service (module 'agent_builder'); falls back to the pre-rebuild direct
    # call only when the budget is exceeded (REASONING_CONFIG).
    try:
        result = builder_decision_llm(prompt, max_tokens=4096, step="proposal")

        if not result or result.startswith("⚠️") or result.startswith('{"error"'):
            return f"❌ Proposal generation failed: {result}", "", "❌ Failed"

        parsed = _parse_agent_json(result)
        if parsed is None:
            return (
                f"❌ Failed to parse proposal JSON.\n\n"
                f"Raw response (first 1000 chars):\n{result[:1000]}...",
                result,
                "❌ JSON Error"
            )

        if "proposal_id" not in parsed:
            parsed["proposal_id"] = f"prop_{datetime.now().strftime('%Y%m%d_%H%M%S')}"

        # Build summary
        summary_md = f"## 📋 Proposal: {parsed.get('proposal_id', 'N/A')}\n\n"
        summary_md += f"**Direction:** {parsed.get('direction', 'N/A')}\n\n"
        summary_md += f"**Summary:** {parsed.get('summary', 'No summary')}\n\n"
        summary_md += f"### Changes ({len(parsed.get('changes', []))})\n\n"

        for i, ch in enumerate(parsed.get('changes', []), 1):
            summary_md += f"**Change {i}:**\n"
            summary_md += f"- **Cell:** {ch.get('cell_index', 'N/A')}\n"
            summary_md += f"- **Section:** {ch.get('section', 'Unknown')}\n"
            summary_md += f"- **Action:** {ch.get('action', 'modify')}\n"
            summary_md += f"- **Rationale:** {ch.get('rationale', 'Not specified')}\n\n"

        summary_md += f"\n### Instructions\n{parsed.get('instructions', 'No instructions')}"

        status = f"✅ {len(parsed.get('changes', []))} change(s) proposed."
        return summary_md, json.dumps(parsed, indent=2), status

    except Exception as e:
        tb = traceback.format_exc()
        return f"❌ Unexpected error: {str(e)}\n\n{tb}", "", "❌ Failed"


def validate_syntax(code):
    """Validate Python syntax using ast.parse."""
    try:
        ast.parse(code)
        return True, None
    except SyntaxError as e:
        return False, f"Line {e.lineno}: {e.msg}"


def five_lens_verdict(change):
    """Run 5-lens automated verdict on a proposed change."""
    prompt = f"""Evaluate this proposed code change using 5 lenses:\n\nCELL INDEX: {change.get('cell_index', 'N/A')}\nACTION: {change.get('action', 'modify')}\nRATIONALE: {change.get('rationale', 'N/A')}\n\nNEW CODE:\n```python\n{change.get('new_code', '')[:2000]}\n```\n\nEvaluate using these 5 lenses:\n1. CORRECTNESS: Is the code syntactically and logically correct?\n2. SAFETY: Does it introduce security risks or dangerous operations?\n3. COMPLETENESS: Does it fully implement the intended change?\n4. COMPATIBILITY: Will it work with the rest of the notebook?\n5. CLARITY: Is the code clear and well-structured?\n\nRespond with ONLY this JSON:\n{{\n    "verdict": "APPROVE" or "REJECT",\n    "confidence": 0.0-1.0,\n    "reasoning": "Brief explanation"\n}}\n\nJSON:"""

    try:
        result = builder_decision_llm(prompt, max_tokens=512, step="five_lens_verdict")
        parsed = _parse_agent_json(result)
        if parsed and "verdict" in parsed:
            return parsed
        return {"verdict": "REJECT", "confidence": 0.0, "reasoning": "Failed to parse verdict"}
    except Exception as e:
        return {"verdict": "REJECT", "confidence": 0.0, "reasoning": f"Error: {str(e)}"}


def apply_proposals(proposals_json, notebook_path):
    """Review and apply approved changes with safety checks."""
    log = []

    try:
        proposals = json.loads(proposals_json)
    except Exception as e:
        return f"❌ Failed to parse proposals JSON: {str(e)}"

    changes = proposals.get('changes', [])
    if not changes:
        return "❌ No changes to apply."

    # Create backup
    backup_dir = Path(DATA_ROOT) / "4cbon_notebook_backups"
    backup_dir.mkdir(parents=True, exist_ok=True)
    timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
    backup_path = backup_dir / f"4CBOn2_Gemini2_backup_{timestamp}.ipynb"

    try:
        shutil.copy(notebook_path, backup_path)
        log.append(f"✅ Backup created: {backup_path}")
    except Exception as e:
        log.append(f"⚠️ Backup failed: {str(e)}")

    # Read notebook
    try:
        with open(notebook_path, 'r') as f:
            nb = json.load(f)
    except Exception as e:
        return f"❌ Failed to read notebook: {str(e)}\n\n" + "\n".join(log)

    # Process each change
    applied = 0
    skipped = 0

    for i, change in enumerate(changes, 1):
        cell_idx = change.get('cell_index')
        new_code = change.get('new_code', '')

        log.append(f"\n--- Change {i}: Cell {cell_idx} ---")

        # Validate syntax
        syntax_ok, syntax_error = validate_syntax(new_code)
        if not syntax_ok:
            log.append(f"❌ REJECTED - Syntax error: {syntax_error}")
            skipped += 1
            continue

        log.append("✅ Syntax validation passed")

        # Run 5-lens verdict
        log.append("🔍 Running 5-lens verdict...")
        verdict = five_lens_verdict(change)

        if verdict.get('verdict') == 'APPROVE':
            log.append(f"✅ APPROVED (confidence: {verdict.get('confidence', 0):.2f})")
            log.append(f"   Reasoning: {verdict.get('reasoning', 'N/A')}")

            # Apply change
            try:
                if cell_idx < len(nb['cells']):
                    nb['cells'][cell_idx]['source'] = [line + '\n' for line in new_code.split('\n')[:-1]] + [new_code.split('\n')[-1]]
                    log.append(f"✅ Applied to Cell {cell_idx}")
                    applied += 1
                else:
                    log.append(f"❌ REJECTED - Cell index {cell_idx} out of range")
                    skipped += 1
            except Exception as e:
                log.append(f"❌ REJECTED - Apply error: {str(e)}")
                skipped += 1
        else:
            log.append(f"❌ REJECTED (confidence: {verdict.get('confidence', 0):.2f})")
            log.append(f"   Reasoning: {verdict.get('reasoning', 'N/A')}")
            skipped += 1

    # Save notebook if any changes were applied
    if applied > 0:
        try:
            with open(notebook_path, 'w') as f:
                json.dump(nb, f, indent=1)
            log.append(f"\n✅ Notebook saved with {applied} change(s)")
        except Exception as e:
            log.append(f"\n❌ Failed to save notebook: {str(e)}")

    # Summary
    log.append(f"\n{'='*50}")
    log.append(f"SUMMARY: {applied} applied, {skipped} skipped")
    log.append(f"{'='*50}")

    return "\n".join(log)


# ============================================================
# OLD BUILDER FUNCTIONS (kept for compatibility)
# ============================================================

def summarise_single_cell(code):
    if not code or not code.strip():
        return "⚠️ No code provided.", "❌ Empty"
    try:
        prompt = f"""Summarise this Python cell in 2-3 sentences, focusing on its purpose and key components:\n\n```python\n{code}\n```\n\nSummary:"""
        result = safe_ask_raw(prompt, max_tokens=256)
        if not result or result.startswith("⚠️") or result.startswith('{"error"'):
            return f"⚠️ Error: {result}", "❌ Failed"
        return result.strip(), "✅ Done"
    except Exception as e:
        return f"⚠️ Exception: {str(e)}", "❌ Error"


def generate_agent_proposal(request, *cell_data):
    try:
        if not request or not request.strip():
            return "❌ Please enter what you want to build.", "", "❌ No instruction"
        cells = []
        for i in range(0, len(cell_data), 2):
            if i+1 < len(cell_data):
                code = cell_data[i] or ""
                summary = cell_data[i+1] or ""
                if code.strip():
                    cells.append({"index": (i//2)+1, "code": code, "summary": summary})
        if not cells:
            return "❌ Please paste at least one cell's code.", "", "❌ No cells"
        notebook_context = ""
        for c in cells:
            notebook_context += f"\n--- CELL {c['index']} ---\nSUMMARY: {c['summary'] or '(no summary)'}\nCODE:\n```python\n{c['code'][:800]}{'...' if len(c['code']) > 800 else ''}\n```\n"
        prompt = f"""You are the 4CBON2 Agent Builder. The user wants to modify or extend their notebook.\n\nUSER REQUEST:\n{request}\n\nCURRENT NOTEBOOK CELLS:\n{notebook_context}\n\nGenerate a JSON proposal with this exact structure:\n{{\n    "proposal_id": "prop_YYYYMMDD_HHMMSS",\n    "request": "repeat the user request",\n    "summary": "Brief summary of changes",\n    "changes": [\n        {{\n            "cell_index": 1,\n            "section": "Agent Profiles",\n            "action": "add_agent",\n            "original_code": "the existing code being modified (or null)",\n            "new_code": "the complete new or modified code",\n            "location": "Cell 4, AGENT_PROFILES dict"\n        }}\n    ],\n    "instructions": "Step-by-step instructions for applying changes"\n}}\n\nRules:\n- If adding a new agent, include the complete AGENT_PROFILES entry.\n- If modifying existing code, show both original and new code.\n- new_code must be valid, runnable Python.\n- Only output valid JSON. No markdown, no explanations outside the JSON.\n\nJSON:"""
        result = safe_ask_raw(prompt, max_tokens=4096)
        if not result or result.startswith("⚠️") or result.startswith('{"error"'):
            return f"❌ Proposal generation failed: {result}", "", "❌ Failed"

        parsed = None
        try:
            parsed = _parse_agent_json(result)
        except Exception as parse_err:
            return (
                f"❌ JSON parsing error: {parse_err}\n\n"
                f"Raw response (first 800 chars):\n{result[:800]}...",
                "",
                "❌ JSON Parse Error"
            )

        if parsed is None:
            cleaned = result.strip()
            if cleaned.startswith("```"):
                cleaned = cleaned.split("\n", 1)[-1]
            if cleaned.endswith("```"):
                cleaned = cleaned.rsplit("```", 1)[0]
            try:
                parsed = json.loads(cleaned)
            except Exception:
                return (
                    f"❌ Failed to parse proposal JSON.\n\n"
                    f"The LLM response could not be converted to valid JSON.\n"
                    f"Try rephrasing your request or pasting simpler cell code.\n\n"
                    f"Raw response (first 800 chars):\n{result[:800]}...",
                    "",
                    "❌ JSON Error"
                )

        proposal = parsed
        if "proposal_id" not in proposal:
            proposal["proposal_id"] = f"prop_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
        proposal_md = f"""## 📋 Proposal: {proposal.get('proposal_id', 'N/A')}"""
        proposal_md += f"\n\n**Request:** {proposal.get('request', 'N/A')}  "
        proposal_md += f"\n**Summary:** {proposal.get('summary', 'No summary')}"
        proposal_md += f"\n\n### Changes ({len(proposal.get('changes', []))})\n"
        code_md = "## 📝 New / Modified Code\n\n"
        for i, ch in enumerate(proposal.get('changes', []), 1):
            proposal_md += f"""\n**Change {i}:**\n- **Cell:** `Cell {ch.get('cell_index', 0)+1}`\n- **Section:** `{ch.get('section', 'Unknown')}`\n- **Action:** `{ch.get('action', 'modify')}`\n- **Location:** `{ch.get('location', 'Not specified')}`\n"""
            code_md += f"""### Cell {ch.get('cell_index', 0)+1}: {ch.get('section', '')} ({ch.get('action', '')})\n\n```python\n{ch.get('new_code', '# No code provided')}\n```\n\n**Instructions:** {ch.get('instructions', f"Replace code in {ch.get('location', 'specified location')}")}\n\n---\n"""
        proposal_md += f"\n### Instructions\n{proposal.get('instructions', 'No instructions')}"
        status = f"✅ {len(proposal.get('changes', []))} change(s) proposed."
        return proposal_md, code_md, status
    except Exception as e:
        tb = traceback.format_exc()
        return f"❌ Unexpected error: {str(e)}\n\n{tb}", "", "❌ Failed"


def run_agent(goal, use_gemini_only, enable_additional, *api_keys):
    """Run the orchestrator with optional API key injection controlled by checkboxes."""
    # Gemini client is available from Cell 1 (OAuth 2.0 authenticated)
    yield f"✅ Using Gemini ({MODEL_NAME}) — authenticated with OAuth 2.0.\n\n"

    # Checkbox logic for additional APIs
    if use_gemini_only:
        yield "🔒 **Gemini Only mode** — all additional API keys ignored.\n\n"
    elif enable_additional:
        yield "🔓 **Additional APIs enabled** — injecting optional API keys.\n\n"
        key_names = ["CALENDAR_API_KEY", "CRM_API_KEY", "COMM_API_KEY", "VISION_API_KEY",
                     "DOCUSIGN_API_KEY", "SOCIAL_SCRAPER_API_KEY", "SEO_API_KEY", "S3_VAULT_KEY", "PUBMED_API_KEY"]
        for name, val in zip(key_names, api_keys):
            if val and val.strip():
                os.environ[name] = val.strip()
    else:
        yield "🔒 **Additional APIs disabled** — using Gemini only.\n\n"

    try:
        for chunk in run_orchestrator_stream(goal):
            yield chunk
    except Exception as e:
        yield f"❌ Orchestrator error: {str(e)}"
    finally:
        key_names = ["CALENDAR_API_KEY", "CRM_API_KEY", "COMM_API_KEY", "VISION_API_KEY",
                     "DOCUSIGN_API_KEY", "SOCIAL_SCRAPER_API_KEY", "SEO_API_KEY", "S3_VAULT_KEY", "PUBMED_API_KEY"]
        for name in key_names:
            os.environ.pop(name, None)


with gr.Blocks(title="4CBON2 — Kaggle4") as demo:
    gr.Markdown("# 🚀 4CBON2 — Kaggle4 Agent Design Workbench")
    gr.Markdown("*Shared 17-layer reasoning service · general research plus a guarded Kaggle design-to-candidate workflow*")
    gr.Markdown(
        f"**Runtime:** {RUNTIME} · **Reasoning:** one shared 17-layer service (`reason()`) · "
        "**Config:** `REASONING_CONFIG` (budgets · fallback · caching)"
    )

    # Shared UI state — Ask-a-Question STANDARD FORM → Agent Builder (optional downstream)
    builder_ask_context = gr.State("")

    with gr.Tabs():
        # ── Upload Tab ──
        with gr.TabItem("📁 Upload Documents"):
            gr.Markdown("Upload .txt, .pdf, or .docx files to the knowledge base.")
            file_input = gr.File(label="Upload file", file_types=[".txt", ".pdf", ".docx"])
            upload_output = gr.Textbox(label="Status", interactive=False)
            upload_btn = gr.Button("Process & Index", variant="primary")
            upload_btn.click(fn=process_document, inputs=[file_input], outputs=[upload_output])

        # ── Ask-a-Question split: general answers vs Kaggle design loop ──
        with gr.TabItem("❓ Ask a Question"):
            with gr.Tabs():
                with gr.TabItem("General questions"):
                    gr.Markdown(
                        "### Frontier Research Question\n"
                        "Ask an ambitious AI, mathematics, or science question. The system automatically routes it "
                        "to the strong local domain databases and can retrieve current records from pertinent scholarly databases."
                    )
                    question_box = gr.Textbox(
                        label="Your Question",
                        lines=4,
                        placeholder="How do I build an AGI-oriented agent? Or: develop a rigorous research approach to one Millennium Prize Problem.",
                    )
                    gr.Examples(
                        examples=[
                            ["How do I build an AGI-oriented agent, and how should I evaluate it safely?"],
                            ["Attempt a rigorous research approach to one of the Millennium Prize Problems."],
                            ["What experiment could distinguish the leading explanations for an unresolved scientific question?"],
                        ],
                        inputs=[question_box],
                    )
                    use_live_databases = gr.Checkbox(
                        value=True,
                        label="Search live scholarly and official databases (legacy mode only)",
                        info="Queries pertinent sources such as arXiv, OpenAlex, Semantic Scholar, Crossref, PubMed, Europe PMC, OEIS, and official sites. Individual sources can be temporarily unavailable.",
                    )
                    with gr.Accordion("Available research databases", open=False):
                        gr.Markdown(
                            "**Persistent local vector databases:** Strong AI, Strong Mathematics, Strong Science, plus uploaded documents. "
                            "Live results are cached into the pertinent local database for later questions.\n\n"
                            "**Live discovery:** arXiv · OpenAlex · Semantic Scholar · Crossref · PubMed · Europe PMC · OEIS · official NIST/Clay/NIH/NASA web results. "
                            "Retrieval supplies evidence; it does not by itself validate a proof or scientific claim."
                        )
                    ASK_MODE_SERVICE = "17-Layer Reasoning Service (STANDARD FORM — no databases)"
                    ASK_MODE_LEGACY = "Legacy: RAG + live scholarly databases"
                    ask_mode = gr.Radio(
                        choices=[ASK_MODE_SERVICE, ASK_MODE_LEGACY],
                        value=ASK_MODE_SERVICE,
                        label="Answer mode",
                        info="Default: prompt → 17-layer reasoning service → STANDARD FORM, with no database "
                             "involvement. The legacy RAG path is retained unchanged as a toggle option.",
                    )
                    ask_output = gr.Textbox(label="Answer", lines=26, interactive=False)
                    ask_status = gr.Textbox(label="Status", lines=4, interactive=False)
                    with gr.Accordion("STANDARD FORM (schema unchanged) — L10 audit + full JSON", open=False):
                        ask_audit = gr.Textbox(label="L10 Audit", lines=6, interactive=False)
                        ask_standard_form = gr.Textbox(label="Standard Form (JSON)", lines=10, interactive=False)
                        ask_to_builder_btn = gr.Button("📤 Send Standard Form to Agent Builder")
                        ask_to_builder_status = gr.Markdown()
                    ask_btn = gr.Button("Research & Answer", variant="primary")

                    def ask_five_lens(question, use_live):
                        if not question or not question.strip():
                            return "❌ Enter a question.", "❌ No question"
                        try:
                            answer, report = handle_ask_question(
                                COLLECTION_NAME,
                                question,
                                use_live_databases=bool(use_live),
                                return_report=True,
                            )
                            return answer, f"✅ Done | {report}"
                        except Exception as e:
                            return f"❌ Error: {str(e)}", "❌ Failed"

                    def ask_dispatch(question, mode, use_live):
                        if mode == ASK_MODE_LEGACY:
                            answer, status = ask_five_lens(question, use_live)
                            return answer, status, "", ""
                        result = ask_question_standard(question)
                        answer = (result.get("outputs") or {}).get("L4") or "(no L4 Final Rewrite — see status)"
                        status = (f"✅ 17-layer reasoning service · STANDARD FORM · "
                                  f"score {result.get('score_before')} → {result.get('score_after')}\n"
                                  f"{result.get('status', '')}")
                        return (answer, status, result.get("audit", "") or "",
                                json.dumps(result, indent=2, ensure_ascii=False))

                    ask_btn.click(
                        fn=ask_dispatch,
                        inputs=[question_box, ask_mode, use_live_databases],
                        outputs=[ask_output, ask_status, ask_audit, ask_standard_form],
                    )
                    ask_to_builder_btn.click(
                        fn=lambda sf: (sf, "✅ Standard form stored. Open **🔨 Builder → 📝 Generate Proposal** "
                                           "and tick 'Include Ask-a-Question standard-form output'."),
                        inputs=[ask_standard_form],
                        outputs=[builder_ask_context, ask_to_builder_status],
                    )

                with gr.TabItem("🏁 Kaggle Agent path"):
                    gr.Markdown(
                        "### Kaggle logs → design prompt → design → candidate patch\n"
                        "Paste the scoring/practice logs here. The Kaggle Agent writes prompt text only; "
                        "Ask-a-Question runs that prompt through the shared 17-layer service. The primary "
                        "Franzen/Duck path below edits a pinned **multi-file source tree** and emits a "
                        "reviewable `.diff`, changed-files `.zip`, and manifest. It never executes, uploads, "
                        "submits, or scores the candidate. The legacy single-file `.py` path remains available.\n\n"
                        "Default source pin is the reviewed Franzen repository commit shown below. Upload a ZIP "
                        "of your exact submitted source instead if its revision differs or this runtime is offline."
                    )
                    ask_kaggle_logs_box = gr.Textbox(
                        label="Kaggle logs / score evidence",
                        lines=12,
                        placeholder="Paste the complete Kaggle run output, score, status, errors, and version/hash...",
                    )
                    ask_kaggle_prompt_btn = gr.Button("🧠 Kaggle Agent: write the v4 design prompt", variant="primary")
                    ask_kaggle_prompt_out = gr.Textbox(
                        label="Design prompt (editable; prompt only)", lines=14,
                        placeholder="The Kaggle Agent will write one self-contained design prompt from the logs.",
                    )
                    with gr.Accordion("🦆 Franzen / Duck multi-file source mode (primary)", open=True):
                        gr.Markdown(
                            "Load the pinned source snapshot or upload a ZIP of the exact source used by your run. "
                            "Choose repo-relative files; add `:start-end` to focus a large file. The emitted patch "
                            "only changes those existing files/ranges. Candidate generation is not scoring or submission."
                        )
                        franzen_source_state = gr.State({})
                        franzen_source_zip = gr.File(
                            label="Optional Franzen source ZIP (takes precedence; useful offline)",
                            file_types=[".zip"], type="filepath",
                        )
                        franzen_commit = gr.Textbox(
                            label="Pinned Franzen source commit (full 40-char SHA)",
                            value=FRANZEN_BASELINE_COMMIT,
                        )
                        franzen_load_btn = gr.Button("📦 Load Franzen baseline", variant="secondary")
                        franzen_load_status = gr.Textbox(label="Baseline identity / load status", lines=2, interactive=False)
                        franzen_source_files = gr.Textbox(label="Source tree preview (ARC3-Inference Python files)", lines=6, interactive=False)
                        franzen_target_specs = gr.Textbox(
                            label="Target files or ranges (one per line; path or path:start-end)",
                            lines=7, value=FRANZEN_DEFAULT_TARGET_SPECS,
                        )
                        franzen_emit_btn = gr.Button("🛠 Emit Franzen multi-file patch from the displayed design", variant="primary")
                        franzen_emit_status = gr.Textbox(label="Multi-file candidate status", lines=3, interactive=False)
                        franzen_patch_file = gr.File(label="Reviewable unified patch (.diff)")
                        franzen_bundle_file = gr.File(label="Changed source files only (.zip)")
                        franzen_manifest_file = gr.File(label="Candidate provenance / validation manifest (.json)")

                    ask_kaggle_baseline_file = gr.File(
                        label="Legacy single-file baseline (.py; optional)",
                        file_types=[".py", ".txt"],
                    )
                    ask_kaggle_run_emit_btn = gr.Button(
                        "▶ Run design → optionally emit legacy .py", variant="primary"
                    )
                    ask_kaggle_design_output = gr.Textbox(
                        label="Design answer (STANDARD FORM L4)", lines=22, interactive=False
                    )
                    ask_kaggle_design_form = gr.Textbox(
                        label="Full design STANDARD FORM (JSON)", lines=12, interactive=False
                    )
                    ask_kaggle_emit_status = gr.Textbox(
                        label="Candidate emission / validation status", lines=4, interactive=False
                    )
                    ask_kaggle_candidate_file = gr.File(label="Legacy single-file candidate agent (.py)")
                    ask_kaggle_candidate_manifest = gr.File(label="Legacy single-file candidate manifest (.json)")
                    ask_kaggle_emit_btn = gr.Button("🛠 Emit legacy .py candidate from the displayed design", variant="secondary")

                    def ask_kaggle_prompt_handler(log_text):
                        return kaggle_write_ask_prompt(log_text)

                    def ask_kaggle_design_emit_handler(prompt_text, log_text, baseline_file):
                        if not prompt_text or not str(prompt_text).strip():
                            return ("", "", "❌ Write the design prompt first.", None, None)
                        result = kaggle_run_ask_question(prompt_text)
                        design_text = (result.get("outputs") or {}).get("L4", "") or ""
                        status = (f"Ask-a-Question · Kaggle design · score "
                                  f"{result.get('score_before')} → {result.get('score_after')}\n"
                                  f"{result.get('status', '')}")
                        standard_json = json.dumps(result, indent=2, ensure_ascii=False)
                        if not design_text.strip():
                            return ("(no L4 design; see standard form/status)", standard_json,
                                    "❌ No design was produced; candidate not emitted.", None, None)
                        if not baseline_file:
                            return (design_text, standard_json,
                                    f"{status}\nℹ️ Design is ready. Use the Franzen multi-file builder above, or upload a .py for the legacy single-file path.",
                                    None, None)
                        agent_path, manifest_path, emit_status = kaggle_emit_submission_agent(
                            design_text=design_text, logs=log_text,
                            design_prompt=prompt_text, baseline_file=baseline_file,
                        )
                        return design_text, standard_json, f"{status}\n{emit_status}", agent_path, manifest_path

                    def ask_kaggle_emit_from_design_handler(design_text, log_text, prompt_text, baseline_file):
                        if not design_text or not str(design_text).strip():
                            return "❌ Run the Kaggle design path first.", None, None
                        if not baseline_file:
                            return "❌ Upload the exact baseline agent .py file first.", None, None
                        return kaggle_emit_submission_agent(
                            design_text=design_text, logs=log_text,
                            design_prompt=prompt_text, baseline_file=baseline_file,
                        )

                    ask_kaggle_prompt_btn.click(
                        fn=ask_kaggle_prompt_handler, inputs=[ask_kaggle_logs_box], outputs=[ask_kaggle_prompt_out]
                    )
                    ask_kaggle_run_emit_btn.click(
                        fn=ask_kaggle_design_emit_handler,
                        inputs=[ask_kaggle_prompt_out, ask_kaggle_logs_box, ask_kaggle_baseline_file],
                        outputs=[ask_kaggle_design_output, ask_kaggle_design_form, ask_kaggle_emit_status,
                                 ask_kaggle_candidate_file, ask_kaggle_candidate_manifest],
                    )
                    ask_kaggle_emit_btn.click(
                        fn=ask_kaggle_emit_from_design_handler,
                        inputs=[ask_kaggle_design_output, ask_kaggle_logs_box,
                                ask_kaggle_prompt_out, ask_kaggle_baseline_file],
                        outputs=[ask_kaggle_emit_status, ask_kaggle_candidate_file,
                                 ask_kaggle_candidate_manifest],
                    )

                    def franzen_load_handler(source_zip, commit):
                        return kaggle_prepare_franzen_baseline(source_zip=source_zip, commit=commit)

                    def franzen_emit_handler(design_text, log_text, prompt_text, source_state, target_specs):
                        return kaggle_emit_franzen_patch(
                            design_text=design_text, logs=log_text, design_prompt=prompt_text,
                            baseline_state=source_state, target_specs=target_specs,
                        )

                    franzen_load_btn.click(
                        fn=franzen_load_handler,
                        inputs=[franzen_source_zip, franzen_commit],
                        outputs=[franzen_source_state, franzen_load_status, franzen_source_files],
                    )
                    franzen_emit_btn.click(
                        fn=franzen_emit_handler,
                        inputs=[ask_kaggle_design_output, ask_kaggle_logs_box, ask_kaggle_prompt_out,
                                franzen_source_state, franzen_target_specs],
                        outputs=[franzen_emit_status, franzen_patch_file,
                                 franzen_bundle_file, franzen_manifest_file],
                    )

        # ── Agent Mode Tab ──
        with gr.TabItem("🤖 Agent Mode"):
            gr.Markdown(
                "Multi-Agent Orchestration with 12 specialists powered by Gemini.\n\n"
                "Decision steps (planning, synthesis, tool-choice) route through the shared 17-layer "
                "reasoning service with a per-action budget; if the budget is exceeded, a cached or "
                "heuristic fallback decision is used instead of hanging. Trivial operations (tool "
                "calls, file reads, keystroke-level actions) never enter the pipeline."
            )
            with gr.Row():
                with gr.Column(scale=2):
                    profile_selector = gr.Dropdown(choices=list(AGENT_PROFILES.keys()), value="New Autonomous Agent", label="Agent Profile")
                    agent_goal = gr.Textbox(label="Goal / Instructions", lines=3, placeholder="e.g. Analyze our competitor positioning and recommend a content strategy...")

                    # Checkboxes
                    chk_gemini_only = gr.Checkbox(
                        label="Use Only Gemini API",
                        value=True,
                        info="When checked, only Gemini is used. All other API keys are ignored."
                    )
                    chk_enable_additional = gr.Checkbox(
                        label="Enable Additional APIs",
                        value=False,
                        info="When checked (and Gemini-only is OFF), optional API key fields become available."
                    )

                    agent_btn = gr.Button("Run Orchestrator", variant="primary")

                with gr.Column(scale=1):
                    additional_keys_accordion = gr.Accordion("🔑 Optional API Keys", open=False, visible=False)
                    with additional_keys_accordion:
                        t_cal = gr.Textbox(label="Calendar", type="password")
                        t_crm = gr.Textbox(label="CRM", type="password")
                        t_comm = gr.Textbox(label="Comm", type="password")
                        t_vision = gr.Textbox(label="Vision/OCR", type="password")
                        t_ds = gr.Textbox(label="DocuSign", type="password")
                        t_social = gr.Textbox(label="Social", type="password")
                        t_seo = gr.Textbox(label="SEO", type="password")
                        t_s3 = gr.Textbox(label="S3/Vault", type="password")
                        t_pubmed = gr.Textbox(label="PubMed", type="password")

            # Visibility logic
            def update_keys_visibility(gemini_only, enable_additional):
                show = (not gemini_only) and enable_additional
                return gr.Accordion(visible=show, open=show)

            chk_gemini_only.change(
                fn=update_keys_visibility,
                inputs=[chk_gemini_only, chk_enable_additional],
                outputs=[additional_keys_accordion]
            )
            chk_enable_additional.change(
                fn=update_keys_visibility,
                inputs=[chk_gemini_only, chk_enable_additional],
                outputs=[additional_keys_accordion]
            )

            agent_output = gr.Textbox(label="Execution Log & Output", lines=25, interactive=False)
            agent_btn.click(
                fn=run_agent,
                inputs=[
                    agent_goal,
                    chk_gemini_only, chk_enable_additional,
                    t_cal, t_crm, t_comm, t_vision, t_ds, t_social, t_seo, t_s3, t_pubmed
                ],
                outputs=agent_output
            )

        # ── Builder Tab (NEW - Replaces Agent Builder) ──
        with gr.TabItem("🔨 Builder"):
            gr.Markdown("""
            ## Automated Notebook Builder

            This tool reads your entire notebook, proposes changes, and applies them with safety checks.

            **Features:**
            - Full notebook context (no manual pasting)
            - Automated syntax validation
            - 5-lens verdict (correctness, safety, completeness, compatibility, clarity)
            - Automatic backups
            - Transparency logging
            """)

            with gr.Tabs():
                # Tab 1: Generate Proposal
                with gr.TabItem("📝 Generate Proposal"):
                    builder_notebook_path = gr.Textbox(
                        label="Notebook Path",
                        value="4CBOn2_Kaggle.ipynb",
                        placeholder="Path to the notebook file"
                    )
                    builder_direction = gr.Textbox(
                        label="Direction",
                        lines=4,
                        placeholder="Describe what you want to change or add...\n\nExample: Add a new cell that creates a data visualization dashboard using plotly"
                    )
                    builder_use_ask = gr.Checkbox(
                        label="Include Ask-a-Question standard-form output (sent from the ❓ tab)",
                        value=False,
                        info="Appends the stored STANDARD FORM result of Ask-a-Question to the direction."
                    )
                    builder_generate_btn = gr.Button("🚀 Generate Proposal", variant="primary")
                    builder_summary = gr.Markdown(value="*Proposal summary will appear here...*")
                    builder_proposals_json = gr.Textbox(
                        label="Proposals JSON",
                        lines=15,
                        interactive=True,
                        visible=False
                    )
                    builder_status = gr.Textbox(label="Status", interactive=False)

                    builder_generate_btn.click(
                        fn=generate_builder_proposal,
                        inputs=[builder_notebook_path, builder_direction, builder_use_ask, builder_ask_context],
                        outputs=[builder_summary, builder_proposals_json, builder_status]
                    )

                # Tab 2: Review & Apply
                with gr.TabItem("✅ Review & Apply"):
                    gr.Markdown("""
                    Review the generated proposals and apply approved changes.

                    **Safety Features:**
                    - ✅ Syntax validation before applying
                    - 🔍 5-lens automated verdict
                    - 💾 Automatic backup creation
                    - 📋 Full transparency log
                    """)
                    review_proposals_json = gr.Textbox(
                        label="Proposals JSON (editable)",
                        lines=20,
                        placeholder="Paste or edit proposals JSON here..."
                    )
                    review_notebook_path = gr.Textbox(
                        label="Notebook Path",
                        value="4CBOn2_Kaggle.ipynb",
                        placeholder="Path to the notebook file"
                    )
                    review_apply_btn = gr.Button("⚡ Run Automated Review & Apply", variant="primary")
                    review_log = gr.Textbox(
                        label="Transparency Log",
                        lines=25,
                        interactive=False
                    )

                    review_apply_btn.click(
                        fn=apply_proposals,
                        inputs=[review_proposals_json, review_notebook_path],
                        outputs=[review_log]
                    )

            # Link proposal JSON between tabs
            builder_generate_btn.click(
                fn=lambda x: x,
                inputs=[builder_proposals_json],
                outputs=[review_proposals_json]
            )

        # ============================================================
        # AI REWRITER TAB — ordered artifact display
        # ============================================================
        with gr.TabItem("🧠 AI Rewriter (17-Layer Pipeline)"):
            gr.Markdown("""
            ## AI Rewriter — 17-Layer Pipeline
            Paste an AI-generated answer and optionally describe its goal. The runtime executes each layer in order,
            scores the answer before and after, and preserves the three L9 Socratic questions for the next run.
            """)
            rewriter_answer = gr.Textbox(
                label="AI-generated answer",
                lines=8,
                placeholder="Paste the answer you want to inspect and improve..."
            )
            rewriter_context = gr.Textbox(
                label="Context / goal (optional)",
                lines=3,
                placeholder="What should the answer achieve, and who will use it?"
            )
            with gr.Row():
                rewriter_score_before = gr.Number(label="Score Before", precision=0, interactive=False)
                rewriter_score_after = gr.Number(label="Score After", precision=0, interactive=False)
            rewriter_status = gr.Markdown("Ready. No pipeline run yet.")
            rewriter_run_btn = gr.Button("▶ Run Pipeline", variant="primary")
            rewriter_l9_state = gr.State([])

            gr.Markdown("### Layer outputs — L0 → P → W → LX → LA → LC → L1 → L2 → LP → L3 → L4 → LR → L6 → L7 → L8 → L9 → L10")
            rewriter_layer_boxes = []
            for _layer in PIPELINE_ORDER:
                _meta = LAYER_METADATA.get(_layer, {"name": "Socratic Integrity Engine"})
                _box = gr.Textbox(
                    label=f"{_layer} — {_meta['name']}",
                    lines=3 if _layer not in ("L4", "L10") else 8,
                    interactive=False
                )
                rewriter_layer_boxes.append(_box)
            rewriter_audit = gr.Textbox(label="Final L10 Audit", lines=10, interactive=False)
            rewriter_copy_btn = gr.Button("⊡ Copy All", variant="secondary")
            rewriter_report = gr.Textbox(label="Copy All report", lines=12, interactive=False)

            _rewriter_outputs = [rewriter_score_before, rewriter_score_after, rewriter_status] + rewriter_layer_boxes + [rewriter_audit, rewriter_l9_state]
            rewriter_run_btn.click(
                fn=run_rewriter,
                inputs=[rewriter_answer, rewriter_context, rewriter_l9_state],
                outputs=_rewriter_outputs,
            )
            rewriter_copy_btn.click(
                fn=copy_all_rewriter_report,
                inputs=[rewriter_score_before, rewriter_score_after, rewriter_status] + rewriter_layer_boxes,
                outputs=[rewriter_report],
            )

        # ============================================================
        # DATABASE DOWNLOADER TAB (Module 1)
        # ============================================================
        with gr.TabItem("⬇️ Database Downloader"):
            gr.Markdown(
                "## Database Downloader (AGI/ARC)\n"
                "**Which** additional AGI/ARC databases to download is decided by the shared 17-layer "
                "reasoning service. The raw download itself is plain I/O and never passes through the "
                "pipeline. All existing databases are untouched — downloads land in "
                "`DATA_ROOT/4cbon2_downloads/`. This module is never called by Ask-a-Question."
            )
            gr.Markdown(downloader_catalog_markdown())
            downloader_note = gr.Textbox(
                label="Optional note to the reasoning service (constraints, goals, disk budget)",
                lines=2, placeholder="e.g. prioritise ARC-AGI-3 preparation; keep it small"
            )
            downloader_decide_btn = gr.Button("🧠 Decide via 17-Layer Service", variant="primary")
            downloader_decision = gr.Textbox(
                label="Service decision — catalog ids to download (editable; manual edits are operator overrides)",
                lines=2, placeholder="e.g. arc_agi_1, arc_agi_2, arc_agi_3_agents"
            )
            downloader_rationale = gr.Textbox(label="Decision rationale", lines=6, interactive=False)
            downloader_download_btn = gr.Button("⬇️ Download Selected (raw I/O — no pipeline)", variant="primary")
            downloader_run_log = gr.Textbox(label="Download log — what / from where / when", lines=10, interactive=False)
            downloader_history_btn = gr.Button("📜 Show full download history")
            downloader_history = gr.Textbox(label="Download history", lines=10, interactive=False)

            downloader_decide_btn.click(
                fn=decide_downloads, inputs=[downloader_note],
                outputs=[downloader_decision, downloader_rationale]
            )
            downloader_download_btn.click(
                fn=download_selected, inputs=[downloader_decision], outputs=[downloader_run_log]
            )
            downloader_history_btn.click(fn=get_download_log_text, inputs=[], outputs=[downloader_history])

        # ============================================================
        # KAGGLE AGENT TAB (coordination component)
        # ============================================================
        with gr.TabItem("🕹️ Kaggle Agent"):
            gr.Markdown(
                "## Kaggle Agent — Coordinator\n"
                "The Kaggle Agent runs on the shared 17-layer reasoning service. It coordinates download decisions and log-grounded design prompts; the new design-to-candidate flow is in Ask a Question → Kaggle Agent path.\n"
                "(a) decide which additional databases the Database Downloader should download;\n"
                "(b) write a log-grounded v4 design prompt handed to Ask-a-Question. "
                "For the full design-to-candidate loop, use Ask a Question → Kaggle Agent path."
            )
            gr.Markdown("### Job (a) — Database download decision")
            kaggle_db_btn = gr.Button("🧠 Decide database downloads", variant="primary")
            kaggle_db_out = gr.Textbox(label="Kaggle Agent decision — clean, copy-pasteable", lines=10, interactive=False)
            kaggle_db_ids = gr.Textbox(label="Decision ids (sent to the Database Downloader)", lines=2, interactive=False)
            kaggle_db_send_btn = gr.Button("📤 Send decision to Database Downloader")
            gr.Markdown("### Job (b) — Log-grounded Kaggle v4 design prompt")
            kaggle_coord_logs = gr.Textbox(label="Kaggle logs (required for a log-grounded prompt)", lines=8)
            kaggle_coord_prompt_btn = gr.Button("🧠 Write the Ask-a-Question prompt", variant="primary")
            kaggle_coord_prompt_out = gr.Textbox(label="Prompt — clean, copy-pasteable (editable)", lines=12)
            kaggle_ask_run_btn = gr.Button("▶ Run Ask-a-Question with this prompt (STANDARD FORM)", variant="primary")
            kaggle_ask_answer = gr.Textbox(label="Ask-a-Question — L4 Final Rewrite", lines=18, interactive=False)
            kaggle_ask_status = gr.Textbox(label="Standard-form status", lines=3, interactive=False)
            kaggle_ask_form = gr.Textbox(label="Full STANDARD FORM (JSON)", lines=10, interactive=False)
            kaggle_to_builder_btn = gr.Button("📤 Send result to Agent Builder")
            kaggle_builder_status = gr.Markdown()

            def kaggle_decide_handler():
                return kaggle_decide_databases()

            def kaggle_ask_handler(prompt_text):
                result = kaggle_run_ask_question(prompt_text)
                answer = (result.get("outputs") or {}).get("L4") or "(no L4 Final Rewrite — see status)"
                status = (f"✅ 17-layer reasoning service · STANDARD FORM · "
                          f"score {result.get('score_before')} → {result.get('score_after')}\n"
                          f"{result.get('status', '')}")
                return answer, status, json.dumps(result, indent=2, ensure_ascii=False)

            kaggle_db_btn.click(fn=kaggle_decide_handler, inputs=[], outputs=[kaggle_db_out, kaggle_db_ids])
            kaggle_db_send_btn.click(fn=lambda ids: ids, inputs=[kaggle_db_ids], outputs=[downloader_decision])
            kaggle_coord_prompt_btn.click(fn=kaggle_write_ask_prompt, inputs=[kaggle_coord_logs], outputs=[kaggle_coord_prompt_out])
            kaggle_ask_run_btn.click(
                fn=kaggle_ask_handler, inputs=[kaggle_coord_prompt_out],
                outputs=[kaggle_ask_answer, kaggle_ask_status, kaggle_ask_form]
            )
            kaggle_to_builder_btn.click(
                fn=lambda sf: (sf, "✅ Standard form stored. Open **🔨 Builder → 📝 Generate Proposal** "
                                   "and tick 'Include Ask-a-Question standard-form output'."),
                inputs=[kaggle_ask_form],
                outputs=[builder_ask_context, kaggle_builder_status]
            )

        # ── Data Dashboard Tab ──
        with gr.TabItem("📊 Data Dashboard"):
            gr.Markdown("""
            ## Task Memory Visualization

            Visualize your agent task history with interactive Plotly charts.

            **Metrics:**
            - Task response length over time
            - Subtask distribution
            - Common words in task goals
            """)

            dashboard_btn = gr.Button("🔄 Load Dashboard", variant="primary")
            dashboard_output = gr.Textbox(label="Status", interactive=False)

            with gr.Row():
                dashboard_plot1 = gr.Plot(label="Task Timeline")
                dashboard_plot2 = gr.Plot(label="Subtask Distribution")

            with gr.Row():
                dashboard_plot3 = gr.Plot(label="Goal Word Frequency")

            def load_dashboard():
                try:
                    figures, error = create_plotly_dashboard()
                    if error:
                        return f"❌ {error}", None, None, None

                    if not figures:
                        return "❌ No figures generated", None, None, None

                    # Return up to 3 figures
                    fig1 = figures[0] if len(figures) > 0 else None
                    fig2 = figures[1] if len(figures) > 1 else None
                    fig3 = figures[2] if len(figures) > 2 else None

                    return f"✅ Loaded {len(figures)} visualization(s)", fig1, fig2, fig3
                except Exception as e:
                    return f"❌ Error: {str(e)}", None, None, None

            dashboard_btn.click(
                fn=load_dashboard,
                inputs=[],
                outputs=[dashboard_output, dashboard_plot1, dashboard_plot2, dashboard_plot3]
            )

        # ── Agent Status Tab ──
        with gr.TabItem("📊 Agent Status"):
            gr.Markdown("View all agent conversation histories.")
            refresh_btn = gr.Button("Refresh")
            agent_status_display = gr.Markdown("Click refresh to load.")
            def get_agent_status():
                output = "## 📊 Agent Status\n\n"
                for agent_id in get_all_agents():
                    agent = load_agent(agent_id)
                    history_len = len(agent.get("conversation_history", []))
                    output += f"- **{agent_id}**: {history_len} messages\n"
                return output
            refresh_btn.click(fn=get_agent_status, outputs=[agent_status_display])
            demo.load(fn=get_agent_status, outputs=[agent_status_display])

demo.queue()
demo.launch(inline=False, share=True)

In [ ]:
# CELL 12 — LEGACY V3 BUILD MANIFEST CONSTANTS (Phase 4 / M3′, additive cell; no LLM,
# no I/O — pure data. Cell 15 turns these into build_manifest.json).
# v2.1 (v3-fixed) build: received v3 audit + 4 approved fixes + the
# design-mandated salience restore (_salience_explore, §4.2). This does not emit v4 candidates.
BUILD = {
    "product": "Legacy ARC-AGI-3 ProgramSynthesis agent v2.1 (v3-fixed)",
    "build_id": "v2.1-M3",
    "generator": ("4CBOn2_Kaggle4.ipynb cells 12-17 (legacy frozen v3 emitter; "
                  "the dynamic Kaggle Agent candidate flow is in cells 10-11)"),
    "artifacts_to_emit": ["program_synthesis_agent_v3.py",
                          "ARC_AGI_3_Kagglea2.ipynb",
                          "build_manifest.json"],
    "scope": {
        "features_v2_1": [
            "state-action exploration graph (tried_cells / clicked_sig dedup)",
            "EXPLORATION_CAP=12 accretion protection (explore_blocked, "
            "<40 states)",
            "typed-evidence world model (settled vs transient frames)",
            "salience-ordered no-signal exploration (_salience_explore — "
            "the §4.2 restore; uncapped, worst case = v2.0 exactly)",
        ],
        "features_v2_0_retained": [
            "frame diffing + no-op blacklist",
            "cross-level verified replay",
            "level-1-first exploration",
        ],
        "baseline_non_negotiable": [
            "never-hang wrapper (try/except decide path + time budget)",
            "protocol discipline (arcengine-or-shim, RESET after GAME_OVER)",
            "UL governance scaffolding (inspiration only, no UL code ships)",
            "v1 core as-is (perception/salience, click-selector bank, "
            "movement probing — not generalized)",
        ],
        "app_cell_changes": "Kaggle4 design flow in cells 0, 8-11; cells 12-17 remain the frozen v3 baseline emitter",
    },
    "hard_boundary": ("Kaggle scoring rerun has no internet, no App access, "
                      "no DB access — agent + ARC-AGI-3 protocol layer fully "
                      "embedded in the kernel notebook at build time"),
    "lineage": {
        "v1_kaggle_score": 0.06,
        "design": "v2_design/DESIGN_v2_1.md (approved 2026-09-30)",
        "M1_agent": ("PASS 2026-09-30 (M1′: compile/contract/kit-compat, "
                     "v2.0 M1 suite parity, frozen-game loop gate, "
                     "accretion-cap gate)"),
        "M2_kernel": ("PASS 2026-09-30 (M2′: kernel a2 gates A-D; 25-game "
                      "head-to-head v3-fixed ≡ v2.0 — 1 level (lp85) each, "
                      "aggregate 0.0890 each)"),
        "M3_builder": "historical v3 baseline build (retained; does not consume Kaggle4 UI candidates)",
    },
    "head_to_head_local_25": {
        "games": 25, "max_steps": 200, "max_actions": 80,
        "v1": {"levels": 0, "wins": 0, "aggregate": 0.0},
        "v2_0": {"levels": 1, "levels_detail": "lp85", "wins": 0,
                 "aggregate": 0.08895044629116651},
        "v3_received_plus_4_fixes": {"levels": 0, "wins": 0,
                                     "aggregate": 0.0,
                                     "note": "regression: blind-cell "
                                             "exploration replaced the "
                                             "salience fallback"},
        "v3_fixed": {"levels": 1, "levels_detail": "lp85", "wins": 0,
                     "aggregate": 0.08895044629116651,
                     "note": "action-for-action identical to v2.0 on all "
                             "25 games"},
    },
    "known_limitations": [
        ("v2.1 features are behaviorally inert on the local 25-game set "
         "(identical actions to v2.0); local ceiling for this agent family "
         "≈ 0.089 via the v1-inherited selector bank + v2.0 discipline"),
        ("graph/cap code paths are first exercised for real on the Kaggle "
         "scoring rerun; worst case bounded to the v2.0 strategy by design"),
        ("local play-local on ls20/vc33 completes 0 levels in 80 actions "
         "(movement probing locked as v1-as-is; real test is the Kaggle "
         "score)"),
    ],
}
print(f"[c12] manifest constants ready — build {BUILD['build_id']} "
      f"(scope locked: {len(BUILD['scope']['features_v2_1'])} v2.1 features "
      f"+ {len(BUILD['scope']['features_v2_0_retained'])} retained v2.0 "
      f"features + baseline; head-to-head provenance embedded)")

[c12] manifest constants ready — build v2.1-M3 (scope locked: 4 v2.1 features + 3 retained v2.0 features + baseline; head-to-head provenance embedded)


In [ ]:
# CELL 13 — AGENT CORE EMBED (frozen source; the single source of truth for
# this build. Byte-identical to the M1-delivered program_synthesis_agent_v3.py
# with M2's kit-compatibility additions). No LLM, no network.
import hashlib
import importlib.util
import os
import tempfile

AGENT_SOURCE = r'''"""program_synthesis_agent_v3.py — self-contained ARC-AGI-3 agent (v2.1)

v2.1 adds two features over v2.0:
  F4 state-action exploration graph (novelty-seeking when no hypothesis)
  F5 typed evidence: transient/revert detection (settled vs animation)

Carried over from v2.0:
  F1 frame diffing + no-op blacklist    F2 cross-level verified replay
  F3 level-1-first exploration

Carried over from v1:
  Protocol layer, perception, micro-DSL, hypothesis bank, never-hang wrapper.

Contract: class MyAgent with choose_action(frames, latest_frame) and
is_done(frames, latest_frame). numpy + stdlib only; no network.
"""
import copy
import time
from collections import deque

import numpy as np

# ============================================================================
# SECTION 1 — PROTOCOL LAYER (unchanged from v2)
# ============================================================================
GRID = 64
CELL = 8

try:
    from arcengine import FrameData as _RealFrameData  # noqa: F401
    from arcengine import GameAction, GameState
    from arcengine.enums import (SimpleAction, ComplexAction, FrameData,
                                 FrameDataRaw, ActionInput)  # noqa: F401
    ARCENGINE_AVAILABLE = True
except Exception:
    ARCENGINE_AVAILABLE = False

    from enum import Enum

    class GameState(str, Enum):
        NOT_PLAYED = "NOT_PLAYED"
        NOT_FINISHED = "NOT_FINISHED"
        WIN = "WIN"
        GAME_OVER = "GAME_OVER"

    class SimpleAction:
        def __init__(self, game_id: str = ""):
            self.game_id = game_id

        def model_dump(self):
            return {"game_id": self.game_id}

    class ComplexAction:
        def __init__(self, game_id: str = "", x: int = 0, y: int = 0):
            if not (0 <= x <= 63 and 0 <= y <= 63):
                raise ValueError("x and y must be within [0, 63]")
            self.game_id = game_id
            self.x = x
            self.y = y

        def model_dump(self):
            return {"game_id": self.game_id, "x": self.x, "y": self.y}

    class GameAction(Enum):
        RESET = (0, SimpleAction)
        ACTION1 = (1, SimpleAction)
        ACTION2 = (2, SimpleAction)
        ACTION3 = (3, SimpleAction)
        ACTION4 = (4, SimpleAction)
        ACTION5 = (5, SimpleAction)
        ACTION6 = (6, ComplexAction)
        ACTION7 = (7, SimpleAction)

        def __init__(self, action_id, action_type):
            self._value_ = action_id
            self._atype = action_type
            self.action_data = action_type()

        def is_simple(self):
            return self._atype is SimpleAction

        def is_complex(self):
            return self._atype is ComplexAction

        def set_data(self, data):
            self.action_data = self._atype(**data)
            return self.action_data

        @classmethod
        def from_id(cls, action_id):
            for a in cls:
                if a.value == action_id:
                    return a
            raise ValueError(f"No GameAction with id {action_id}")

    class ActionInput:
        def __init__(self, id=GameAction.RESET, data=None, reasoning=None):
            self.id = id
            self.data = data or {}
            self.reasoning = reasoning

    class FrameData:
        def __init__(self, game_id="", frame=None, state=GameState.NOT_PLAYED,
                     levels_completed=0, win_levels=0, action_input=None,
                     guid=None, full_reset=False, available_actions=None):
            self.game_id = game_id
            self.frame = frame or []
            self.state = state
            self.levels_completed = levels_completed
            self.win_levels = win_levels
            self.action_input = action_input or ActionInput()
            self.guid = guid
            self.full_reset = full_reset
            self.available_actions = list(available_actions or [])

        def is_empty(self):
            return len(self.frame) == 0


def _pick_frame(frames, latest_frame):
    """Prefer the freshest frame that actually carries grid data.

    The kit's main() seeds `frames` with a dummy FrameData and passes the
    converted current observation as latest_frame, so latest_frame wins
    whenever it has pixels; the history covers callers that pass a stale
    or empty second argument but real frames in `frames`.
    """
    try:
        if getattr(latest_frame, "frame", None):
            return latest_frame
    except Exception:
        pass
    try:
        for cand in reversed(list(frames or [])):
            if getattr(cand, "frame", None):
                return cand
    except Exception:
        pass
    return latest_frame


def _state_name(state):
    try:
        return str(getattr(state, "value", state)).split(".")[-1]
    except Exception:
        return ""


def _grid_of(frame):
    try:
        layers = getattr(frame, "frame", None)
        if not layers:
            layers = getattr(frame, "_frame", None)
        if not layers:
            return None
        arr = np.asarray(layers[0])
        if arr.ndim != 2:
            return None
        return arr.astype(np.int64, copy=False)
    except Exception:
        return None


def merged_view(frame):
    try:
        layers = getattr(frame, "frame", None) or []
        if not layers:
            return None
        base = np.asarray(layers[0]).astype(np.int64, copy=False)
        for extra in layers[1:]:
            arr = np.asarray(extra)
            if arr.shape == base.shape:
                base = np.where(arr != 0, arr, base)
        return base
    except Exception:
        return None


# ============================================================================
# SECTION 2 — PERCEPTION (unchanged from v2)
# ============================================================================
def find_objects(grid):
    objects = []
    visited = set()
    h, w = grid.shape
    for r in range(h):
        for c in range(w):
            if grid[r, c] == 0 or (r, c) in visited:
                continue
            color = int(grid[r, c])
            obj_id = len(objects) + 1
            q = deque([(r, c)])
            visited.add((r, c))
            pixels = []
            min_r, max_r, min_c, max_c = r, r, c, c
            while q:
                cr, cc = q.popleft()
                pixels.append((cr, cc))
                min_r, max_r = min(min_r, cr), max(max_r, cr)
                min_c, max_c = min(min_c, cc), max(max_c, cc)
                for dr, dc in ((-1, 0), (1, 0), (0, -1), (0, 1)):
                    nr, nc = cr + dr, cc + dc
                    if (0 <= nr < h and 0 <= nc < w and (nr, nc) not in visited
                            and grid[nr, nc] == color):
                        visited.add((nr, nc))
                        q.append((nr, nc))
            cy = sum(p[0] for p in pixels) / len(pixels)
            cx = sum(p[1] for p in pixels) / len(pixels)
            objects.append({"id": obj_id, "color": color,
                            "pixels": frozenset(pixels),
                            "bbox": (min_r, min_c, max_r, max_c),
                            "size": len(pixels), "centroid": (cy, cx)})
    return objects


def frame_diff(grid_a, grid_b):
    if grid_a is None or grid_b is None or grid_a.shape != grid_b.shape:
        return np.ones_like(np.asarray(grid_b), dtype=bool)
    return grid_a != grid_b


def salience_rank(objects):
    color_count = {}
    for o in objects:
        color_count[o["color"]] = color_count.get(o["color"], 0) + 1
    ranked = sorted(objects,
                    key=lambda o: (color_count[o["color"]], o["size"], o["bbox"]))
    return ranked, color_count


# ============================================================================
# SECTION 3 — MICRO-DSL (unchanged from v2)
# ============================================================================
def prim_recolor_region(grid, mask, new_color):
    g = copy.deepcopy(grid)
    g[mask] = new_color
    return g


def prim_move_region(grid, mask, dr, dc):
    g = copy.deepcopy(grid)
    color = int(g[mask][0]) if mask.any() else 0
    g[mask] = 0
    rows = np.nonzero(mask)[0] + dr
    cols = np.nonzero(mask)[1] + dc
    keep = ((rows >= 0) & (rows < g.shape[0])
            & (cols >= 0) & (cols < g.shape[1]))
    g[rows[keep], cols[keep]] = color
    return g


# ============================================================================
# SECTION 4 — GOVERNANCE (UL-derived, unchanged from v2)
# ============================================================================
EFFECT_NO_EFFECT = "NO_EFFECT"
EFFECT_LOCAL = "LOCAL_CHANGE"
EFFECT_GLOBAL = "GLOBAL_CHANGE"
EFFECT_LEVEL = "LEVEL_ADVANCED"
EFFECT_TRANSIENT = "TRANSIENT_REVERT"   # v2.1 new

EVIDENCE_VERIFIED = "VERIFIED"
EVIDENCE_CORROBORATED = "CORROBORATED"

STATUS_ACTIVE = "ACTIVE"
STATUS_STALE = "STALE"
STATUS_QUARANTINED = "QUARANTINED"

BLACKLIST_CAP = 64
SEQ_STORE_CAP = 60
GLOBAL_CHANGE_FRACTION = 0.25

# v2.1 state-graph constants
GRAPH_HASH_STRIDE = 4               # downsample grid for hashing
GRAPH_RECENT_CAP = 12               # history of recent distinct states
GRAPH_TRANSIENT_MAX_DIFF = 32       # pixel diff threshold for transient
EXPLORATION_CAP = 12                # consecutive no-progress explore actions
                                    # (design §4.3: worst case = v2.0 minus
                                    #  at most EXPLORATION_CAP actions)


class EpisodeLedger:
    def __init__(self):
        self.events = []

    def append(self, kind, **payload):
        eid = len(self.events)
        self.events.append(dict(id=eid, kind=kind, **payload))
        return eid

    def __len__(self):
        return len(self.events)


class RuleRecord:
    def __init__(self, rule_id, kind):
        self.rule_id = rule_id
        self.kind = kind
        self.evidence_class = None
        self.status = STATUS_ACTIVE
        self.success = 0
        self.fail = 0
        self.rotations = 0
        self.evidence = []

    def cite(self, eid):
        self.evidence.append(eid)


# ============================================================================
# SECTION 4.5 — STATE-ACTION EXPLORATION GRAPH (v2.1 new)
# ============================================================================
class StateGraph:
    """Novelty-seeking state-action graph over settled frames.

    - States are hashed via a downsampled grid to filter single-pixel noise.
    - `recent` keeps distinct consecutive state hashes (auto-capped).
    - `visits` counts how many times each state was observed.
    - `transitions` records (from_state -> action_key -> to_state).
    - `action_uses` counts how many transitions each action participated in.

    Transient detection: a new state is flagged as transient (revert) iff
    it matches the state two distinct-changes ago AND the frame diff is
    small (< GRAPH_TRANSIENT_MAX_DIFF pixels). Small reverts are animation.
    """

    def __init__(self):
        self.recent = deque(maxlen=GRAPH_RECENT_CAP)
        self.visits = {}
        self.transitions = {}
        self.action_uses = {}

    @staticmethod
    def hash_grid(grid):
        if grid is None:
            return None
        try:
            arr = np.asarray(grid)
            if arr.ndim != 2:
                return None
            small = arr[::GRAPH_HASH_STRIDE, ::GRAPH_HASH_STRIDE]
            return hash(small.tobytes())
        except Exception:
            return None

    def is_revert(self, h, diff_pixels):
        """True if h matches the state two changes ago AND diff is small."""
        if h is None or diff_pixels > GRAPH_TRANSIENT_MAX_DIFF:
            return False
        if len(self.recent) < 2:
            return False
        return self.recent[-2] == h

    def observe(self, h):
        """Record this frame's hash. Idempotent for consecutive repeats."""
        if h is None:
            return
        if self.recent and self.recent[-1] == h:
            return
        self.recent.append(h)
        self.visits[h] = self.visits.get(h, 0) + 1

    def record_transition(self, from_h, key, to_h):
        if from_h is None or to_h is None or key is None:
            return
        self.transitions.setdefault(from_h, {})[key] = to_h
        aid = key if isinstance(key, int) else key[0]
        self.action_uses[aid] = self.action_uses.get(aid, 0) + 1

    def novelty_action(self, current_h, avail):
        """Pick a novelty-seeking action at the current state.
        Priority: untried simple action (least-used globally), then untried
        click cell (if 6 in avail), then action leading to least-visited
        state, else None."""
        if not avail:
            return None
        tried = self.transitions.get(current_h, {}) if current_h is not None else {}
        # 1) untried simple actions
        simple_untried = [a for a in avail if a != 6 and a not in tried]
        if simple_untried:
            return min(simple_untried,
                       key=lambda a: self.action_uses.get(a, 0))
        # 2) untried click cells (only if 6 is available)
        if 6 in avail:
            tried_cells = {k[1:] for k in tried
                           if isinstance(k, tuple) and k[0] == 6}
            for r in range(8):
                for c in range(8):
                    if (r, c) not in tried_cells:
                        return (6, r, c)
        # 3) least-visited successor (FIX audit B2: self-loops are dead
        # ends — a no-effect pair must never win the novelty pick)
        candidates = [(k, v) for k, v in tried.items() if v != current_h]
        if candidates:
            return min(candidates,
                       key=lambda p: self.visits.get(p[1], 0))[0]
        return None

    def clear_recent(self):
        self.recent.clear()


class GameKnowledge:
    def __init__(self, game_id):
        self.game_id = game_id
        self.rules = {}
        self.move = None
        self.sequences = {}
        self.graph = StateGraph()   # v2.1 persistent graph

    def rule(self, name):
        if name not in self.rules:
            self.rules[name] = RuleRecord(name, "click")
        return self.rules[name]

    def best_selector(self, order):
        def key(sel):
            rec = self.rules.get(sel)
            if rec is None:
                return (2, 0, 0)
            if rec.status == STATUS_QUARANTINED:
                return (9, 0, 0)
            if rec.status == STATUS_STALE:
                return (1, 0, 0)
            rank = {EVIDENCE_VERIFIED: 0, EVIDENCE_CORROBORATED: 1, None: 2}
            return (0, rank.get(rec.evidence_class, 2), -rec.success)
        return sorted(order, key=key)


def classify_effect(prev, cur, levels_before, levels_after, transient=False):
    if levels_after > levels_before:
        return EFFECT_LEVEL, 0
    d = frame_diff(prev, cur)
    n = int(d.sum())
    if n == 0:
        return EFFECT_NO_EFFECT, 0
    if transient:
        return EFFECT_TRANSIENT, n
    if n > GLOBAL_CHANGE_FRACTION * prev.size or n > 1024:
        return EFFECT_GLOBAL, n
    return EFFECT_LOCAL, n


# ============================================================================
# SECTION 5 — PROGRAM SYNTHESIS CORE v2.1
# ============================================================================
CLICK_SELECTORS = ["rarest_color", "smallest_object",
                   "corner_sample_color", "largest_object"]
PROBE_ATTEMPT_CAP = 3
REPLAY_ABORT_STALE = 2


def _action_key(pend):
    """Hashable key identifying the pending action for the state graph."""
    if not pend:
        return None
    aid = pend.get("aid")
    if aid is None:
        return None
    if aid == 6:
        data = pend.get("data") or {}
        x = int(data.get("x", 0))
        y = int(data.get("y", 0))
        return (6, y // CELL, x // CELL)
    return aid


class ProgramSynthesisV2Core:

    GAME_MEMORY = {}

    def __init__(self, verbose=False):
        self.verbose = verbose
        self._reset_episode()

    def _reset_episode(self):
        self.game_id = ""
        self.ledger = EpisodeLedger()
        self.history = []
        self.pending = None
        self.level = 0
        self.mode = None
        self.sel_order = list(CLICK_SELECTORS)
        self.queue = []
        self.queue_ptr = 0
        self.clicked_sig = set()
        self.blacklist = set()
        self.dirs = {}
        self.probe_attempts = {}
        self.player_color = None
        self.stuck = 0
        self.replay = None
        self.level_actions = []
        self.level_verified = False
        # v2.1 exploration budget (design §4.3)
        self.explore_streak = 0
        self.explore_blocked = False
        self._explore_rr = 0
        # v2.1: transient graph, replaced in new_episode with persistent one
        self.graph = StateGraph()

    def new_episode(self, game_id):
        know = ProgramSynthesisV2Core.GAME_MEMORY.get(game_id)
        self._reset_episode()
        self.game_id = game_id or ""
        if know is None:
            know = GameKnowledge(game_id)
            ProgramSynthesisV2Core.GAME_MEMORY[game_id] = know
        self.knowledge = know
        self.graph = know.graph                     # v2.1 persistent
        if know.move:
            self.dirs = dict(know.move.get("dirs") or {})
            self.player_color = know.move.get("player_color")
        self.sel_order = know.best_selector(list(CLICK_SELECTORS))
        self.ledger.append("EPISODE_START", game_id=self.game_id)

    # ------------------------------------------------------------- main entry
    def next_action(self, frame):
        gid = ""
        try:
            gid = getattr(frame, "game_id", "") or ""
        except Exception:
            pass
        if gid and gid != self.game_id:
            self.new_episode(gid)

        grid = self._grid(frame)
        if grid is None:
            return 0, {"game_id": self.game_id}, {"note": "no frame data"}

        if self.history:
            try:
                self._process_feedback(self.history[-1], grid, frame)
            except Exception:
                self.pending = None
        self.history.append(grid)

        state = _state_name(getattr(frame, "state", None))
        if state in ("NOT_PLAYED", "GAME_OVER"):
            self._clear_level_state()
            self.ledger.append("RESET_ISSUED", why=state)
            return 0, {"game_id": self.game_id}, {
                "strategy": "reset", "why": state,
                "hypothesis": self.sel_order[0] if self.sel_order else None}

        avail = self._available(frame)
        if not avail:
            return 0, {"game_id": self.game_id}, {"note": "no available actions"}
        self.mode = "click" if 6 in avail else "move"

        if self.replay is None and self.mode:
            seqrec = self.knowledge.sequences.get(self.level)
            if seqrec and not seqrec.get("stale"):
                self.replay = {"actions": seqrec["actions"], "idx": 0}
                self.ledger.append("REPLAY_STARTED", level=self.level,
                                   evidence_class=seqrec["evidence_class"],
                                   n_actions=len(seqrec["actions"]))
                if self.verbose:
                    print(f"    [v3] replay level {self.level}: "
                          f"{len(seqrec['actions'])} actions")
        if self.replay is not None:
            return self._replay_step(grid, frame, avail)

        if self.mode == "click":
            return self._click_policy(grid, frame, avail)
        return self._move_policy(grid, frame, avail)

    # ------------------------------------------------------------- feedback
    def _process_feedback(self, prev, cur, frame):
        levels_now = int(getattr(frame, "levels_completed", 0) or 0)
        pend = self.pending or {}
        self.pending = None
        aid = pend.get("aid")

        # v2.1 state graph: hash both frames
        h_prev = self.graph.hash_grid(prev)
        h_cur = self.graph.hash_grid(cur)
        diff_px = int(frame_diff(prev, cur).sum())

        # v2.1 transient detection: revert to state-2-ago with small diff
        is_revert = (h_cur is not None and h_prev != h_cur
                     and self.graph.is_revert(h_cur, diff_px))

        # v2.1 observe the new state (records visit + recent)
        if h_cur is not None:
            self.graph.observe(h_cur)

        effect, npx = classify_effect(prev, cur, self.level, levels_now,
                                      transient=is_revert)
        self.ledger.append("EFFECT_CLASSIFIED", action=aid, effect=effect,
                           changed=npx, level=levels_now,
                           replay=bool(pend.get("replay")))

        # v2.1 record transition (settled only; transients never recorded).
        # FIX (audit B2): NO_EFFECT pairs are recorded as SELF-LOOPS so the
        # graph counts them as tried — otherwise novelty_action re-proposes
        # the same dead action forever (the v1 _pending bug reborn: a frozen
        # game burned 71/80 actions on ACTION1).
        if (h_prev is not None and h_cur is not None
                and effect != EFFECT_TRANSIENT):
            key = _action_key(pend)
            if key is not None and (h_prev != h_cur
                                    or effect == EFFECT_NO_EFFECT):
                self.graph.record_transition(h_prev, key, h_cur)

        # v2.1 transients don't update rules; they're animation noise
        if effect == EFFECT_TRANSIENT:
            return

        # prediction validation
        verified = False
        predicted = pend.get("predicted")
        if predicted is not None and effect in (EFFECT_LOCAL, EFFECT_GLOBAL):
            ok = float((predicted == cur).mean()) if predicted.shape == cur.shape else 0.0
            if ok >= 0.97:
                verified = True

        if aid is not None:
            self.level_actions.append({
                "aid": aid, "data": pend.get("data") or {},
                "effect": effect, "verified": verified})
            if verified:
                self.level_verified = True
        if verified:
            self.explore_streak = 0   # progress: exploration budget resets

        if pend.get("replay"):
            expected = pend.get("expected")
            if effect != expected and effect != EFFECT_LEVEL:
                self._replay_abort(cause=f"expected {expected}, saw {effect}")
                return
            if effect == EFFECT_LEVEL:
                self.replay = None
                self.ledger.append("REPLAY_STEP_DONE", effect=effect)
                self._on_level_advanced(levels_now, pend)
                return
            self.ledger.append("REPLAY_STEP_OK", effect=effect)
            return

        if effect == EFFECT_LEVEL:
            self._on_level_advanced(levels_now, pend)
            return

        kind = pend.get("kind")
        if effect == EFFECT_NO_EFFECT:
            self._on_no_effect(pend)
            if kind == "probe":
                self._learn_direction(prev, cur, aid)
        elif kind == "click":
            self._on_click_feedback(pend, effect, verified)
        elif kind == "probe":
            self._learn_direction(prev, cur, aid)
        elif kind == "move":
            self._on_move_feedback(pend, prev, cur, aid, verified)
        # v2.1: "explore" actions update the graph but no rules

    def _on_level_advanced(self, levels_now, pend):
        seq = [a for a in self.level_actions if a["effect"] != EFFECT_NO_EFFECT]
        if 0 < len(self.level_actions) <= SEQ_STORE_CAP and seq:
            self.knowledge.sequences[self.level] = {
                "actions": seq,
                "evidence_class": (EVIDENCE_VERIFIED if self.level_verified
                                   else EVIDENCE_CORROBORATED),
                "aborts": 0, "stale": False}
            self.ledger.append("WIN_SEQUENCE_STORED", level=self.level,
                               n_actions=len(seq))
        rule_name = pend.get("rule")
        if self.mode == "move" and self.knowledge.move:
            rec = self.knowledge.move["record"]
            rec.success += 1
            if rec.evidence_class is None:
                rec.evidence_class = EVIDENCE_CORROBORATED
        elif rule_name:
            rec = self.knowledge.rule(rule_name)
            rec.success += 1
            if rec.evidence_class is None:
                rec.evidence_class = EVIDENCE_CORROBORATED
        self.ledger.append("LEVEL_ADVANCED", to=levels_now)
        self.level = levels_now
        self._clear_level_state()

    def _on_no_effect(self, pend):
        aid = pend.get("aid")
        kind = pend.get("kind")
        if kind == "click" and pend.get("sig"):
            self.clicked_sig.add(pend["sig"])
            if len(self.clicked_sig) > BLACKLIST_CAP:
                self.clicked_sig.pop()
        elif kind in ("simple", None) and self.mode != "move" and aid is not None:
            key = (self.level, aid)
            if key not in self.blacklist and len(self.blacklist) < BLACKLIST_CAP:
                self.blacklist.add(key)
                self.ledger.append("ACTION_BLACKLISTED", action=aid,
                                   level=self.level)
        if kind == "click" and pend.get("rule"):
            rec = self.knowledge.rule(pend["rule"])
            rec.fail += 1
        if kind == "move":
            self.stuck += 1

    def _on_click_feedback(self, pend, effect, verified):
        rule_name = pend.get("rule")
        if not rule_name:
            return
        rec = self.knowledge.rule(rule_name)
        if verified:
            rec.success += 1
            if rec.evidence_class != EVIDENCE_VERIFIED:
                rec.evidence_class = EVIDENCE_VERIFIED
                if self.verbose:
                    print(f"    [v3] VERIFIED '{rule_name}'")
            rec.cite(self.ledger.append("RULE_VERIFIED", rule=rule_name))
        elif pend.get("predicted") is not None:
            rec.fail += 1
            if rec.evidence_class == EVIDENCE_VERIFIED:
                rec.status = STATUS_STALE
                self.ledger.append("RULE_STALE", rule=rule_name,
                                   cause="prediction mismatch")
            self._demote(rule_name, why="prediction mismatch")
        else:
            rec.fail += 1

    def _on_move_feedback(self, pend, prev, cur, aid, verified):
        if self.knowledge.move is None:
            return
        rec = self.knowledge.move["record"]
        if verified:
            rec.success += 1
            if rec.evidence_class != EVIDENCE_VERIFIED:
                rec.evidence_class = EVIDENCE_VERIFIED
        self._learn_direction(prev, cur, aid)
        moved = bool(frame_diff(prev, cur).any())
        self.stuck = 0 if moved else self.stuck + 1

    def _learn_direction(self, prev, cur, aid):
        if aid is None or aid in self.dirs:
            return
        hole = (prev != 0) & (cur == 0)
        fill = (prev == 0) & (cur != 0)
        if hole.any() and fill.any():
            hr, hc = np.nonzero(hole)
            fr, fc = np.nonzero(fill)
            dr = int(round(fr.mean() - hr.mean()))
            dc = int(round(fc.mean() - hc.mean()))
            self.dirs[aid] = (int(np.sign(dr)) if abs(dr) > 1 else 0,
                              int(np.sign(dc)) if abs(dc) > 1 else 0)
            self.player_color = int(np.bincount(prev[hole]).argmax())
            if self.knowledge.move is None:
                self.knowledge.move = {
                    "dirs": {}, "player_color": None,
                    "record": RuleRecord("move", "control")}
            self.knowledge.move["dirs"] = dict(self.dirs)
            self.knowledge.move["player_color"] = self.player_color
        else:
            self.probe_attempts[aid] = self.probe_attempts.get(aid, 0) + 1
            if self.probe_attempts[aid] >= PROBE_ATTEMPT_CAP:
                self.dirs[aid] = (0, 0)

    # ------------------------------------------------------------- replay
    def _replay_step(self, grid, frame, avail):
        seq = self.replay["actions"]
        idx = self.replay["idx"]
        if idx >= len(seq):
            self.replay = None
            return (self._click_policy(grid, frame, avail) if self.mode == "click"
                    else self._move_policy(grid, frame, avail))
        stored = seq[idx]
        aid, data = stored["aid"], dict(stored["data"])
        self.replay["idx"] = idx + 1
        self.pending = {
            "aid": aid, "data": data, "kind": "replay", "replay": True,
            "replay_idx": idx, "expected": stored["effect"], "predicted": None}
        return aid, data, {"strategy": "verified_replay",
                           "step": f"{idx + 1}/{len(seq)}"}

    def _replay_abort(self, cause):
        seqrec = self.knowledge.sequences.get(self.level)
        if seqrec is not None:
            seqrec["aborts"] = seqrec.get("aborts", 0) + 1
            if seqrec["aborts"] >= REPLAY_ABORT_STALE:
                seqrec["stale"] = True
                self.ledger.append("SEQUENCE_STALE", level=self.level,
                                   cause=cause)
        self.ledger.append("REPLAY_ABORT", level=self.level, cause=cause)
        self.replay = None

    # ------------------------------------------------------------- click games
    def _select_targets(self, sel, objs):
        if sel == "rarest_color":
            counts = {}
            for o in objs:
                counts[o["color"]] = counts.get(o["color"], 0) + o["size"]
            if not counts:
                return []
            rare = min(counts, key=lambda c: (counts[c], c))
            return [o for o in objs if o["color"] == rare]
        if sel == "smallest_object":
            return [min(objs, key=lambda o: (o["size"], o["bbox"]))]
        if sel == "corner_sample_color":
            corner = [o for o in objs
                      if o["bbox"][0] < CELL and o["bbox"][1] < CELL]
            if not corner:
                return []
            sample = corner[0]
            return [o for o in objs
                    if o["color"] == sample["color"] and o["id"] != sample["id"]]
        if sel == "largest_object":
            return [max(objs, key=lambda o: o["size"])]
        return []

    def _click_policy(self, grid, frame, avail, _depth=0):
        objs = find_objects(grid)
        if not objs:
            return self._no_signal_explore(grid, avail)
        active = [s for s in self.sel_order
                  if self.knowledge.rules.get(s) is None
                  or self.knowledge.rules[s].status != STATUS_QUARANTINED]
        if not active:
            active = list(self.sel_order)
        sel = active[0]
        if not self.queue:
            targets = self._select_targets(sel, objs)
            counts = {}
            for o in objs:
                counts[o["color"]] = counts.get(o["color"], 0) + 1
            targets = sorted(targets, key=lambda o: (counts[o["color"]], o["size"],
                                                     o["bbox"]))
            self.queue = targets
            self.queue_ptr = 0
        target = None
        while self.queue_ptr < len(self.queue):
            cand = self.queue[self.queue_ptr]
            cy, cx = cand["centroid"]
            cell = (int(cy) // CELL, int(cx) // CELL)
            if (cand["color"], cell) not in self.clicked_sig:
                target = cand
                break
            self.queue_ptr += 1
        if target is None:
            if _depth >= len(CLICK_SELECTORS):
                # v2.1 M2' FIX: design §4.2 — no-signal click exploration
                # must be SALIENCE-ORDERED. The received code's blind cell
                # sweeps lost lp85's level-1 clear in the 25-game head-to-
                # head (v2.0's proven fallback clicked salient objects).
                return self._salience_explore(grid, objs, avail)
            self._demote(sel, why="all candidates clicked, no progress")
            return self._click_policy(grid, frame, avail, _depth + 1)
        mask = np.zeros_like(grid, dtype=bool)
        for (r, c) in target["pixels"]:
            mask[r, c] = True
        predicted = prim_recolor_region(grid, mask, 0)
        cy, cx = target["centroid"]
        x = min(max(int(round(cx)), 0), 63)
        y = min(max(int(round(cy)), 0), 63)
        sig = (target["color"], (int(cy) // CELL, int(cx) // CELL))
        self.pending = {"aid": 6, "kind": "click", "rule": sel,
                        "predicted": predicted, "sig": sig,
                        "data": {"x": x, "y": y, "game_id": self.game_id}}
        self.queue_ptr += 1
        rec = self.knowledge.rule(sel)
        return 6, dict(self.pending["data"]), {
            "strategy": "program_synthesis_v3", "hypothesis": sel,
            "evidence_class": rec.evidence_class}

    # ------------------------------------------------------------- move games
    def _move_policy(self, grid, frame, avail):
        avail = [a for a in avail if a != 0]
        unprobed = [a for a in avail
                    if a not in self.dirs
                    and self.probe_attempts.get(a, 0) < PROBE_ATTEMPT_CAP]
        if unprobed:
            aid = unprobed[0]
            self.pending = {"aid": aid, "kind": "probe", "predicted": None,
                            "data": {"game_id": self.game_id}}
            return aid, {"game_id": self.game_id}, {
                "strategy": "program_synthesis_v3", "phase": "probe controls"}
        objs = find_objects(grid)
        if not objs or self.player_color is None:
            return self._no_signal_explore(grid, avail)
        players = [o for o in objs if o["color"] == self.player_color]
        others = [o for o in objs if o["color"] != self.player_color]
        if not players or not others:
            return self._no_signal_explore(grid, avail)
        player = players[0]
        goal = salience_rank(others)[0][0]
        py, px = player["centroid"]
        gy, gx = goal["centroid"]

        def dist_after(a):
            dr, dc = self.dirs.get(a, (0, 0))
            return abs(py + dr * CELL - gy) + abs(px + dc * CELL - gx)

        ordered = sorted(avail, key=dist_after)
        if self.stuck >= 2:
            ordered = ordered[1:] + ordered[:1]
            self.stuck = 0
        aid = ordered[0]
        dr, dc = self.dirs.get(aid, (0, 0))
        mask = np.zeros_like(grid, dtype=bool)
        for (r, c) in player["pixels"]:
            mask[r, c] = True
        predicted = prim_move_region(grid, mask, dr * CELL, dc * CELL)
        self.pending = {"aid": aid, "kind": "move", "predicted": predicted,
                        "data": {"game_id": self.game_id}}
        return aid, {"game_id": self.game_id}, {
            "strategy": "program_synthesis_v3",
            "controls": {str(k): v for k, v in self.dirs.items()}}

    # ------------------------------------------------------------- v2.1 explore
    def _salience_explore(self, grid, objs, avail):
        """M2' FIX (design §4.2): salience-ordered no-signal click
        exploration — v2.0's proven exhaustion fallback, restored, with
        graph bookkeeping (objects already tried at this state are
        skipped). These clicks are v2-baseline behaviour and deliberately
        do NOT count against EXPLORATION_CAP (worst case = v2.0 exactly).
        Blind cell enumeration remains the last resort."""
        ranked, _ = salience_rank(objs)
        h = self.graph.hash_grid(grid)
        tried = self.graph.transitions.get(h, {}) if h is not None else {}
        tried_cells = {k[1:] for k in tried
                       if isinstance(k, tuple) and k[0] == 6}
        for o in ranked:
            cy, cx = o["centroid"]
            cell = (int(cy) // CELL, int(cx) // CELL)
            if cell in tried_cells:
                continue
            if (o["color"], cell) in self.clicked_sig:
                continue
            x = min(max(int(round(cx)), 0), 63)
            y = min(max(int(round(cy)), 0), 63)
            self.pending = {"aid": 6, "kind": "explore", "predicted": None,
                            "sig": (o["color"], cell),
                            "data": {"x": x, "y": y, "game_id": self.game_id}}
            return 6, dict(self.pending["data"]), {
                "strategy": "salience_explore", "object": o["id"],
                "color": o["color"], "cell": cell,
                "note": "salience-ordered no-signal click"}
        # every salient object tried at this state: blind cell sweep
        return self._no_signal_explore(grid, avail)

    def _no_signal_explore(self, grid, avail):
        """Novelty-seeking fallback. Uses the state-action graph to pick
        an untried action at the current state, or the least-visited
        successor. Never returns None unless avail is empty."""
        if not avail:
            return 0, {"game_id": self.game_id}, {"note": "no actions"}

        def _rotate(note):
            # FIX (audit B2/B3): bounded fallback — never repeat one dead
            # action forever; rotate through the legal actions instead
            aid = int(avail[self._explore_rr % len(avail)])
            self._explore_rr += 1
            data = {"game_id": self.game_id}
            if aid == 6:
                data = {"x": 32, "y": 32, "game_id": self.game_id}
            self.pending = {"aid": aid, "kind": "explore",
                            "predicted": None, "data": data}
            return aid, dict(data), {
                "strategy": "state_graph_explore", "action": aid,
                "note": note}

        # FIX (audit B3): exploration budget cap — after EXPLORATION_CAP
        # consecutive explore actions without progress, degrade to the
        # bounded rotation for the rest of the level (design §4.3)
        if self.explore_blocked:
            return _rotate("exploration capped")

        h = self.graph.hash_grid(grid)
        choice = self.graph.novelty_action(h, avail)

        # Novelty might return (6, r, c) tuple for untried click cells
        if isinstance(choice, tuple) and len(choice) == 3 and choice[0] == 6:
            _, r, c = choice
            x = min(max(c * CELL + CELL // 2, 0), 63)
            y = min(max(r * CELL + CELL // 2, 0), 63)
            cell = (r, c)
            self.pending = {"aid": 6, "kind": "explore", "predicted": None,
                            "sig": cell,
                            "data": {"x": x, "y": y, "game_id": self.game_id}}
            self.explore_streak += 1
            if self.explore_streak >= EXPLORATION_CAP:
                self.explore_blocked = True
                self.ledger.append("EXPLORATION_CAPPED",
                                   streak=self.explore_streak)
            return 6, dict(self.pending["data"]), {
                "strategy": "state_graph_explore", "action": 6,
                "cell": cell, "state": h,
                "note": "novel click cell"}
        if choice is not None:
            aid = int(choice)
            data = {"game_id": self.game_id}
            if aid == 6:
                data = {"x": 32, "y": 32, "game_id": self.game_id}
            self.pending = {"aid": aid, "kind": "explore", "predicted": None,
                            "data": data}
            self.explore_streak += 1
            if self.explore_streak >= EXPLORATION_CAP:
                self.explore_blocked = True
                self.ledger.append("EXPLORATION_CAPPED",
                                   streak=self.explore_streak)
            return aid, dict(data), {
                "strategy": "state_graph_explore", "action": aid,
                "state": h, "note": "novel action"}

        # Graph exhausted at this state: bounded rotation
        return _rotate("graph exhausted")

    # ------------------------------------------------------------- helpers
    def _demote(self, sel, why):
        if sel in self.sel_order:
            self.sel_order.remove(sel)
            self.sel_order.append(sel)
        self.queue, self.queue_ptr = [], 0
        rec = self.knowledge.rule(sel)
        rec.rotations += 1
        if rec.rotations >= 2 and rec.success == 0:
            rec.status = STATUS_QUARANTINED
            self.ledger.append("RULE_QUARANTINED", rule=sel, cause=why)
        else:
            self.ledger.append("RULE_DEMOTED", rule=sel, cause=why)

    def _clear_level_state(self):
        self.queue, self.queue_ptr = [], 0
        self.clicked_sig = set()
        self.blacklist = set()
        self.pending = None
        self.stuck = 0
        self.level_actions = []
        self.level_verified = False
        self.replay = None
        # v2.1: recent buffer restarts fresh on new level; the exploration
        # cap also resets (design §4.3: cap is per level)
        self.explore_streak = 0
        self.explore_blocked = False
        if self.graph is not None:
            self.graph.clear_recent()

    def _grid(self, frame):
        g = merged_view(frame)
        if g is None:
            g = _grid_of(frame)
        return g

    def _available(self, frame):
        try:
            avail = list(getattr(frame, "available_actions", None) or [])
            return [int(a) for a in avail]
        except Exception:
            return []


# ============================================================================
# SECTION 6 — KIT-FACING AGENT (never-hang wrapper + protocol discipline)
# ============================================================================
# FIX (audit B1): the ARC-AGI-3-Agents framework drives agents via
# Agent.main(); when the framework is importable we subclass its base so
# main()/frames/counter all work (local `make play-local` and the Kaggle
# gateway). Standalone (no framework), the class duck-types the same
# surface. Restored verbatim from the M2-proven v2 wrapper.
try:
    from agents.agent import Agent as _KitAgent   # ARC-AGI-3-Agents framework
    KIT_FRAMEWORK = True
except Exception:
    _KitAgent = object
    KIT_FRAMEWORK = False


class ProgramSynthesisV2Agent(_KitAgent):
    """MyAgent for the ARC-AGI-3 Kaggle starter kit.

    - choose_action / is_done match the official Agent ABC surface; when the
      framework is present the inherited main() loop drives this class.
    - Every decide path is wrapped: any exception or missing field falls
      back to the cheapest legal action (RESET when required) — it can
      degrade, never hang, never raise.
    - Constructor accepts any kit-style arguments (game_id is sniffed from
      kwargs or the second positional argument when present).
    """

    MAX_ACTIONS = 80
    TIME_BUDGET_S = 1.0

    def __init__(self, *args, **kwargs):
        verbose = bool(kwargs.pop("verbose", False))
        self._kit = False
        if _KitAgent is not object:
            try:
                super().__init__(*args, **kwargs)
                self._kit = True
            except Exception:
                self._kit = False   # bad args for the kit signature: standalone
        if not self._kit:
            self.agent_name = (kwargs.get("agent_name")
                               or (args[2] if len(args) > 2 else None)
                               or "ProgramSynthesisV3")
        gid = (kwargs.get("game_id")
               or (args[1] if len(args) > 1 and isinstance(args[1], str) else "")
               or "")
        self.core = ProgramSynthesisV2Core(verbose=verbose)
        self.action_counter = 0     # bumped by the harness (official main())
        self._steps = 0             # internal mirror, bumped by choose_action
        self.begin_episode(gid or "")

    # -- lifecycle -----------------------------------------------------------
    def begin_episode(self, game_id):
        self.core.new_episode(game_id)

    # -- official ABC surface --------------------------------------------------
    def is_done(self, frames, latest_frame):
        try:
            frame = _pick_frame(frames, latest_frame)
            if _state_name(getattr(frame, "state", None)) == "WIN":
                return True
            return (self.action_counter >= self.MAX_ACTIONS
                    or self._steps >= self.MAX_ACTIONS)
        except Exception:
            return False

    def choose_action(self, frames, latest_frame):
        try:
            # kit main() passes the converted current observation as arg 2
            # (frames[0] is a dummy); runner mode passes frames[-1] itself
            frame = _pick_frame(frames, latest_frame)
            t0 = time.perf_counter()
            aid, data, reasoning = self.core.next_action(frame)
            action = self._make_action(aid, data, reasoning)
            self._steps += 1
            if time.perf_counter() - t0 > self.TIME_BUDGET_S:
                self.core.ledger.append(
                    "SLOW_DECISION",
                    seconds=round(time.perf_counter() - t0, 3))
            return action
        except Exception:
            self._steps += 1
            return self._fallback(frames, latest_frame)

    @staticmethod
    def _make_action(aid, data, reasoning):
        action = GameAction.from_id(int(aid))
        if data:
            try:
                action.set_data({k: (int(v) if k in ("x", "y") else v)
                                 for k, v in data.items()})
            except Exception:
                pass
        try:
            action.reasoning = reasoning
        except Exception:
            pass
        return action

    def _fallback(self, frames, latest_frame):
        try:
            state = _state_name(getattr(latest_frame, "state", None))
            if state in ("NOT_PLAYED", "GAME_OVER"):
                return self._make_action(0, {"game_id": self.core.game_id},
                                         {"strategy": "fallback_reset"})
            avail = []
            try:
                avail = [int(a) for a in
                         (getattr(latest_frame, "available_actions", None) or [])]
            except Exception:
                avail = []
            if not avail:
                avail = [1, 2, 3, 4, 6]
            aid = avail[0]
            if aid == 6:
                data = {"x": 32, "y": 32, "game_id": self.core.game_id}
            else:
                data = {"game_id": self.core.game_id}
            return self._make_action(aid, data, {"strategy": "safe_fallback"})
        except Exception:
            try:
                return GameAction.RESET
            except Exception:
                return None


MyAgent = ProgramSynthesisV2Agent
'''
AGENT_SHA256 = hashlib.sha256(AGENT_SOURCE.encode("utf-8")).hexdigest()

# embedding guards (checked at build time by the generator, re-checked here)
assert "'''" not in AGENT_SOURCE, "cannot embed: contains '''"
assert AGENT_SOURCE.endswith("\n"), "agent source must end with newline"
compile(AGENT_SOURCE, "program_synthesis_agent_v3.py", "exec")  # syntax gate

# import from a temp path so downstream cells get a live module without
# touching the artifacts cell 15 will emit
_tmp_path = os.path.join(tempfile.gettempdir(), "_psa_v3_builder_import.py")
with open(_tmp_path, "w") as f:
    f.write(AGENT_SOURCE)
_spec = importlib.util.spec_from_file_location(
    "program_synthesis_agent_v3", _tmp_path)
module = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(module)
MyAgent = module.MyAgent
assert MyAgent is not None, "agent module must expose MyAgent"

print(f"[c13] agent core frozen: {len(AGENT_SOURCE):,} bytes "
      f"sha256={AGENT_SHA256[:16]}… "
      f"kit_framework={bool(getattr(module, 'KIT_FRAMEWORK', False))}")


[c13] agent core frozen: 46,500 bytes sha256=ffce9e893f983c63… kit_framework=False


In [ ]:
# CELL 14 — OFFLINE TEST CELL (runs the frozen agent core from cell 13
# against adversarial inputs and the three embedded demo games; must all WIN
# before cell 15 emits anything). No LLM, no network.
import random
import time

# protocol names come from the frozen agent module loaded in cell 2
GameActionX = module.GameAction
GameStateX = module.GameState
FrameDataX = module.FrameData
GRID, CELL = module.GRID, module.CELL

# ---- compact demo environments (v1 cell3 port; same reset/step surface)
def _blank():
    return [[0] * GRID for _ in range(GRID)]

def _paint_block(grid, cell_r, cell_c, color, shape="full"):
    r0, c0 = cell_r * CELL, cell_c * CELL
    if shape == "full":
        for r in range(r0, r0 + CELL):
            for c in range(c0, c0 + CELL):
                grid[r][c] = color
    elif shape == "dot":
        for r in range(r0 + 2, r0 + 6):
            for c in range(c0 + 2, c0 + 6):
                grid[r][c] = color

class DemoGame:
    game_id, actions, n_levels, level_budget = "demo-base", [6], 3, 30

    def __init__(self, seed=0):
        self.rng = random.Random(seed)
        self.seed = seed
        self.state = GameStateX.NOT_PLAYED
        self.levels_completed = self.level_actions = self.total_actions = 0
        self._level = 0

    def reset(self):
        self._level = 0
        self.levels_completed = self.level_actions = self.total_actions = 0
        self.state = GameStateX.NOT_FINISHED
        self._build_level()
        return self._frame()

    def step(self, action, data=None, reasoning=None):
        if self.state == GameStateX.NOT_PLAYED:
            self.reset()
        if self.state == GameStateX.WIN:
            return self._frame()
        if action == GameActionX.RESET:
            self.state = GameStateX.NOT_FINISHED
            self.level_actions = 0
            self._build_level()
            return self._frame()
        if self.state == GameStateX.GAME_OVER:
            return self._frame()
        if action.value not in self.actions:
            return self._frame()
        self.level_actions += 1
        self.total_actions += 1
        advanced = self._on_action(action, data or {})
        if self.levels_completed >= self.n_levels:
            self.state = GameStateX.WIN
        elif self.level_actions >= self.level_budget:
            self.state = GameStateX.GAME_OVER
        else:
            self.state = GameStateX.NOT_FINISHED
        if advanced:
            self._level = self.levels_completed
            self.level_actions = 0
            if self.state != GameStateX.WIN:
                self._build_level()
        return self._frame()

    def _build_level(self):
        raise NotImplementedError

    def _on_action(self, action, data):
        raise NotImplementedError

    def _frame(self):
        return FrameDataX(
            game_id=self.game_id, frame=[self._render()],
            state=self.state, levels_completed=self.levels_completed,
            win_levels=self.n_levels, guid=f"{self.game_id}:{self.seed}",
            available_actions=list(self.actions))

class OddOneOutGame(DemoGame):
    game_id, n_levels, level_budget = "demo-odd-one-out", 4, 25

    def _build_level(self):
        cells = [(r, c) for r in (1, 3, 5) for c in (1, 3, 5)]
        self.rng.shuffle(cells)
        base = [1, 3, 4, 6, 7, 9]
        self.common_color = self.rng.choice(base)
        self.odd_cell = cells[0]
        mode = self.rng.choice(["color", "shape", "shape"])
        pool = [c for c in base if c != self.common_color]
        self.odd_color = self.rng.choice(pool) if mode == "color" \
            else self.common_color
        self.odd_shape = "dot" if mode == "shape" else "full"
        self.cells = cells[1:]

    def _on_action(self, action, data):
        x, y = int(data.get("x", -1)), int(data.get("y", -1))
        if (y // CELL, x // CELL) == self.odd_cell:
            self.levels_completed += 1
            return True
        return False

    def _render(self):
        g = _blank()
        for (r, c) in self.cells:
            _paint_block(g, r, c, self.common_color, "full")
        _paint_block(g, self.odd_cell[0], self.odd_cell[1],
                     self.odd_color, self.odd_shape)
        return g

class FollowSampleGame(DemoGame):
    game_id, n_levels, level_budget = "demo-follow-sample", 3, 25

    def _build_level(self):
        self.sample_color = self.rng.choice([1, 2, 3, 4, 6, 7, 9])
        free = [(r, c) for r in (2, 4, 6) for c in (2, 4, 6)]
        self.rng.shuffle(free)
        n_match = self.rng.randint(2, 3)
        self.targets = set(free[:n_match])
        palette = [c for c in [1, 2, 3, 4, 6, 7, 9]
                   if c != self.sample_color]
        self.others = {cell: self.rng.choice(palette)
                       for cell in free[n_match:n_match + 6]}

    def _on_action(self, action, data):
        x, y = int(data.get("x", -1)), int(data.get("y", -1))
        cr, cc = y // CELL, x // CELL
        if (cr, cc) in self.targets:
            self.targets.discard((cr, cc))
            if not self.targets:
                self.levels_completed += 1
                return True
        return False

    def _render(self):
        g = _blank()
        _paint_block(g, 0, 0, self.sample_color, "full")
        for (r, c) in self.targets:
            _paint_block(g, r, c, self.sample_color, "full")
        for (r, c), col in self.others.items():
            _paint_block(g, r, c, col, "full")
        return g

class ReachGoalGame(DemoGame):
    game_id, n_levels, level_budget = "demo-reach-goal", 3, 40
    actions = [1, 2, 3, 4]
    DELTA = {1: (-1, 0), 2: (1, 0), 3: (0, -1), 4: (0, 1)}

    def _build_level(self):
        while True:
            self.player = (self.rng.randint(0, 3),
                           self.rng.randint(0, 7))
            self.goal = (self.rng.randint(4, 7), self.rng.randint(0, 7))
            walls = {(self.rng.randint(1, 6), self.rng.randint(1, 6))
                     for _ in range(self._level + 2)}
            walls -= {self.player, self.goal}
            if self._greedy_path_clear(self.player, self.goal, walls):
                self.walls = walls
                return

    def _greedy_path_clear(self, start, goal, walls):
        r, c = start
        for _ in range(24):
            if (r, c) == goal:
                return True
            dr = 1 if goal[0] > r else (-1 if goal[0] < r else 0)
            dc = 1 if goal[1] > c else (-1 if goal[1] < c else 0)
            if dr and (r + dr, c) not in walls and 0 <= r + dr < 8:
                r += dr
            elif dc and (r, c + dc) not in walls and 0 <= c + dc < 8:
                c += dc
            else:
                return False
        return (r, c) == goal

    def _on_action(self, action, data):
        dr, dc = self.DELTA[action.value]
        nr, nc = self.player[0] + dr, self.player[1] + dc
        if 0 <= nr < 8 and 0 <= nc < 8 and (nr, nc) not in self.walls:
            self.player = (nr, nc)
        if self.player == self.goal:
            self.levels_completed += 1
            return True
        return False

    def _render(self):
        g = _blank()
        for (r, c) in self.walls:
            _paint_block(g, r, c, 5, "full")
        _paint_block(g, self.goal[0], self.goal[1], 2, "full")
        _paint_block(g, self.player[0], self.player[1], 8, "full")
        return g

def run_episode(env, agent, game_id, max_actions=80):
    """Framework-free mirror of agents.agent.Agent.main()."""
    cap = max_actions or agent.MAX_ACTIONS
    agent.begin_episode(game_id)
    frames = [env.reset()]
    steps = 0
    while not agent.is_done(frames, frames[-1]) and steps < cap:
        action = agent.choose_action(frames, frames[-1])
        data = action.action_data.model_dump()
        frame = env.step(action, data=data,
                         reasoning=getattr(action, "reasoning", None))
        if frame is None:
            break
        frames.append(frame)
        agent.action_counter += 1
        steps += 1
    last = frames[-1]
    return {"won": module._state_name(last.state) == "WIN",
            "levels": last.levels_completed,
            "win_levels": last.win_levels,
            "actions": steps}

DEMO = [OddOneOutGame, FollowSampleGame, ReachGoalGame]
OFFLINE_TEST_RESULTS = {"adversarial": None, "games": {}}

# --- adversarial: the agent must survive garbage input without raising ------
try:
    _a = MyAgent()
    _t0 = time.perf_counter()
    for _bad in (None, object(), 12345, "frame", [None, None]):
        _a.choose_action([_bad], _bad)
        _a.is_done([_bad], _bad)
    _a.choose_action([], None)
    _a.is_done([], None)
    _kw = MyAgent(card_id="c", game_id="demo-adversarial", agent_name="probe",
                  record=False)
    _kw.choose_action([], None)
    _dt = (time.perf_counter() - _t0) * 1000
    assert _dt < 5000, f"adversarial smoke too slow: {_dt:.0f} ms"
    OFFLINE_TEST_RESULTS["adversarial"] = round(_dt, 2)
    print(f"[c14] PASS — adversarial smoke ({_dt:.1f} ms, no exceptions)")
except Exception as _exc:
    raise RuntimeError(f"OFFLINE TEST FAILED (adversarial): {_exc!r}")

# --- demo games: fresh win, same-seed replay (<= fresh), new-seed abort win -
for _gcls in DEMO:
    _gid = _gcls.game_id
    _r1 = run_episode(_gcls(seed=0), MyAgent(), _gid, 80)
    _r2 = run_episode(_gcls(seed=0), MyAgent(), _gid, 80)   # replay path
    _r3 = run_episode(_gcls(seed=7), MyAgent(), _gid, 80)   # abort path
    assert _r1["won"] and _r2["won"] and _r3["won"], \
        f"OFFLINE TEST FAILED: {_gid} {_r1} {_r2} {_r3}"
    assert _r2["actions"] <= _r1["actions"], \
        f"replay did not help on {_gid}: ep2={_r2['actions']} ep1={_r1['actions']}"
    OFFLINE_TEST_RESULTS["games"][_gid] = {
        "fresh_actions": _r1["actions"],
        "replay_actions": _r2["actions"],
        "newseed_actions": _r3["actions"],
        "levels": f"{_r1['levels']}/{_r1['win_levels']}",
    }
    print(f"[c14] PASS — {_gid}: WIN {_r1['levels']}/{_r1['win_levels']} "
          f"(fresh {_r1['actions']}a, replay {_r2['actions']}a, "
          f"new-seed {_r3['actions']}a)")

OFFLINE_TEST_RESULTS["verdict"] = "PASS"
print("[c14] ALL OFFLINE TESTS PASS — safe to emit artifacts")


[c14] PASS — adversarial smoke (0.1 ms, no exceptions)
[c14] PASS — demo-odd-one-out: WIN 4/4 (fresh 4a, replay 4a, new-seed 10a)
[c14] PASS — demo-follow-sample: WIN 3/3 (fresh 10a, replay 7a, new-seed 12a)
[c14] PASS — demo-reach-goal: WIN 3/3 (fresh 21a, replay 21a, new-seed 35a)
[c14] ALL OFFLINE TESTS PASS — safe to emit artifacts


In [ ]:
# CELL 15 — LEGACY V3 EMITTER (writes the frozen baseline artifacts into the working directory:
# the v3 agent file, the competition kernel notebook, and build_manifest.json).
# The kernel is assembled from the frozen cell templates below + the frozen
# agent source from cell 13 — byte-identical to the M2′-validated
# ARC_AGI_3_Kagglea2.ipynb.
import hashlib
import json
import os
import platform
import time

# frozen kernel cell templates (JSON-encoded at build time; values are the
# exact v2_dev/kernel/*.py + *.md sources, with __AGENT_SOURCE__ as the
# injection point in c2)
KERNEL_CELLS = {
    "md0_title.md": "# ARC-AGI-3 \u2014 ProgramSynthesis v2.1 (competition kernel)\n\n**Generated by the Phase 4 v2.1 build (M2\u2032).** The agent source is\nfrozen into this notebook as the constant `AGENT_SOURCE` (cell 2) and is\nbyte-identical to `program_synthesis_agent_v3.py`.\n\n## Run modes\n\n| Mode | Where | What happens | Output |\n|---|---|---|---|\n| **Competition rerun** | Kaggle scoring | cell 4 runs the official ARC-AGI-3-Agents framework against the scoring gateway | `/kaggle/working/submission.parquet` |\n| **Commit / local** | Kaggle editor, Colab, laptop | cell 5 plays three embedded demo games (must all WIN) and writes a schema-verified parquet | `working/submission.parquet` |\n\nCell 1 (offline wheel install) and cell 3 (starter-kit `make setup` +\n`make play-local` pipeline) detect their environment and skip automatically \u2014\nthe notebook is safe to **Run All** everywhere. To skip the local pipeline\nmanually set `SKIP_LOCAL_PIPELINE=1`; game subset via `PIPELINE_GAMES`\n(default `ls20,vc33`).\n\n## Hard boundary (physics)\n\nThe Kaggle scoring runtime has **no internet, no app access, no DB access**.\nEverything the agent needs is baked into this notebook at build time: the\nagent is pure Python with zero model calls, and its ARC-AGI-3 protocol layer\n(`GameAction` / `GameState` / `FrameData`) is embedded in the agent file\nitself.\n",
    "c1_wheels.py": "# CELL 1 \u2014 Kaggle-only: install the offline wheels from the competition\n# dataset (no internet: --no-index means the pip resolver never goes online).\n# Everywhere else this cell detects it is not on Kaggle and skips harmlessly.\nimport os\nimport subprocess\n\n\ndef _sh(cmd, timeout=900):\n    r = subprocess.run(cmd, shell=True, capture_output=True, text=True,\n                       timeout=timeout)\n    return r.returncode, (r.stdout or \"\") + (r.stderr or \"\")\n\n\nif os.path.isdir(\"/kaggle/input\"):\n    rc, out = _sh(\n        \"pip install --no-index --find-links \"\n        \"/kaggle/input/competitions/arc-prize-2026-arc-agi-3/arc_agi_3_wheels \"\n        \"arc-agi python-dotenv\")\n    print(f\"[wheels] pip rc={rc}\")\n    print(out[-600:])\nelse:\n    print(\"[wheels] not on Kaggle \u2014 skipping offline wheel install \"\n          \"(the agent's protocol shim covers standalone runs)\")\n",
    "c2_agent.py": "# CELL 2 \u2014 Freeze the agent. In every mode this writes the byte-identical\n# agent source to /tmp/my_agent.py (the path the competition rerun picks up),\n# imports it, and runs a quick adversarial smoke test. Outside the competition\n# rerun it additionally drops `program_synthesis_agent_v3.py` and\n# `agent/my_agent.py` into the working directory so the local pipeline and the\n# user can consume them.\nimport os\nimport importlib.util\nimport time\n\nAGENT_SOURCE = __AGENT_SOURCE__\n\nRERUN = os.environ.get(\"KAGGLE_IS_COMPETITION_RERUN\") is not None\nWORKDIR = os.path.join(os.getcwd(), \"working\")\nos.makedirs(WORKDIR, exist_ok=True)\n\n# Always: /tmp/my_agent.py \u2014 the only agent path the competition rerun uses.\nwith open(\"/tmp/my_agent.py\", \"w\") as f:\n    f.write(AGENT_SOURCE)\n\nif not RERUN:\n    with open(os.path.join(WORKDIR, \"program_synthesis_agent_v3.py\"), \"w\") as f:\n        f.write(AGENT_SOURCE)\n    os.makedirs(os.path.join(WORKDIR, \"agent\"), exist_ok=True)\n    with open(os.path.join(WORKDIR, \"agent\", \"my_agent.py\"), \"w\") as f:\n        f.write(AGENT_SOURCE)\n\nspec = importlib.util.spec_from_file_location(\n    \"program_synthesis_agent_v3\",\n    \"/tmp/my_agent.py\")\nmodule = importlib.util.module_from_spec(spec)\nspec.loader.exec_module(module)\nMyAgent = module.MyAgent\nassert MyAgent is not None, \"agent module must expose class MyAgent\"\n\n# --- adversarial smoke: the agent must survive garbage input without raising\n_smoke_ok = True\ntry:\n    a = MyAgent()\n    t0 = time.perf_counter()\n    for bad in (None, object(), 12345):\n        try:\n            a.choose_action([bad], bad)\n            a.is_done([bad], bad)\n        except Exception:\n            raise\n    a.choose_action([], None)\n    assert a.is_done([], None) in (True, False)\n    dt = (time.perf_counter() - t0) * 1000\n    assert dt < 5000, f\"smoke too slow: {dt:.0f} ms\"\nexcept Exception as exc:  # pragma: no cover\n    _smoke_ok = False\n    raise RuntimeError(f\"agent smoke test FAILED: {exc!r}\")\n\nprint(f\"[agent] MyAgent loaded ({len(AGENT_SOURCE)} bytes, \"\n      f\"kit={bool(getattr(module, 'KIT_FRAMEWORK', False))}) \"\n      f\"\u2014 smoke ok ({dt:.1f} ms)\")\n",
    "c3_pipeline.py": "# CELL 3 \u2014 Local developer pipeline (auto-skips on Kaggle and in the\n# competition rerun; set SKIP_LOCAL_PIPELINE=1 to skip manually).\n# This is the canonical ARC-AGI-3-Kaggle-Starter pipeline used by the\n# v1 submission: clone the starter, install its venv, drop our agent into\n# agent/my_agent.py, run `make play-local` (MPLBACKEND=Agg) on a small\n# game set. It must never break Run All.\nimport os\nimport shutil\nimport subprocess\n\nPIPELINE = {\"clone\": False, \"setup\": False, \"play_local\": False, \"output\": \"\"}\nPIPELINE_GAMES = os.environ.get(\"PIPELINE_GAMES\", \"ls20,vc33\")\nON_KAGGLE = os.path.isdir(\"/kaggle/input\")\nRERUN = os.environ.get(\"KAGGLE_IS_COMPETITION_RERUN\") is not None\nLOCAL_DIR = os.getcwd()\n# cell 2 writes the emitted agent into <cwd>/working/\nAGENT_FILE = os.path.join(LOCAL_DIR, \"working\",\n                          \"program_synthesis_agent_v3.py\")\nif ON_KAGGLE or RERUN or os.environ.get(\"SKIP_LOCAL_PIPELINE\"):\n    print(\"[pipeline] skipped (Kaggle / rerun / SKIP_LOCAL_PIPELINE=1)\")\nelse:\n    def _sh(cmd, timeout):\n        r = subprocess.run(cmd, shell=True, capture_output=True, text=True,\n                           timeout=timeout)\n        return r.returncode, (r.stdout or \"\") + (r.stderr or \"\")\n\n    starter = os.path.join(LOCAL_DIR, \"ARC-AGI-3-Kaggle-Starter\")\n    try:\n        if not os.path.isdir(starter):\n            rc, out = _sh(\n                \"git clone --depth 1 \"\n                \"https://github.com/arcprize/ARC-AGI-3-Kaggle-Starter.git \"\n                f\"{starter}\", 300)\n            if rc != 0:\n                raise RuntimeError(f\"clone failed: {out[-300:]}\")\n        else:\n            print(\"[pipeline] starter already cloned\")\n        PIPELINE[\"clone\"] = True\n\n        # agent \u2192 starter/agent/my_agent.py  (same cp as the manual pipeline)\n        shutil.copyfile(AGENT_FILE,\n                        os.path.join(starter, \"agent\", \"my_agent.py\"))\n\n        # venv + deps (Makefile default is python3.12; fall back to python3)\n        py = \"python3.12\" if shutil.which(\"python3.12\") else \"python3\"\n        rc, out = _sh(f\"make -C {starter} setup PYTHON={py}\", 900)\n        if rc != 0:\n            raise RuntimeError(f\"setup failed: {out[-400:]}\")\n        PIPELINE[\"setup\"] = True\n\n        # offline harness run on a small game set\n        rc, out = _sh(\n            f\"make -C {starter} play-local GAME={PIPELINE_GAMES} \"\n            \"MAX_STEPS=200 MPLBACKEND=Agg\", 1800)\n        PIPELINE[\"output\"] = out\n        if rc != 0 or \"SUMMARY\" not in out:\n            raise RuntimeError(f\"play-local failed: {out[-400:]}\")\n        PIPELINE[\"play_local\"] = True\n        # echo the play-local summary + scorecard for the user (the per-step\n        # action log can be hundreds of lines, so we show the scorecard\n        # section plus a one-line action histogram)\n        lines = out.splitlines()\n        try:\n            si = next(i for i, l in enumerate(lines)\n                      if \"SUMMARY\" in l)\n            tail = lines[si:si + 12]\n        except StopIteration:\n            tail = [l for l in lines if l.strip()][-8:]\n        hist = {}\n        for l in lines:\n            for a in (\"ACTION1\", \"ACTION2\", \"ACTION3\", \"ACTION4\", \"ACTION5\",\n                      \"ACTION6\", \"ACTION7\", \"RESET\"):\n                if f\"- {a}:\" in l:\n                    hist[a] = hist.get(a, 0) + 1\n        print(\"[pipeline] OK \u2014 clone/setup/play-local on \"\n              f\"[{PIPELINE_GAMES}]\")\n        for l in tail:\n            print(\"  | \" + l)\n        print(f\"  | action histogram: \"\n              f\"{dict(sorted(hist.items())) or 'none'}\")\n    except Exception as exc:\n        print(f\"[pipeline] FAILED (non-fatal): {exc}\")\n",
    "c4_rerun.py": "# CELL 4 \u2014 Competition rerun (the cell Kaggle executes for scoring).\n# Follows the ARC-AGI-3-Kaggle-Starter rerun contract exactly:\n# 1. copy the ARC-AGI-3-Agents framework from the competition dataset\n# 2. install our agent as agents/templates/my_agent.py + register it\n# 3. write the .env pointing at the scoring gateway\n# 4. run `python main.py --agent myagent` \u2014 the gateway records every\n#    action and emits /kaggle/working/submission.parquet\nimport os\nimport subprocess\n\nif not os.environ.get(\"KAGGLE_IS_COMPETITION_RERUN\"):\n    print(\"[rerun] not a competition rerun \u2014 skipping\")\nelse:\n    COMP = \"/kaggle/input/competitions/arc-prize-2026-arc-agi-3\"\n    if not os.path.isdir(COMP):\n        print(\"[rerun] competition dataset not mounted \u2014 DRY RUN ONLY, \"\n              \"skipping gateway call\")\n    else:\n        def _sh(cmd, timeout):\n            r = subprocess.run(cmd, shell=True, capture_output=True,\n                               text=True, timeout=timeout)\n            return r.returncode, (r.stdout or \"\") + (r.stderr or \"\")\n\n        # wait for the scoring gateway to come up (retries ~10 minutes)\n        _sh(\"curl --fail --retry 999 --retry-all-errors --retry-delay 5 \"\n            \"--retry-max-time 600 http://gateway:8001/api/games\", 700)\n\n        WORK = \"/kaggle/working\"\n        rc, out = _sh(f\"cp -r {COMP}/ARC-AGI-3-Agents {WORK}/\", 120)\n        if rc != 0:\n            raise RuntimeError(f\"framework copy failed: {out[-300:]}\")\n        rc, out = _sh(f\"cp /tmp/my_agent.py {WORK}/ARC-AGI-3-Agents/\"\n                      \"agents/templates/my_agent.py\", 30)\n        if rc != 0:\n            raise RuntimeError(f\"agent copy failed: {out[-300:]}\")\n\n        # slim agents/__init__.py registering exactly our agent\n        AGENTS_INIT = '''from agents.agent import Agent, Playback\nfrom agents.swarm import Swarm\nfrom agents.templates.random_agent import Random\nfrom agents.templates.my_agent import MyAgent\nfrom dotenv import load_dotenv\n\nload_dotenv()\n\nagents = {\"random\": Random, \"myagent\": MyAgent}\n'''\n        with open(f\"{WORK}/ARC-AGI-3-Agents/agents/__init__.py\", \"w\") as f:\n            f.write(AGENTS_INIT)\n\n        ENV_TEXT = \"\"\"SCHEME=http\nHOST=gateway\nPORT=8001\nARC_API_KEY=test-key-123\nARC_BASE_URL=http://gateway:8001/\nOPERATION_MODE=online\nENVIRONMENTS_DIR=\nRECORDINGS_DIR=/kaggle/working/server_recording\n\"\"\"\n        with open(f\"{WORK}/ARC-AGI-3-Agents/.env\", \"w\") as f:\n            f.write(ENV_TEXT)\n\n        rc, out = _sh(\n            f\"cd {WORK}/ARC-AGI-3-Agents && \"\n            \"MPLBACKEND=agg python main.py --agent myagent\", 3600)\n        print(out[-3000:])\n        if rc != 0:\n            raise RuntimeError(f\"main.py failed rc={rc}\")\n        print(\"[rerun] completed \u2014 gateway wrote submission.parquet\")\n",
    "c5_parquet.py": "# CELL 5 \u2014 Offline self-test + submission.parquet (skipped during the\n# competition rerun: there the gateway owns scoring and writes the parquet).\n# Otherwise: play three embedded demo games end-to-end with the frozen agent,\n# require all three to WIN, then write submission.parquet with the official\n# schema [row_id, game_id, end_of_game, score] and read it back.\nimport os\nimport random\nimport time\n\nRERUN = os.environ.get(\"KAGGLE_IS_COMPETITION_RERUN\") is not None\n\nif RERUN:\n    print(\"[selftest] competition rerun \u2014 gateway owns submission.parquet, \"\n          \"skipping\")\nelse:\n    # protocol names come from the frozen agent module loaded in cell 2\n    GameActionX = module.GameAction\n    GameStateX = module.GameState\n    FrameDataX = module.FrameData\n    GRID, CELL = module.GRID, module.CELL\n\n    # ---- compact demo environments (v1 cell3 port; same reset/step surface)\n    def _blank():\n        return [[0] * GRID for _ in range(GRID)]\n\n    def _paint_block(grid, cell_r, cell_c, color, shape=\"full\"):\n        r0, c0 = cell_r * CELL, cell_c * CELL\n        if shape == \"full\":\n            for r in range(r0, r0 + CELL):\n                for c in range(c0, c0 + CELL):\n                    grid[r][c] = color\n        elif shape == \"dot\":\n            for r in range(r0 + 2, r0 + 6):\n                for c in range(c0 + 2, c0 + 6):\n                    grid[r][c] = color\n\n    class DemoGame:\n        game_id, actions, n_levels, level_budget = \"demo-base\", [6], 3, 30\n\n        def __init__(self, seed=0):\n            self.rng = random.Random(seed)\n            self.seed = seed\n            self.state = GameStateX.NOT_PLAYED\n            self.levels_completed = self.level_actions = self.total_actions = 0\n            self._level = 0\n\n        def reset(self):\n            self._level = 0\n            self.levels_completed = self.level_actions = self.total_actions = 0\n            self.state = GameStateX.NOT_FINISHED\n            self._build_level()\n            return self._frame()\n\n        def step(self, action, data=None, reasoning=None):\n            if self.state == GameStateX.NOT_PLAYED:\n                self.reset()\n            if self.state == GameStateX.WIN:\n                return self._frame()\n            if action == GameActionX.RESET:\n                self.state = GameStateX.NOT_FINISHED\n                self.level_actions = 0\n                self._build_level()\n                return self._frame()\n            if self.state == GameStateX.GAME_OVER:\n                return self._frame()\n            if action.value not in self.actions:\n                return self._frame()\n            self.level_actions += 1\n            self.total_actions += 1\n            advanced = self._on_action(action, data or {})\n            if self.levels_completed >= self.n_levels:\n                self.state = GameStateX.WIN\n            elif self.level_actions >= self.level_budget:\n                self.state = GameStateX.GAME_OVER\n            else:\n                self.state = GameStateX.NOT_FINISHED\n            if advanced:\n                self._level = self.levels_completed\n                self.level_actions = 0\n                if self.state != GameStateX.WIN:\n                    self._build_level()\n            return self._frame()\n\n        def _build_level(self):\n            raise NotImplementedError\n\n        def _on_action(self, action, data):\n            raise NotImplementedError\n\n        def _frame(self):\n            return FrameDataX(\n                game_id=self.game_id, frame=[self._render()],\n                state=self.state, levels_completed=self.levels_completed,\n                win_levels=self.n_levels, guid=f\"{self.game_id}:{self.seed}\",\n                available_actions=list(self.actions))\n\n    class OddOneOutGame(DemoGame):\n        game_id, n_levels, level_budget = \"demo-odd-one-out\", 4, 25\n\n        def _build_level(self):\n            cells = [(r, c) for r in (1, 3, 5) for c in (1, 3, 5)]\n            self.rng.shuffle(cells)\n            base = [1, 3, 4, 6, 7, 9]\n            self.common_color = self.rng.choice(base)\n            self.odd_cell = cells[0]\n            mode = self.rng.choice([\"color\", \"shape\", \"shape\"])\n            pool = [c for c in base if c != self.common_color]\n            self.odd_color = self.rng.choice(pool) if mode == \"color\" \\\n                else self.common_color\n            self.odd_shape = \"dot\" if mode == \"shape\" else \"full\"\n            self.cells = cells[1:]\n\n        def _on_action(self, action, data):\n            x, y = int(data.get(\"x\", -1)), int(data.get(\"y\", -1))\n            if (y // CELL, x // CELL) == self.odd_cell:\n                self.levels_completed += 1\n                return True\n            return False\n\n        def _render(self):\n            g = _blank()\n            for (r, c) in self.cells:\n                _paint_block(g, r, c, self.common_color, \"full\")\n            _paint_block(g, self.odd_cell[0], self.odd_cell[1],\n                         self.odd_color, self.odd_shape)\n            return g\n\n    class FollowSampleGame(DemoGame):\n        game_id, n_levels, level_budget = \"demo-follow-sample\", 3, 25\n\n        def _build_level(self):\n            self.sample_color = self.rng.choice([1, 2, 3, 4, 6, 7, 9])\n            free = [(r, c) for r in (2, 4, 6) for c in (2, 4, 6)]\n            self.rng.shuffle(free)\n            n_match = self.rng.randint(2, 3)\n            self.targets = set(free[:n_match])\n            palette = [c for c in [1, 2, 3, 4, 6, 7, 9]\n                       if c != self.sample_color]\n            self.others = {cell: self.rng.choice(palette)\n                           for cell in free[n_match:n_match + 6]}\n\n        def _on_action(self, action, data):\n            x, y = int(data.get(\"x\", -1)), int(data.get(\"y\", -1))\n            cr, cc = y // CELL, x // CELL\n            if (cr, cc) in self.targets:\n                self.targets.discard((cr, cc))\n                if not self.targets:\n                    self.levels_completed += 1\n                    return True\n            return False\n\n        def _render(self):\n            g = _blank()\n            _paint_block(g, 0, 0, self.sample_color, \"full\")\n            for (r, c) in self.targets:\n                _paint_block(g, r, c, self.sample_color, \"full\")\n            for (r, c), col in self.others.items():\n                _paint_block(g, r, c, col, \"full\")\n            return g\n\n    class ReachGoalGame(DemoGame):\n        game_id, n_levels, level_budget = \"demo-reach-goal\", 3, 40\n        actions = [1, 2, 3, 4]\n        DELTA = {1: (-1, 0), 2: (1, 0), 3: (0, -1), 4: (0, 1)}\n\n        def _build_level(self):\n            while True:\n                self.player = (self.rng.randint(0, 3),\n                               self.rng.randint(0, 7))\n                self.goal = (self.rng.randint(4, 7), self.rng.randint(0, 7))\n                walls = {(self.rng.randint(1, 6), self.rng.randint(1, 6))\n                         for _ in range(self._level + 2)}\n                walls -= {self.player, self.goal}\n                if self._greedy_path_clear(self.player, self.goal, walls):\n                    self.walls = walls\n                    return\n\n        def _greedy_path_clear(self, start, goal, walls):\n            r, c = start\n            for _ in range(24):\n                if (r, c) == goal:\n                    return True\n                dr = 1 if goal[0] > r else (-1 if goal[0] < r else 0)\n                dc = 1 if goal[1] > c else (-1 if goal[1] < c else 0)\n                if dr and (r + dr, c) not in walls and 0 <= r + dr < 8:\n                    r += dr\n                elif dc and (r, c + dc) not in walls and 0 <= c + dc < 8:\n                    c += dc\n                else:\n                    return False\n            return (r, c) == goal\n\n        def _on_action(self, action, data):\n            dr, dc = self.DELTA[action.value]\n            nr, nc = self.player[0] + dr, self.player[1] + dc\n            if 0 <= nr < 8 and 0 <= nc < 8 and (nr, nc) not in self.walls:\n                self.player = (nr, nc)\n            if self.player == self.goal:\n                self.levels_completed += 1\n                return True\n            return False\n\n        def _render(self):\n            g = _blank()\n            for (r, c) in self.walls:\n                _paint_block(g, r, c, 5, \"full\")\n            _paint_block(g, self.goal[0], self.goal[1], 2, \"full\")\n            _paint_block(g, self.player[0], self.player[1], 8, \"full\")\n            return g\n\n    def run_episode(env, agent, game_id, max_actions=80):\n        \"\"\"Framework-free mirror of agents.agent.Agent.main().\"\"\"\n        cap = max_actions or agent.MAX_ACTIONS\n        agent.begin_episode(game_id)\n        frames = [env.reset()]\n        steps = 0\n        while not agent.is_done(frames, frames[-1]) and steps < cap:\n            action = agent.choose_action(frames, frames[-1])\n            data = action.action_data.model_dump()\n            frame = env.step(action, data=data,\n                             reasoning=getattr(action, \"reasoning\", None))\n            if frame is None:\n                break\n            frames.append(frame)\n            agent.action_counter += 1\n            steps += 1\n        last = frames[-1]\n        return {\"won\": module._state_name(last.state) == \"WIN\",\n                \"levels\": last.levels_completed,\n                \"win_levels\": last.win_levels,\n                \"actions\": steps}\n\n    DEMO = [OddOneOutGame, FollowSampleGame, ReachGoalGame]\n    results = {}\n    for gcls in DEMO:\n        t0 = time.perf_counter()\n        r = run_episode(gcls(seed=0), MyAgent(), gcls.game_id, 80)\n        r[\"seconds\"] = round(time.perf_counter() - t0, 2)\n        results[gcls.game_id] = r\n        print(f\"[selftest] {gcls.game_id}: \"\n              f\"{'WIN' if r['won'] else 'FAIL'} \"\n              f\"({r['levels']}/{r['win_levels']} levels, \"\n              f\"{r['actions']} actions, {r['seconds']}s)\")\n\n    losses = [g for g, r in results.items() if not r[\"won\"]]\n    if losses:\n        raise SystemExit(f\"SELF-TEST FAILED \u2014 agent lost: {losses}\")\n\n    # ---- submission.parquet (official schema; dummy rows in commit mode)\n    import pandas as pd\n    try:\n        import pyarrow  # noqa: F401  (parquet engine)\n    except ImportError as exc:\n        raise SystemExit(\n            \"pyarrow missing \u2014 needed to write submission.parquet \"\n            f\"(pip install pyarrow). {exc!r}\")\n\n    rows = [{\"row_id\": gid, \"game_id\": gid, \"end_of_game\": True,\n             \"score\": r[\"levels\"]}\n            for gid, r in results.items()]\n    SUB_PATH = os.path.join(WORKDIR, \"submission.parquet\")\n    pd.DataFrame(rows, columns=[\"row_id\", \"game_id\", \"end_of_game\",\n                                \"score\"]).to_parquet(SUB_PATH)\n\n    back = pd.read_parquet(SUB_PATH)\n    assert list(back.columns) == [\"row_id\", \"game_id\", \"end_of_game\",\n                                  \"score\"], back.columns\n    assert len(back) == len(results)\n    print(f\"[selftest] 3/3 WIN \u2014 submission.parquet written \"\n          f\"({len(back)} rows, schema ok): {SUB_PATH}\")\n",
    "md6_summary.md": "## Summary\n\n- **Cell 1** \u2014 offline wheel install (Kaggle only, `--no-index`)\n- **Cell 2** \u2014 agent frozen as `AGENT_SOURCE` \u2192 `/tmp/my_agent.py` (+\n  `working/` copies outside rerun), import + adversarial smoke\n- **Cell 3** \u2014 local starter-kit pipeline: clone \u2192 `make setup` \u2192\n  `MPLBACKEND=Agg make play-local` (auto-skip on Kaggle/rerun)\n- **Cell 4** \u2014 competition rerun: framework + gateway + `main.py --agent\n  myagent` \u2192 gateway emits `submission.parquet`\n- **Cell 5** \u2014 offline self-test: 3 demo games must all WIN \u2192\n  `working/submission.parquet` (schema `[row_id, game_id, end_of_game,\n  score]`, read back and verified)\n\n**Guarantee:** Run All never hangs \u2014 every shell step is `subprocess.run`\nwith a timeout, every mode-guard prints why it skipped, and the only hard\nfailure modes are the self-test losing a demo game or a missing parquet\nengine, both of which should abort the commit loudly.\n",
}

OUT_DIR = os.getcwd()          # Kaggle: /kaggle/working — local: cwd
ARTIFACTS = {}


def _sha256_bytes(b):
    return hashlib.sha256(b).hexdigest()


def _write(name, data):
    path = os.path.join(OUT_DIR, name)
    _b = data.encode("utf-8")
    with open(path, "wb") as f:
        f.write(_b)
    ARTIFACTS[name] = {"bytes": len(_b), "sha256": _sha256_bytes(_b)}
    return path


# --- artifact 1: the agent --------------------------------------------------
agent_path = _write("program_synthesis_agent_v3.py", AGENT_SOURCE)

# --- artifact 2: the competition kernel notebook ----------------------------
# assembly identical to v2_dev/build_kernel_notebook.py (M2) so the bytes
# match the M2′-validated deliverable exactly
_c2 = KERNEL_CELLS["c2_agent.py"].replace(
    "__AGENT_SOURCE__", "r'''" + AGENT_SOURCE + "'''")
assert "__AGENT_SOURCE__" not in _c2


def _code_cell(src, cid):
    return {"cell_type": "code", "id": cid, "execution_count": None,
            "metadata": {}, "outputs": [], "source": src.splitlines(keepends=True)}


def _md_cell(src, cid):
    return {"cell_type": "markdown", "id": cid, "metadata": {},
            "source": src.splitlines(keepends=True)}


_nb = {
    "cells": [
        _md_cell(KERNEL_CELLS["md0_title.md"], "md0-title"),
        _code_cell(KERNEL_CELLS["c1_wheels.py"], "c1-wheels"),
        _code_cell(_c2, "c2-agent"),
        _code_cell(KERNEL_CELLS["c3_pipeline.py"], "c3-pipeline"),
        _code_cell(KERNEL_CELLS["c4_rerun.py"], "c4-rerun"),
        _code_cell(KERNEL_CELLS["c5_parquet.py"], "c5-parquet"),
        _md_cell(KERNEL_CELLS["md6_summary.md"], "md6-summary"),
    ],
    "metadata": {
        "kernelspec": {"display_name": "Python 3", "language": "python",
                       "name": "python3"},
        "language_info": {"name": "python", "version": "3.11"},
        "kaggle": {
            "accelerator": "none",
            "dataSources": [
                {"sourceId": 0,
                 "sourceType": "competition",
                 "datasetId": None,
                 "competitionId": 0,
                 "filePath": ""}
            ],
            "isGpuEnabled": False,
            "isInternetEnabled": False,
            "language": "python",
            "sourceType": "notebook",
        },
    },
    "nbformat": 4,
    "nbformat_minor": 5,
}
_kernel_text = json.dumps(_nb, indent=1) + "\n"
kernel_path = _write("ARC_AGI_3_Kagglea2.ipynb", _kernel_text)

# --- artifact 3: build_manifest.json ----------------------------------------
_manifest = dict(BUILD)
_manifest.update({
    "generated_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
    "artifacts": [
        {"name": "program_synthesis_agent_v3.py",
         "bytes": ARTIFACTS["program_synthesis_agent_v3.py"]["bytes"],
         "sha256": AGENT_SHA256},
        {"name": "ARC_AGI_3_Kagglea2.ipynb",
         "bytes": ARTIFACTS["ARC_AGI_3_Kagglea2.ipynb"]["bytes"],
         "sha256": ARTIFACTS["ARC_AGI_3_Kagglea2.ipynb"]["sha256"]},
    ],
    "agent_sha256": AGENT_SHA256,
    "kernel_sha256": ARTIFACTS["ARC_AGI_3_Kagglea2.ipynb"]["sha256"],
    "offline_tests": OFFLINE_TEST_RESULTS,
    "pipeline_note": ("working-pipeline executor runs as cell 16 AFTER this "
                      "manifest is written; outcome is printed by the cell-17 "
                      "build report (and recorded in stdout)"),
    "environment": {
        "python": platform.python_version(),
        "platform": platform.platform(),
        "builder_notebook": "4CBOn2_Kaggle3.ipynb",
    },
})
_manifest_text = json.dumps(_manifest, indent=2) + "\n"
manifest_path = _write("build_manifest.json", _manifest_text)

print(f"[c15] emitted 3 artifacts into {OUT_DIR}:")
for _n in ("program_synthesis_agent_v3.py", "ARC_AGI_3_Kagglea2.ipynb",
           "build_manifest.json"):
    print(f"[c15]   {_n:34s} {ARTIFACTS[_n]['bytes']:>9,} bytes  "
          f"sha256={ARTIFACTS[_n]['sha256'][:16]}…")


[c15] emitted 3 artifacts into /content:
[c15]   program_synthesis_agent_v3.py         46,536 bytes  sha256=ffce9e893f983c63…
[c15]   ARC_AGI_3_Kagglea2.ipynb              87,400 bytes  sha256=01f2bc276e2212ea…
[c15]   build_manifest.json                    4,691 bytes  sha256=c408cf5f5f6f0040…


In [ ]:
# CELL 15B — PINNED UPSTREAM SUBMISSION-AGENT STAGER (additive; needs cell 15).
# Stages mohamtur1/arc-agi-3-agent @ pinned revision BESIDE the frozen v3
# baseline, reusing cell 15's emitter (_write/ARTIFACTS) so cell 17's build
# report verifies the new hashes too. The frozen baseline (cell 13) is never
# replaced — cell 17 hard-fails on any byte change to it, by design.
# No Kaggle API, no submission, no candidate scoring, no git push, no model /
# provider change. Network is optional (E1-E4 fallbacks below).
import difflib
import glob
import hashlib
import importlib.util
import inspect
import json
import os
import subprocess
import tempfile

PIN_REPO = "https://github.com/mohamtur1/arc-agi-3-agent"
PIN_REV = "5aa2b23f0722e2ebd2884b6f139d9f727d27c838"
PIN_FILE = "program_synthesis_agent.py"
PIN_SHA = "07676998e07b9839b1397076ef8d0884a32ea3483696332cf52e1c83692b8c98"
PIN_BYTES = 25611
PIN_ENTRY = "ProgramSynthesisKitAgent"
STAGE = "submission_agent"
VENDOR = ["_vendor/" + PIN_FILE, "Kaggle/" + PIN_FILE]
KIT_KWARGS = {"card_id", "game_id", "agent_name", "ROOT_URL", "record",
              "arc_env", "tags"}
FOOTER = ("\n\n# --- added by cell 15B; NOT part of the pinned upstream file ---\n"
          "# ARC-AGI-3-Kaggle-Starter needs a class named MyAgent inside\n"
          "# agent/my_agent.py (scripts/play_local.py enforces this).\n"
          "MyAgent = ProgramSynthesisKitAgent\n")

if "_write" not in globals() or "ARTIFACTS" not in globals():
    raise RuntimeError("[c15b] run cell 15 first: this cell reuses its emitter "
                       "(_write/ARTIFACTS); it does not duplicate it")


def _sha256(_b):
    return hashlib.sha256(_b).hexdigest()


_src = _how = None
_tried = []

# E1 - the exact git revision (canonical pinned source; skippable offline).
if not os.environ.get("SUBMISSION_AGENT_OFFLINE"):
    _tmp = tempfile.mkdtemp(prefix="pinned_agent_")
    try:
        for _c in (["git", "clone", "--quiet", PIN_REPO, _tmp],
                   ["git", "-C", _tmp, "checkout", "--quiet", "--detach",
                    PIN_REV]):
            subprocess.run(_c, check=True, capture_output=True, text=True,
                           timeout=300)
        _head = subprocess.run(["git", "-C", _tmp, "rev-parse", "HEAD"],
                               check=True, capture_output=True, text=True,
                               timeout=60).stdout.strip()
        _b = open(os.path.join(_tmp, PIN_FILE), "rb").read()
        if _head != PIN_REV or _sha256(_b) != PIN_SHA:
            raise RuntimeError(f"HEAD={_head[:12]} sha256={_sha256(_b)[:12]}")
        _src, _how = _b, f"git:{PIN_REPO}@{PIN_REV[:12]}"
    except Exception as _exc:
        _tried.append(f"E1 git: {type(_exc).__name__}: {str(_exc)[:90]}")

# E2-E4 - a hash-verified local copy IS the pin (same bytes = same revision).
if _src is None:
    _cands = [os.environ.get("SUBMISSION_AGENT_SRC")] + VENDOR
    _cands += sorted(glob.glob("/kaggle/input/*/" + PIN_FILE))
    for _p in [c for c in _cands if c]:
        if os.path.isfile(_p):
            _b = open(_p, "rb").read()
            if _sha256(_b) == PIN_SHA:
                _src, _how = _b, f"vendored:{_p}"
                break
            _tried.append(f"{_p}: sha256={_sha256(_b)[:12]} mismatch")

if _src is None:
    raise RuntimeError(
        "[c15b] PINNED SOURCE UNAVAILABLE - nothing staged, frozen baseline "
        "untouched.\n"
        f"  need  : {PIN_FILE} @ {PIN_REV}\n"
        f"  sha256: {PIN_SHA} ({PIN_BYTES} bytes)\n"
        "  tried : " + ("; ".join(_tried) or "(nothing)") + "\n"
        "  pick one:\n"
        "   E1 internet: make git available, re-run this cell.\n"
        "   E2 upload the file, then set\n"
        "      os.environ['SUBMISSION_AGENT_SRC'] = "
        "'/path/program_synthesis_agent.py'\n"
        "   E3 Kaggle dataset: upload program_synthesis_agent.py as a private\n"
        "      dataset, add it as notebook input, re-run (auto-found under\n"
        "      /kaggle/input/*/ ).\n"
        "   E4 copy it to ./_vendor/program_synthesis_agent.py, re-run.\n"
        "  A byte-identical copy already ships in your own repo at\n"
        "  Kaggle/program_synthesis_agent.py (same sha256) -> E2/E4 are quick.")

# --- 1) read-only comparison with the frozen v3 core (cell 13) --------------
CMP = {"frozen_cell13_sha256": globals().get("AGENT_SHA256", ""),
       "upstream_pin_sha256": PIN_SHA}
CMP["equal"] = CMP["frozen_cell13_sha256"] == PIN_SHA
if not CMP["equal"]:
    _a = globals().get("AGENT_SOURCE", "").split("\n")
    _b = _src.decode("utf-8").split("\n")
    _d = list(difflib.unified_diff(_a, _b, "frozen_v3_cell13", "upstream_pin",
                                   lineterm="", n=1))
    CMP.update({"frozen_bytes": len(globals().get("AGENT_SOURCE", "")),
                "frozen_lines": len(_a), "pin_lines": len(_b),
                "changed_hunks": sum(1 for _l in _d if _l.startswith("@@")),
                "sample": _d[:8]})

# --- 2) stage through cell 15's emitter (no re-implementation) -------------
_txt = _src.decode("utf-8")
if _sha256(_txt.encode("utf-8")) != PIN_SHA:
    raise RuntimeError("[c15b] utf-8 round-trip would change bytes; aborted")

os.makedirs(os.path.join(OUT_DIR, STAGE, "agent"), exist_ok=True)
_rel = [os.path.join(STAGE, PIN_FILE), os.path.join(STAGE, "agent",
                                                    "my_agent.py"),
        os.path.join(STAGE, "STAGED_SOURCE.json")]

_p_src = _write(_rel[0], _txt)
_p_drop = _write(_rel[1], _txt + FOOTER)
if open(_p_drop, "rb").read()[:len(_src)] != _src:
    raise RuntimeError("[c15b] drop-in body is not byte-identical; aborted")

# --- 3) entry point / API checks (structural = hard fail) -----------------
_spec = importlib.util.spec_from_file_location("staged_pinned_agent", _p_src)
_mod = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(_mod)
_ent = getattr(_mod, PIN_ENTRY, None)
_kw = set(inspect.signature(_ent.__init__).parameters) - {"self"} \
    if _ent is not None else set()
API = {"entry_point": _ent is not None,
       "core_class": hasattr(_mod, "ProgramSynthesisCore"),
       "choose_action": callable(getattr(_ent, "choose_action", None)),
       "is_done": callable(getattr(_ent, "is_done", None)),
       "max_actions": getattr(_ent, "MAX_ACTIONS", None),
       "starter_kwargs_ok": KIT_KWARGS <= _kw,
       "missing_kwargs": sorted(KIT_KWARGS - _kw),
       "arcengine_present": bool(getattr(_mod, "ARCENGINE_AVAILABLE", False))}
for _k in ("entry_point", "core_class", "choose_action", "is_done",
           "starter_kwargs_ok"):
    if not API[_k]:
        raise RuntimeError(f"[c15b] API check failed: {_k} -> {API}")

# --- 4) behavioural smoke (informational only, never fatal) ---------------
_smoke = {"real_frame": None, "null_frame_robust": None}
try:
    _g = [[0] * 64 for _ in range(64)]
    for _i in range(0, 64, 8):
        _g[_i][_i] = 3
    _fr = _mod.FrameData(game_id="stage-smoke", frame=[_g],
                         state=_mod.GameState.NOT_FINISHED,
                         levels_completed=0, win_levels=1,
                         available_actions=[1, 2, 3, 4, 6])
    _ag = _ent()
    _act = _ag.choose_action([_fr], _fr)
    _smoke["real_frame"] = {"action": str(getattr(_act, "name", _act)),
                            "is_done": bool(_ag.is_done([_fr], _fr))}
except Exception as _exc:
    _smoke["real_frame"] = {"error": f"{type(_exc).__name__}: {_exc}"}
try:
    _ent().choose_action([], None)
    _smoke["null_frame_robust"] = True
except Exception:
    _smoke["null_frame_robust"] = False   # upstream has no never-hang wrapper

# --- 5) provenance record + manifest registration (reuses _write) ---------
PROV = {"repo": PIN_REPO, "rev": PIN_REV, "path_in_repo": PIN_FILE,
        "sha256": PIN_SHA, "bytes": PIN_BYTES, "fetched_via": _how,
        "entry_point": PIN_ENTRY,
        "staged": {_rel[0]: PIN_SHA,
                   _rel[1]: ARTIFACTS[_rel[1]]["sha256"]},
        "kit_kwargs_forwarded": sorted(KIT_KWARGS),
        "frozen_core_comparison": CMP,
        "frozen_baseline_replaced": False,
        "api_checks": API, "offline_smoke": _smoke,
        "manual_run_hint": (
            "cp submission_agent/agent/my_agent.py "
            "ARC-AGI-3-Kaggle-Starter/agent/my_agent.py && "
            "make -C ARC-AGI-3-Kaggle-Starter play-local GAME=ls20 "
            "MPLBACKEND=Agg"),
        "not_done_here": ["no Kaggle API call", "no submission", "no scoring",
                          "no git push", "no model/provider change"]}
_write(_rel[2], json.dumps(PROV, indent=2) + "\n")

_m = json.loads(open(os.path.join(OUT_DIR, "build_manifest.json")).read())
_m["artifacts"] = [a for a in _m.get("artifacts", [])
                   if a.get("name") not in set(_rel)]
for _n in _rel:
    _m["artifacts"].append({"name": _n, "bytes": ARTIFACTS[_n]["bytes"],
                            "sha256": ARTIFACTS[_n]["sha256"]})
_m["pinned_submission_agent"] = PROV
_write("build_manifest.json", json.dumps(_m, indent=2) + "\n")

# --- 6) report ------------------------------------------------------------
print("[c15b] staged pinned submission agent — frozen v3 baseline untouched")
print(f"  pin    : {PIN_REPO}@{PIN_REV[:12]} via {_how}")
print(f"  source : {_rel[0]} ({PIN_BYTES:,} B, sha256={PIN_SHA[:16]}…)")
print(f"  dropin : {_rel[1]} sha256="
      f"{ARTIFACTS[_rel[1]]['sha256'][:16]}…")
if CMP["equal"]:
    print("  frozen : IDENTICAL to cell 13 core")
else:
    print(f"  frozen : DIFFERS from cell 13 core "
          f"({CMP['frozen_lines']} vs {CMP['pin_lines']} lines, "
          f"{CMP['changed_hunks']} hunks, kept as-is)")
    for _l in CMP["sample"]:
        print("    | " + _l[:88])
print(f"  api    : {PIN_ENTRY} + core OK, starter kwargs OK, "
      f"arcengine={API['arcengine_present']}, smoke={_smoke['real_frame']}")
print(f"  smoke  : null-frame robust={_smoke['null_frame_robust']} "
      f"(cell 13 keeps its never-hang wrapper)")
print("  verify : cell 17 re-hashes these staged files from the manifest")
print("  manual : " + PROV["manual_run_hint"])
print("  note   : no score here (offline-practice 48 and the pending official "
      "result are separate); Kaggle Submit stays a human decision.")


[c15b] staged pinned submission agent — frozen v3 baseline untouched
  pin    : https://github.com/mohamtur1/arc-agi-3-agent@5aa2b23f0722 via git:https://github.com/mohamtur1/arc-agi-3-agent@5aa2b23f0722
  source : submission_agent/program_synthesis_agent.py (25,611 B, sha256=07676998e07b9839…)
  dropin : submission_agent/agent/my_agent.py sha256=fb8059f3baab2e24…
  frozen : DIFFERS from cell 13 core (1179 vs 654 lines, 24 hunks, kept as-is)
    | --- frozen_v3_cell13
    | +++ upstream_pin
    | @@ -1,28 +1,20 @@
    | -"""program_synthesis_agent_v3.py — self-contained ARC-AGI-3 agent (v2.1)
    | -
    | -v2.1 adds two features over v2.0:
    | -  F4 state-action exploration graph (novelty-seeking when no hypothesis)
    | -  F5 typed evidence: transient/revert detection (settled vs animation)
  api    : ProgramSynthesisKitAgent + core OK, starter kwargs OK, arcengine=False, smoke={'action': 'ACTION6', 'is_done': False}
  smoke  : null-frame robust=False (cell 13 keeps its never-hang

In [ ]:
# CELL 16 — WORKING-PIPELINE EXECUTOR (the proven v1 submission pipeline,
# run against the artifact just emitted by cell 15: clone the official
# starter, install its venv, drop the agent into agent/my_agent.py, run
# `MPLBACKEND=Agg make play-local` on a small game set). Auto-skips with
# SKIP_LOCAL_PIPELINE=1; any failure is non-fatal to Run All.
import os
import shutil
import subprocess

PIPELINE = {"clone": False, "setup": False, "play_local": False,
            "games": os.environ.get("PIPELINE_GAMES", "ls20,vc33"),
            "output": ""}

if os.environ.get("SKIP_LOCAL_PIPELINE"):
    print("[c16] pipeline skipped (SKIP_LOCAL_PIPELINE=1)")
else:
    import gc as _gc
    _gc.collect()   # low-RAM hosts: release what we can before subprocesses

    def _sh(cmd, timeout):
        r = subprocess.run(cmd, shell=True, capture_output=True, text=True,
                           timeout=timeout)
        return r.returncode, (r.stdout or "") + (r.stderr or "")

    _dir = os.getcwd()
    _agent = os.path.join(_dir, "program_synthesis_agent_v3.py")
    _starter = os.path.join(_dir, "ARC-AGI-3-Kaggle-Starter")
    try:
        assert os.path.exists(_agent), "cell 15 artifact missing"
        if not os.path.isdir(_starter):
            _rc, _out = _sh(
                "git clone --depth 1 "
                "https://github.com/arcprize/ARC-AGI-3-Kaggle-Starter.git "
                f"{_starter}", 300)
            if _rc != 0:
                raise RuntimeError(f"clone failed: {_out[-300:]}")
        else:
            print("[c16] starter already cloned")
        PIPELINE["clone"] = True

        shutil.copyfile(_agent, os.path.join(_starter, "agent", "my_agent.py"))

        _py = "python3.12" if shutil.which("python3.12") else "python3"
        _rc, _out = _sh(f"make -C {_starter} setup PYTHON={_py}", 900)
        if _rc != 0:
            raise RuntimeError(f"setup failed: {_out[-400:]}")
        PIPELINE["setup"] = True

        _rc, _out = _sh(
            f"make -C {_starter} play-local GAME={PIPELINE['games']} "
            "MAX_STEPS=200 MPLBACKEND=Agg", 1800)
        PIPELINE["output"] = _out
        if _rc != 0 or "SUMMARY" not in _out:
            raise RuntimeError(f"play-local failed: {_out[-400:]}")
        PIPELINE["play_local"] = True

        _lines = _out.splitlines()
        try:
            _si = next(i for i, l in enumerate(_lines) if "SUMMARY" in l)
            _tail = _lines[_si:_si + 12]
        except StopIteration:
            _tail = [l for l in _lines if l.strip()][-8:]
        print(f"[c16] pipeline OK — clone/setup/play-local on "
              f"[{PIPELINE['games']}]")
        for _l in _tail:
            print("  | " + _l)
    except Exception as _exc:
        print(f"[c16] pipeline FAILED (non-fatal): {_exc}")


[c16] pipeline FAILED (non-fatal): setup failed: need to install the python3-venv
package using the following command.

    apt install python3.12-venv

You may need to use sudo with that command.  After installing the python3-venv
package, recreate your virtual environment.

Failing command: /content/ARC-AGI-3-Kaggle-Starter/.venv/bin/python3.12

make: Leaving directory '/content/ARC-AGI-3-Kaggle-Starter'
make: *** [Makefile:39: setup] Error 1



In [ ]:
# CELL 17 — BUILD REPORT (verifies the three artifacts on disk against the
# frozen sources and the manifest, then prints the final report. Read-only.)
import hashlib
import json
import os

_dir = os.getcwd()
REPORT = {"checks": [], "pipeline": None}

_agent_disk = open(os.path.join(_dir, "program_synthesis_agent_v3.py"),
                   "rb").read()
_kernel_disk = open(os.path.join(_dir, "ARC_AGI_3_Kagglea2.ipynb"),
                    "rb").read()
_manifest_disk = open(os.path.join(_dir, "build_manifest.json"), "rb").read()

# 1) agent byte-identity with the frozen core
_ok1 = _agent_disk.decode("utf-8") == AGENT_SOURCE
REPORT["checks"].append(("agent byte-identical to frozen AGENT_SOURCE", _ok1))

# 2) kernel: valid JSON, 7 cells, embedded agent byte-identical, metadata
_kb = json.loads(_kernel_disk.decode("utf-8"))
_src2 = "".join(_kb["cells"][2]["source"])
_emb = _src2.split("AGENT_SOURCE = r'''", 1)[1].rsplit("'''", 1)[0]
_ok2 = (len(_kb["cells"]) == 7 and _emb == AGENT_SOURCE
        and _kb["metadata"]["kaggle"]["isInternetEnabled"] is False
        and _kb["metadata"]["kaggle"]["accelerator"] == "none")
REPORT["checks"].append(("kernel notebook: 7 cells, embedded agent "
                         "byte-identical, internet-off metadata", _ok2))

# 3) manifest: loads, artifact hashes match the files on disk
_m = json.loads(_manifest_disk.decode("utf-8"))
_ok3 = True
for _a in _m["artifacts"]:
    _b = open(os.path.join(_dir, _a["name"]), "rb").read()
    if (hashlib.sha256(_b).hexdigest() != _a["sha256"]
            or len(_b) != _a["bytes"]):
        _ok3 = False
_ok3 = _ok3 and _m.get("offline_tests", {}).get("verdict") == "PASS"
REPORT["checks"].append(("build_manifest.json valid; artifact hashes match "
                         "disk; offline tests recorded PASS", _ok3))

REPORT["pipeline"] = {
    "clone": PIPELINE["clone"], "setup": PIPELINE["setup"],
    "play_local": PIPELINE["play_local"], "games": PIPELINE["games"],
}

print()
print("=" * 72)
print(f"BUILD REPORT — {BUILD['product']} ({BUILD['build_id']})")
print("=" * 72)
for _label, _ok in REPORT["checks"]:
    print(f"  {'PASS' if _ok else 'FAIL'} — {_label}")
print(f"  pipeline: clone={REPORT['pipeline']['clone']} "
      f"setup={REPORT['pipeline']['setup']} "
      f"play-local={REPORT['pipeline']['play_local']} "
      f"[{REPORT['pipeline']['games']}]")
print(f"  offline demo games: "
      f"{json.dumps(_m['offline_tests']['games'])}")
print(f"  scope: {len(BUILD['scope']['features_v2_1'])} features + baseline; "
      f"App cell changes: {BUILD['scope']['app_cell_changes']}")
print(f"  agent sha256={AGENT_SHA256[:32]}…")
print("=" * 72)
if not all(_ok for _, _ok in REPORT["checks"]):
    raise RuntimeError("BUILD REPORT: at least one artifact check FAILED")
print("BUILD COMPLETE — 3 artifacts verified: "
      "program_synthesis_agent_v3.py, ARC_AGI_3_Kagglea2.ipynb, "
      "build_manifest.json")


BUILD REPORT — Legacy ARC-AGI-3 ProgramSynthesis agent v2.1 (v3-fixed) (v2.1-M3)
  PASS — agent byte-identical to frozen AGENT_SOURCE
  PASS — kernel notebook: 7 cells, embedded agent byte-identical, internet-off metadata
  PASS — build_manifest.json valid; artifact hashes match disk; offline tests recorded PASS
  pipeline: clone=True setup=False play-local=False [ls20,vc33]
  offline demo games: {"demo-odd-one-out": {"fresh_actions": 4, "replay_actions": 4, "newseed_actions": 10, "levels": "4/4"}, "demo-follow-sample": {"fresh_actions": 10, "replay_actions": 7, "newseed_actions": 12, "levels": "3/3"}, "demo-reach-goal": {"fresh_actions": 21, "replay_actions": 21, "newseed_actions": 35, "levels": "3/3"}}
  scope: 4 features + baseline; App cell changes: Kaggle4 design flow in cells 0, 8-11; cells 12-17 remain the frozen v3 baseline emitter
  agent sha256=ffce9e893f983c632e9cf82e85face00…
BUILD COMPLETE — 3 artifacts verified: program_synthesis_agent_v3.py, ARC_AGI_3_Kagglea2.ipynb, bu